# 예제 03. UI 와 한계 — 그리고 완성품 평가

전산실험 · AI Agent 만들기 · **13주차** · 예상 소요 35분

## 목표
- Gradio 로 UI 를 붙인다
- **UI 가 예뻐지면 사람들이 믿는다** — 한계를 고지한다
- 11주차 하네스로 **완성품을 평가**한다
- 결과를 **구간과 함께** 보고한다

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

사용 예:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때."""


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때."""


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    Returns
    -------
    str : JSON 으로 보이는 부분

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError."""
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    Parameters
    ----------
    schema  : validate() 형식의 스키마
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
    n_retry : 최대 시도 횟수
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
    verbose : 시도 과정을 출력

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
%%writefile memory.py
"""전산실험2 4주차 : 대화 상태와 메모리.

모델은 이전 대화를 기억하지 못한다. 기억은 **우리 코드가** 만든다.
그런데 대화를 통째로 다시 보내다 보면 컨텍스트 윈도우에 부딪힌다.

이 모듈은 세 가지를 제공한다.

    Conversation   대화 기록을 들고 있는 그릇 (system 은 항상 보존)
    예산 관리       토큰을 세고, 넘치면 오래된 턴을 잘라낸다
    요약 압축       잘라내는 대신 요약으로 갈아끼운다 (정보 손실 최소화)

사용 예:
    from kmu_llm import LLM
    from memory import Conversation

    llm = LLM()
    conv = Conversation(llm, system="너는 통계 조교다.", max_tokens=2000)
    print(conv.say("표준편차가 뭔가요?"))
    print(conv.say("그럼 표준오차는요?"))     # 앞 대화를 기억한다
    conv.summary()                          # 현재 토큰 사용 현황
"""

from __future__ import annotations

from typing import Any, Literal

__all__ = ["estimate_tokens", "Message", "Conversation"]

Role = Literal["system", "user", "assistant"]


# --------------------------------------------------------------------------
# 토큰 추정
# --------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """토큰 수를 대략 추정한다. 서버에 묻지 않고 빠르게 계산한다.

    정확한 값은 모델의 토크나이저에 따라 다르다. 여기서는 경험적 근사를 쓴다.

        한글 1글자  ~ 1.2 토큰   (자소 단위로 잘게 쪼개짐)
        영문 1글자  ~ 0.3 토큰   (단어가 통째로 1토큰이 되는 경우가 많음)

    루프를 돌 때마다 서버에 물어보면 호출이 배로 늘어난다. 예산 관리는
    **근사로 충분**하고, 정확한 값이 필요하면 `LLM.count_tokens()` 를 쓴다.
    """
    if not text:
        return 0
    hangul = sum(1 for ch in text if "가" <= ch <= "힣")
    other = len(text) - hangul
    return int(hangul * 1.2 + other * 0.3) + 1


class Message(dict):
    """{"role": ..., "content": ...} 에 토큰 수를 얹은 것."""

    def __init__(self, role: Role, content: str) -> None:
        super().__init__(role=role, content=content)

    @property
    def tokens(self) -> int:
        return estimate_tokens(self["content"]) + 4  # 역할 표시 등 부대 비용

    def as_api(self) -> dict[str, str]:
        return {"role": self["role"], "content": self["content"]}


# --------------------------------------------------------------------------
# 대화
# --------------------------------------------------------------------------
class Conversation:
    """대화 기록을 관리하고 예산을 넘지 않게 유지한다.

    Parameters
    ----------
    llm : LLM 인스턴스
    system : 대화 전체에 적용되는 지시. **절대 잘려나가지 않는다.**
    max_tokens : 전체 대화의 토큰 예산 (응답 공간은 별도)
    strategy :
        "drop"    오래된 턴을 그냥 버린다 (빠르고 단순, 정보 손실)
        "summary" 오래된 턴을 요약으로 갈아끼운다 (호출 1회 추가, 손실 적음)
        "none"    아무것도 하지 않는다 (한계를 직접 보고 싶을 때)
    keep_recent : 최근 몇 개의 메시지는 무조건 원본으로 보존할지
    """

    def __init__(
        self,
        llm: Any,
        system: str | None = None,
        *,
        max_tokens: int = 3000,
        strategy: str = "summary",
        keep_recent: int = 4,
    ) -> None:
        if strategy not in ("drop", "summary", "none"):
            raise ValueError(f"알 수 없는 strategy: {strategy!r}")

        self.llm = llm
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.keep_recent = keep_recent

        self.system = Message("system", system) if system else None
        self.history: list[Message] = []
        self.compressions = 0          # 몇 번 압축했는지
        self.dropped = 0               # 몇 개를 버렸는지

    # -- 상태 확인 --------------------------------------------------------
    def messages(self) -> list[dict[str, str]]:
        """API 로 보낼 형태."""
        head = [self.system] if self.system else []
        return [m.as_api() for m in head + self.history]

    @property
    def tokens(self) -> int:
        head = self.system.tokens if self.system else 0
        return head + sum(m.tokens for m in self.history)

    def summary(self) -> dict[str, Any]:
        """현재 상태를 한눈에."""
        return {
            "메시지수": len(self.history) + (1 if self.system else 0),
            "추정토큰": self.tokens,
            "예산": self.max_tokens,
            "사용률": round(self.tokens / self.max_tokens, 3),
            "압축횟수": self.compressions,
            "버린메시지": self.dropped,
        }

    # -- 대화 -------------------------------------------------------------
    def add(self, role: Role, content: str) -> None:
        self.history.append(Message(role, content))

    def say(self, text: str, **kw: Any) -> str:
        """사용자 발화를 넣고 모델의 답을 받아 기록에 남긴다.

        예산은 **보내기 전과 받은 뒤 두 번** 맞춘다. 그래야
        "대화는 항상 예산 안에 있다"는 불변식이 유지되어,
        지금 상태를 보고 다음 호출이 들어갈지 예측할 수 있다.
        """
        self.add("user", text)
        self.fit()                                  # 보내기 전
        reply = self.llm.chat(self.messages(), **kw)
        self.add("assistant", reply)
        self.fit()                                  # 받은 뒤
        return reply

    # -- 예산 관리 --------------------------------------------------------
    def fit(self) -> None:
        """예산을 넘으면 전략에 따라 줄인다."""
        if self.strategy == "none" or self.tokens <= self.max_tokens:
            return
        if self.strategy == "drop":
            self._drop_old()
        else:
            self._summarize_old()

    def _split(self) -> tuple[list[Message], list[Message]]:
        """(압축 대상인 오래된 것, 보존할 최근 것)"""
        if len(self.history) <= self.keep_recent:
            return [], self.history
        return self.history[: -self.keep_recent], self.history[-self.keep_recent :]

    def _drop_old(self) -> None:
        """가장 오래된 것부터 예산에 맞을 때까지 버린다.

        `keep_recent` 는 보존하려 하지만, 그것만으로도 예산을 넘으면
        보존 약속을 깨고 더 버린다. 마지막 메시지 하나는 남긴다 —
        그것이 사용자의 방금 질문이기 때문이다.
        """
        old, recent = self._split()
        while old and self.tokens > self.max_tokens:
            old.pop(0)
            self.dropped += 1
            self.history = old + recent
        self.history = old + recent

        # 최근 것만 남았는데도 넘치면, 보존 약속을 깨고 더 버린다
        while len(self.history) > 1 and self.tokens > self.max_tokens:
            self.history.pop(0)
            self.dropped += 1

        if self.tokens > self.max_tokens:
            # 마지막 메시지 하나가 예산보다 크다. 내용을 몰래 자르지는 않는다.
            # 조용히 틀리느니 시끄럽게 알리는 편이 낫다. (3주차와 같은 원칙)
            print(
                f"[경고] 마지막 메시지 하나가 예산({self.max_tokens} 토큰)을 "
                f"넘습니다 (현재 {self.tokens}). 예산을 늘리거나 입력을 줄이세요."
            )

    def _summarize_old(self) -> None:
        """오래된 대화를 요약 한 덩어리로 갈아끼운다."""
        old, recent = self._split()
        if not old:
            self._drop_old()          # 줄일 것이 최근 것뿐이면 버리는 수밖에 없다
            return

        기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
        지시 = (
            "다음은 사용자와 조수의 지난 대화다. 이후 대화를 이어가는 데 필요한 "
            "사실만 5문장 이내로 요약하라. 결정된 내용, 사용자가 밝힌 정보, "
            "아직 해결되지 않은 질문을 우선 남겨라. 인사말은 버려라.\n\n"
            f"{기록}"
        )
        try:
            요약 = self.llm.ask(지시, temperature=0.0, max_tokens=400)
        except Exception:
            self._drop_old()          # 요약 호출이 실패하면 버리기로 대체
            return

        self.history = [Message("user", f"[이전 대화 요약]\n{요약}")] + recent
        self.compressions += 1

        if self.tokens > self.max_tokens:   # 요약해도 넘치면 결국 버린다
            self._drop_old()

    # -- 저장 / 복원 ------------------------------------------------------
    def to_dict(self) -> dict[str, Any]:
        return {
            "system": self.system["content"] if self.system else None,
            "history": [m.as_api() for m in self.history],
            "compressions": self.compressions,
            "dropped": self.dropped,
        }

    @classmethod
    def from_dict(cls, llm: Any, data: dict[str, Any], **kw: Any) -> "Conversation":
        conv = cls(llm, system=data.get("system"), **kw)
        for m in data.get("history", []):
            conv.add(m["role"], m["content"])
        conv.compressions = data.get("compressions", 0)
        conv.dropped = data.get("dropped", 0)
        return conv


In [ ]:
%%writefile tools.py
"""전산실험2 5주차 : 도구(Tool)와 도구 레지스트리.

에이전트가 바깥 세계에 손을 뻗는 방법.

핵심 원칙:
    **계산은 모델이 아니라 도구가 한다.**

    LLM은 "어떤 도구를 어떤 인자로 부를지" 만 정한다.
    평균을 구하고 p값을 계산하는 것은 numpy 와 scipy 가 한다.
    1주차에 확인했듯 LLM은 난수도 제대로 못 뽑는다. 계산을 맡기면 안 된다.

구성:
    Tool          이름 · 설명 · 인자 명세 · 실제 함수
    ToolResult    실행 결과 (성공/실패, 값, 모델에게 돌려줄 관찰 문자열)
    ToolRegistry  도구를 등록·직렬화·호출한다
    StatToolbox   DataFrame 에 묶인 통계 도구 모음

사용 예:
    import pandas as pd
    from tools import StatToolbox

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    print(box.describe())                     # 프롬프트에 넣을 도구 목록
    r = box.call("compare_two_groups",
                 {"value_column": "bmi", "group_column": "group"})
    print(r.observation())
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = ["Tool", "ToolResult", "ToolRegistry", "StatToolbox"]


# --------------------------------------------------------------------------
# 도구 정의
# --------------------------------------------------------------------------
@dataclass
class Tool:
    """도구 하나.

    `parameters` 형식::

        {
            "column": {"type": "str", "desc": "분석할 열 이름", "required": True},
            "alpha":  {"type": "float", "desc": "유의수준", "default": 0.05},
            "method": {"type": "str", "enum": ["pearson", "spearman"]},
        }
    """

    name: str
    description: str
    parameters: dict[str, dict[str, Any]]
    func: Callable[..., Any]

    def spec_line(self) -> str:
        """프롬프트에 넣을 한 줄 명세."""
        args = []
        for pname, p in self.parameters.items():
            t = p.get("type", "str")
            mark = "" if p.get("required") else "?"
            enum = f"={'|'.join(map(str, p['enum']))}" if "enum" in p else ""
            args.append(f"{pname}{mark}: {t}{enum}")
        return f"- {self.name}({', '.join(args)}) — {self.description}"

    def validate(self, args: dict[str, Any]) -> list[str]:
        """인자를 검사한다. 문제 목록을 돌려준다 (빈 목록 = 통과).

        3주차 `structured.validate()` 와 같은 발상이다.
        **모델이 준 것을 믿지 않고 코드가 확인한다.**
        """
        problems: list[str] = []

        for pname, p in self.parameters.items():
            if p.get("required") and pname not in args:
                problems.append(f"필수 인자 '{pname}' 가 없습니다.")

        known = set(self.parameters)
        for given in args:
            if given not in known:
                problems.append(
                    f"'{given}' 는 이 도구의 인자가 아닙니다. 가능한 인자: {sorted(known)}"
                )

        for pname, value in args.items():
            p = self.parameters.get(pname)
            if p is None:
                continue
            if "enum" in p and value not in p["enum"]:
                problems.append(
                    f"'{pname}' 의 값 {value!r} 는 허용되지 않습니다. 가능한 값: {p['enum']}"
                )
            expected = p.get("type")
            if expected == "int" and not isinstance(value, int):
                problems.append(f"'{pname}' 는 정수여야 합니다 (받은 값: {value!r}).")
            elif expected == "float" and not isinstance(value, (int, float)):
                problems.append(f"'{pname}' 는 숫자여야 합니다 (받은 값: {value!r}).")
            elif expected == "bool" and not isinstance(value, bool):
                problems.append(f"'{pname}' 는 true/false 여야 합니다 (받은 값: {value!r}).")
            elif expected == "str" and not isinstance(value, str):
                problems.append(f"'{pname}' 는 문자열이어야 합니다 (받은 값: {value!r}).")
        return problems


@dataclass
class ToolResult:
    """도구 실행 결과."""

    tool: str
    ok: bool
    value: Any = None
    error: str | None = None
    hint: str | None = None      # 실패했을 때 다음에 무엇을 하면 되는지

    def observation(self, max_chars: int = 1200) -> str:
        """모델에게 돌려줄 문자열.

        **실패도 관찰이다.** 조용히 넘기지 않고 무엇이 왜 안 됐는지 알려준다.
        그래야 모델이 다음 행동을 고칠 수 있다 (3주차 되먹임과 같은 구조).
        """
        if not self.ok:
            msg = f"[{self.tool} 실패] {self.error}"
            if self.hint:
                msg += f"\n힌트: {self.hint}"
            return msg

        if isinstance(self.value, str):
            body = self.value
        else:
            body = json.dumps(self.value, ensure_ascii=False, indent=2, default=str)
        if len(body) > max_chars:
            body = body[:max_chars] + f"\n... (생략, 전체 {len(body)}자)"
        return f"[{self.tool} 결과]\n{body}"


# --------------------------------------------------------------------------
# 레지스트리
# --------------------------------------------------------------------------
class ToolRegistry:
    """도구를 등록하고, 프롬프트용으로 직렬화하고, 호출한다."""

    def __init__(self) -> None:
        self._tools: dict[str, Tool] = {}
        self.log: list[ToolResult] = []      # 실행 이력 (6주차 trace 의 씨앗)

    # -- 등록 -------------------------------------------------------------
    def register(self, tool: Tool) -> Tool:
        self._tools[tool.name] = tool
        return tool

    def tool(self, name: str, description: str, parameters: dict | None = None):
        """데코레이터로 등록한다.

            @registry.tool("mean", "평균을 구한다", {"col": {"type": "str", "required": True}})
            def _(col): ...
        """
        def deco(func: Callable[..., Any]) -> Callable[..., Any]:
            self.register(Tool(name, description, parameters or {}, func))
            return func

        return deco

    # -- 조회 -------------------------------------------------------------
    def names(self) -> list[str]:
        return sorted(self._tools)

    def __contains__(self, name: object) -> bool:
        return name in self._tools

    def __getitem__(self, name: str) -> Tool:
        return self._tools[name]

    def __len__(self) -> int:
        return len(self._tools)

    def describe(self) -> str:
        """프롬프트에 그대로 넣을 도구 목록."""
        lines = [t.spec_line() for _, t in sorted(self._tools.items())]
        return "\n".join(lines)

    # -- 실행 -------------------------------------------------------------
    def call(self, name: str, args: dict[str, Any] | None = None) -> ToolResult:
        """도구를 실행한다. **예외를 밖으로 던지지 않는다.**

        에이전트 루프 안에서 도구가 예외를 던지면 루프 전체가 죽는다.
        실패를 `ToolResult` 로 감싸 돌려주면 모델이 보고 고칠 수 있다.
        """
        args = args or {}

        if name not in self._tools:
            result = ToolResult(
                name, False,
                error=f"'{name}' 라는 도구가 없습니다.",
                hint=f"사용 가능한 도구: {self.names()}",
            )
            self.log.append(result)
            return result

        tool = self._tools[name]
        problems = tool.validate(args)
        if problems:
            result = ToolResult(
                name, False,
                error="인자가 잘못되었습니다: " + "; ".join(problems),
                hint=tool.spec_line(),
            )
            self.log.append(result)
            return result

        try:
            value = tool.func(**args)
            result = ToolResult(name, True, value=value)
        except Exception as exc:
            result = ToolResult(
                name, False,
                error=f"{type(exc).__name__}: {exc}",
                hint="인자 값을 확인하세요. 열 이름은 list_columns 로 확인할 수 있습니다.",
            )
        self.log.append(result)
        return result


# --------------------------------------------------------------------------
# 통계 도구 모음
# --------------------------------------------------------------------------
class StatToolbox(ToolRegistry):
    """DataFrame 하나에 묶인 통계 도구들.

    도구는 **작고, 하나만 하고, 결과를 구조화해서** 돌려줘야 한다.
    거대한 만능 도구를 하나 만들면 모델이 인자를 제대로 채우지 못한다.
    """

    def __init__(self, df: Any, name: str = "data") -> None:
        super().__init__()
        self.df = df
        self.name = name
        self._register_all()

    # -- 내부 헬퍼 --------------------------------------------------------
    def _numeric(self, column: str):
        """열을 수치로 강제 변환하고 결측을 뺀다.

        실제 데이터에는 '58세', ' 47 ' 같은 값이 섞여 있다.
        도구가 이것을 조용히 통과시키면 이후 계산이 전부 틀어진다.
        """
        import pandas as pd

        if column not in self.df.columns:
            raise KeyError(f"'{column}' 열이 없습니다. 있는 열: {list(self.df.columns)}")
        s = pd.to_numeric(self.df[column], errors="coerce")
        n_bad = int(s.isna().sum() - self.df[column].isna().sum())
        s = s.dropna()
        if len(s) < 3:
            raise ValueError(f"'{column}' 의 유효한 수치가 {len(s)}개뿐입니다.")
        return s, n_bad

    def _check_columns(self, *columns: str) -> None:
        missing = [c for c in columns if c not in self.df.columns]
        if missing:
            raise KeyError(f"{missing} 열이 없습니다. 있는 열: {list(self.df.columns)}")

    # -- 도구 등록 --------------------------------------------------------
    def _register_all(self) -> None:
        import numpy as np
        import pandas as pd
        from scipy import stats

        # ---------------------------------------------------------------
        @self.tool(
            "list_columns",
            "데이터의 열 이름, 타입, 결측 수, 고유값 수를 돌려준다. 항상 먼저 호출할 것.",
            {},
        )
        def _list_columns() -> dict:
            rows = []
            for c in self.df.columns:
                s = self.df[c]
                num = pd.to_numeric(s, errors="coerce")
                rows.append({
                    "column": c,
                    "dtype": str(s.dtype),
                    "n_missing": int(s.isna().sum()),
                    "n_unique": int(s.nunique(dropna=True)),
                    "numeric_ok": bool(num.notna().sum() > len(s) * 0.8),
                    "sample": [str(v) for v in s.dropna().head(3).tolist()],
                })
            return {"n_rows": int(len(self.df)), "columns": rows}

        # ---------------------------------------------------------------
        @self.tool(
            "describe_column",
            "한 열의 기술통계를 돌려준다. 수치형이면 평균·표준편차·사분위수, 범주형이면 빈도표.",
            {"column": {"type": "str", "desc": "열 이름", "required": True}},
        )
        def _describe_column(column: str) -> dict:
            self._check_columns(column)
            s = self.df[column]
            num = pd.to_numeric(s, errors="coerce")

            if num.notna().sum() > len(s) * 0.8:      # 수치형으로 본다
                v = num.dropna()
                q1, q3 = v.quantile([0.25, 0.75])
                return {
                    "column": column, "kind": "numeric", "n": int(len(v)),
                    "n_missing": int(s.isna().sum()),
                    "n_non_numeric": int(num.isna().sum() - s.isna().sum()),
                    "mean": round(float(v.mean()), 4),
                    "sd": round(float(v.std(ddof=1)), 4),
                    "min": round(float(v.min()), 4),
                    "q1": round(float(q1), 4),
                    "median": round(float(v.median()), 4),
                    "q3": round(float(q3), 4),
                    "max": round(float(v.max()), 4),
                    "skew": round(float(v.skew()), 3),
                }
            counts = s.value_counts(dropna=True)
            return {
                "column": column, "kind": "categorical",
                "n": int(s.notna().sum()), "n_missing": int(s.isna().sum()),
                "n_levels": int(counts.size),
                "counts": {str(k): int(v) for k, v in counts.head(15).items()},
            }

        # ---------------------------------------------------------------
        @self.tool(
            "check_normality",
            "한 열의 정규성을 Shapiro-Wilk 로 검정한다. 검정 방법을 고르기 전에 확인할 것.",
            {"column": {"type": "str", "desc": "수치형 열 이름", "required": True},
             "group_column": {"type": "str", "desc": "집단별로 나눠 검정할 열 (선택)"}},
        )
        def _check_normality(column: str, group_column: str | None = None) -> dict:
            if group_column is None:
                v, n_bad = self._numeric(column)
                W, p = stats.shapiro(v[:5000])
                return {"column": column, "n": int(len(v)),
                        "n_non_numeric": n_bad,
                        "statistic_W": round(float(W), 4), "p_value": float(p),
                        "normal_at_0.05": bool(p >= 0.05)}

            self._check_columns(column, group_column)
            out = {}
            work = self.df[[column, group_column]].copy()
            work[column] = pd.to_numeric(work[column], errors="coerce")
            for g, sub in work.dropna().groupby(group_column):
                if len(sub) < 3:
                    out[str(g)] = {"n": int(len(sub)), "error": "표본이 3 미만"}
                    continue
                W, p = stats.shapiro(sub[column][:5000])
                out[str(g)] = {"n": int(len(sub)),
                               "statistic_W": round(float(W), 4),
                               "p_value": float(p),
                               "normal_at_0.05": bool(p >= 0.05)}
            return {"column": column, "by": group_column, "groups": out}

        # ---------------------------------------------------------------
        @self.tool(
            "compare_two_groups",
            "두 독립 집단의 평균(또는 분포)을 비교한다. 등분산은 Welch 로 보정하며, "
            "method=auto 면 정규성 검정 결과에 따라 t검정/Mann-Whitney 를 고른다.",
            {"value_column": {"type": "str", "desc": "비교할 수치형 열", "required": True},
             "group_column": {"type": "str", "desc": "두 수준을 가진 집단 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "mannwhitney"],
                        "desc": "기본 auto"},
             "alpha": {"type": "float", "desc": "유의수준, 기본 0.05"}},
        )
        def _compare_two_groups(value_column: str, group_column: str,
                                method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            levels = sorted(work[group_column].astype(str).unique())
            if len(levels) != 2:
                raise ValueError(
                    f"'{group_column}' 의 수준이 {len(levels)}개입니다 ({levels[:5]}). "
                    f"두 집단 비교에는 2개여야 합니다. 3개 이상이면 compare_many_groups 를 쓰세요."
                )
            a = work[work[group_column].astype(str) == levels[0]][value_column]
            b = work[work[group_column].astype(str) == levels[1]][value_column]

            chosen = method
            if method == "auto":
                pa = stats.shapiro(a[:5000]).pvalue if len(a) >= 3 else 0.0
                pb = stats.shapiro(b[:5000]).pvalue if len(b) >= 3 else 0.0
                chosen = "t" if (pa >= 0.05 and pb >= 0.05) else "mannwhitney"

            if chosen == "t":
                res = stats.ttest_ind(a, b, equal_var=False)      # Welch
                test_name = "Welch independent t-test"
                # Hedges 보정 없는 Cohen's d (pooled sd)
                sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                             / (len(a) + len(b) - 2))
                effect = {"name": "Cohen's d", "value": round(float((a.mean() - b.mean()) / sp), 4)}
            else:
                res = stats.mannwhitneyu(a, b, alternative="two-sided")
                test_name = "Mann-Whitney U"
                # rank-biserial correlation
                r = 1 - (2 * res.statistic) / (len(a) * len(b))
                effect = {"name": "rank-biserial r", "value": round(float(r), 4)}

            return {
                "test": test_name,
                "method_requested": method, "method_used": chosen,
                "groups": {levels[0]: {"n": int(len(a)), "mean": round(float(a.mean()), 4),
                                       "sd": round(float(a.std(ddof=1)), 4),
                                       "median": round(float(a.median()), 4)},
                           levels[1]: {"n": int(len(b)), "mean": round(float(b.mean()), 4),
                                       "sd": round(float(b.std(ddof=1)), 4),
                                       "median": round(float(b.median()), 4)}},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha,
                "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_paired",
            "같은 대상에서 두 번 측정한 값(전/후)을 비교한다. 대응표본 t검정 또는 Wilcoxon.",
            {"before_column": {"type": "str", "desc": "이전 측정값 열", "required": True},
             "after_column": {"type": "str", "desc": "이후 측정값 열", "required": True},
             "method": {"type": "str", "enum": ["auto", "t", "wilcoxon"]},
             "alpha": {"type": "float"}},
        )
        def _compare_paired(before_column: str, after_column: str,
                            method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(before_column, after_column)
            work = self.df[[before_column, after_column]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            before, after = work[before_column], work[after_column]
            diff = after - before

            chosen = method
            if method == "auto":
                chosen = "t" if stats.shapiro(diff[:5000]).pvalue >= 0.05 else "wilcoxon"

            if chosen == "t":
                res = stats.ttest_rel(before, after)
                test_name = "paired t-test"
                effect = {"name": "Cohen's dz",
                          "value": round(float(diff.mean() / diff.std(ddof=1)), 4)}
            else:
                res = stats.wilcoxon(before, after)
                test_name = "Wilcoxon signed-rank"
                effect = {"name": "median difference", "value": round(float(diff.median()), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "n_pairs": int(len(work)),
                "mean_before": round(float(before.mean()), 4),
                "mean_after": round(float(after.mean()), 4),
                "mean_difference": round(float(diff.mean()), 4),
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
            }

        # ---------------------------------------------------------------
        @self.tool(
            "compare_many_groups",
            "세 집단 이상의 평균(또는 분포)을 비교한다. 일원분산분석 또는 Kruskal-Wallis.",
            {"value_column": {"type": "str", "required": True},
             "group_column": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "anova", "kruskal"]},
             "alpha": {"type": "float"}},
        )
        def _compare_many_groups(value_column: str, group_column: str,
                                 method: str = "auto", alpha: float = 0.05) -> dict:
            self._check_columns(value_column, group_column)
            work = self.df[[value_column, group_column]].copy()
            work[value_column] = pd.to_numeric(work[value_column], errors="coerce")
            work = work.dropna()

            groups = [g[value_column].values for _, g in work.groupby(group_column)]
            labels = [str(k) for k, _ in work.groupby(group_column)]
            if len(groups) < 3:
                raise ValueError(
                    f"집단이 {len(groups)}개입니다. 3개 이상이어야 합니다. "
                    f"2개면 compare_two_groups 를 쓰세요."
                )

            chosen = method
            if method == "auto":
                정규 = all(len(g) >= 3 and stats.shapiro(g[:5000]).pvalue >= 0.05 for g in groups)
                등분산 = stats.levene(*groups).pvalue >= 0.05
                chosen = "anova" if (정규 and 등분산) else "kruskal"

            if chosen == "anova":
                res = stats.f_oneway(*groups)
                test_name = "one-way ANOVA"
                grand = np.concatenate(groups)
                ss_b = sum(len(g) * (g.mean() - grand.mean()) ** 2 for g in groups)
                ss_t = ((grand - grand.mean()) ** 2).sum()
                effect = {"name": "eta squared", "value": round(float(ss_b / ss_t), 4)}
            else:
                res = stats.kruskal(*groups)
                test_name = "Kruskal-Wallis"
                n = len(np.concatenate(groups))
                effect = {"name": "epsilon squared",
                          "value": round(float(res.statistic / ((n ** 2 - 1) / (n + 1))), 4)}

            return {
                "test": test_name, "method_used": chosen,
                "groups": {lab: {"n": int(len(g)), "mean": round(float(g.mean()), 4),
                                 "sd": round(float(g.std(ddof=1)), 4)}
                           for lab, g in zip(labels, groups)},
                "statistic": round(float(res.statistic), 4),
                "p_value": float(res.pvalue),
                "alpha": alpha, "significant": bool(res.pvalue < alpha),
                "effect_size": effect,
                "note": "유의하면 사후검정(Tukey 등)이 필요합니다.",
            }

        # ---------------------------------------------------------------
        @self.tool(
            "test_association",
            "두 범주형 변수의 연관성을 카이제곱 검정한다. 기대빈도가 5 미만이면 Fisher 를 권한다.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "alpha": {"type": "float"}},
        )
        def _test_association(column_a: str, column_b: str, alpha: float = 0.05) -> dict:
            self._check_columns(column_a, column_b)
            table = pd.crosstab(self.df[column_a], self.df[column_b])
            if table.size < 4:
                raise ValueError("분할표가 2x2 보다 작습니다.")
            chi2, p, dof, expected = stats.chi2_contingency(table)
            min_exp = float(expected.min())
            n = int(table.values.sum())
            cramers_v = float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))
            return {
                "test": "chi-square test of independence",
                "table": table.to_dict(),
                "chi2": round(float(chi2), 4), "df": int(dof),
                "p_value": float(p), "alpha": alpha,
                "significant": bool(p < alpha),
                "min_expected_count": round(min_exp, 2),
                "effect_size": {"name": "Cramer's V", "value": round(cramers_v, 4)},
                "warning": (None if min_exp >= 5 else
                            "기대빈도가 5 미만인 칸이 있습니다. Fisher 정확검정을 고려하세요."),
            }

        # ---------------------------------------------------------------
        @self.tool(
            "correlate",
            "두 수치형 변수의 상관을 구한다. method=auto 면 정규성에 따라 pearson/spearman 선택.",
            {"column_a": {"type": "str", "required": True},
             "column_b": {"type": "str", "required": True},
             "method": {"type": "str", "enum": ["auto", "pearson", "spearman"]}},
        )
        def _correlate(column_a: str, column_b: str, method: str = "auto") -> dict:
            self._check_columns(column_a, column_b)
            work = self.df[[column_a, column_b]].apply(
                lambda c: pd.to_numeric(c, errors="coerce")).dropna()
            if len(work) < 3:
                raise ValueError(f"두 열이 모두 있는 행이 {len(work)}개뿐입니다.")
            a, b = work[column_a], work[column_b]

            chosen = method
            if method == "auto":
                정규 = (stats.shapiro(a[:5000]).pvalue >= 0.05
                        and stats.shapiro(b[:5000]).pvalue >= 0.05)
                chosen = "pearson" if 정규 else "spearman"

            fn = stats.pearsonr if chosen == "pearson" else stats.spearmanr
            res = fn(a, b)
            return {
                "method_used": chosen, "n": int(len(work)),
                "coefficient": round(float(res[0]), 4),
                "p_value": float(res[1]),
                "note": "상관은 인과가 아닙니다.",
            }


In [ ]:
%%writefile react.py
"""전산실험2 6주차 : ReAct 루프 — 에이전트의 심장.

    Thought  → 지금 무엇을 알아야 하는가
    Action   → 어떤 도구를 어떤 인자로 부를 것인가
    Observation → 도구가 돌려준 결과 (실패도 관찰이다)
        ↓ 목표가 달성될 때까지 반복

이 구조는 3주차 `ask_json()` 의 되먹임 루프와 같다.
달라진 것은 "관찰"이 파싱 결과가 아니라 **도구 실행 결과**라는 점뿐이다.

설계에서 중요한 것 세 가지:
    1. 종료 조건을 여러 겹으로 둔다 (정답 / 스텝 상한 / 연속 실패 / 반복 감지)
    2. 실패를 루프 밖으로 던지지 않는다
    3. 모든 단계를 기록한다 (Trace) — 11주차 평가의 재료가 된다

사용 예:
    import pandas as pd
    from kmu_llm import LLM
    from tools import StatToolbox
    from react import ReActAgent

    box = StatToolbox(pd.read_csv("sample_clinic.csv"))
    agent = ReActAgent(LLM(), box, max_steps=8)
    trace = agent.run("치료 전후로 혈압이 유의하게 떨어졌나요?")
    print(trace.final_answer)
    trace.show()
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

__all__ = ["Step", "Trace", "ReActAgent", "REACT_SYSTEM"]


# --------------------------------------------------------------------------
# 실행 기록
# --------------------------------------------------------------------------
@dataclass
class Step:
    """한 사이클의 기록."""

    n: int
    thought: str = ""
    action: str | None = None
    args: dict[str, Any] = field(default_factory=dict)
    observation: str = ""
    ok: bool = False
    raw: str = ""           # 모델의 원문 (디버깅용)
    error: str | None = None

    def summary(self) -> str:
        head = f"[{self.n}] {self.action}({json.dumps(self.args, ensure_ascii=False)})"
        mark = "OK" if self.ok else "실패"
        return f"{head}  -> {mark}"


@dataclass
class Trace:
    """한 번의 `run()` 전체 기록.

    11주차에서 이 객체를 모아 성능을 추정한다.
    """

    goal: str
    steps: list[Step] = field(default_factory=list)
    final_answer: str | None = None
    stop_reason: str = ""
    success: bool = False

    def __len__(self) -> int:
        return len(self.steps)

    @property
    def tools_used(self) -> list[str]:
        return [s.action for s in self.steps if s.action]

    @property
    def n_failed(self) -> int:
        return sum(1 for s in self.steps if not s.ok)

    def show(self, max_obs: int = 220) -> None:
        """사람이 읽는 실행 경로."""
        print(f"목표: {self.goal}")
        print("=" * 70)
        for s in self.steps:
            print(f"[{s.n}] 생각: {s.thought[:150]}")
            print(f"     행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            obs = s.observation.replace("\n", " ")
            print(f"     관찰: {'OK  ' if s.ok else '실패 '}{obs[:max_obs]}")
            print()
        print("=" * 70)
        print(f"종료 사유: {self.stop_reason}  |  스텝 {len(self.steps)}  |  실패 {self.n_failed}")
        print(f"최종 답변: {self.final_answer}")

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal": self.goal,
            "steps": [
                {"n": s.n, "thought": s.thought, "action": s.action,
                 "args": s.args, "ok": s.ok, "observation": s.observation[:500]}
                for s in self.steps
            ],
            "final_answer": self.final_answer,
            "stop_reason": self.stop_reason,
            "success": self.success,
        }


# --------------------------------------------------------------------------
# 프롬프트
# --------------------------------------------------------------------------
REACT_SYSTEM = """\
너는 통계 분석 에이전트다. 도구를 하나씩 사용해 사용자의 질문에 답한다.

절대 규칙:
- 너는 계산하지 않는다. 평균, 표준편차, p값은 반드시 도구로 구한다.
- 한 번에 도구 하나만 부른다.
- 도구가 돌려준 값만 사실로 인정한다. 추측한 숫자를 말하지 않는다.

매 단계마다 아래 형식의 JSON 하나만 출력한다. 다른 말은 하지 않는다.

행동할 때:
{"thought": "왜 이 도구가 필요한지 한 문장", "action": "도구이름", "args": {...}}

답이 충분히 모였을 때:
{"thought": "왜 끝내는지 한 문장", "action": "finish", "args": {"answer": "최종 답변"}}

권장 순서:
1. list_columns 로 실제 열 이름과 결측 상황을 먼저 확인한다.
2. 필요하면 describe_column / check_normality 로 데이터를 진단한다.
3. 적절한 검정 도구를 부른다.
4. finish 로 결과를 해석해 답한다. p값과 효과크기를 함께 언급한다."""


def build_prompt(goal: str, toolbox: Any, steps: list[Step], max_steps: int) -> str:
    """이번 단계에 보낼 user 메시지를 만든다."""
    parts = [
        f"## 사용 가능한 도구\n{toolbox.describe()}\n",
        f'## 질문\n{goal}\n',
    ]

    if steps:
        parts.append("## 지금까지의 진행")
        for s in steps:
            parts.append(f"[{s.n}] 생각: {s.thought}")
            parts.append(f"    행동: {s.action}({json.dumps(s.args, ensure_ascii=False)})")
            parts.append(f"    관찰: {s.observation}")
        parts.append("")

    남은 = max_steps - len(steps)
    if 남은 <= 2:
        parts.append(
            f"** 남은 단계가 {남은}번뿐이다. 지금까지 얻은 정보로 finish 하라. **\n"
        )
    parts.append("다음에 할 일을 JSON 하나로 출력하라.")
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 에이전트
# --------------------------------------------------------------------------
class ReActAgent:
    """Thought → Action → Observation 을 반복한다.

    Parameters
    ----------
    llm : LLM 인스턴스
    toolbox : ToolRegistry (보통 StatToolbox)
    max_steps : 스텝 상한. 무한 루프 방지
    max_consecutive_failures : 연속 실패가 이만큼이면 포기
    detect_repeat : 같은 (도구, 인자) 를 반복하면 경고를 관찰로 넣는다
    verbose : 진행 상황 출력
    """

    def __init__(
        self,
        llm: Any,
        toolbox: Any,
        *,
        max_steps: int = 8,
        max_consecutive_failures: int = 3,
        detect_repeat: bool = True,
        verbose: bool = True,
        system: str = REACT_SYSTEM,
    ) -> None:
        self.llm = llm
        self.toolbox = toolbox
        self.max_steps = max_steps
        self.max_consecutive_failures = max_consecutive_failures
        self.detect_repeat = detect_repeat
        self.verbose = verbose
        self.system = system

    # ----------------------------------------------------------------
    def run(self, goal: str, **kw: Any) -> Trace:
        """목표를 받아 끝까지 돌린다. **예외를 밖으로 던지지 않는다.**"""
        from structured import JSONParseError, parse_json

        trace = Trace(goal=goal)
        연속실패 = 0
        본적있는행동: set[str] = set()

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 600)

        while len(trace.steps) < self.max_steps:
            n = len(trace.steps) + 1
            step = Step(n=n)

            # --- 1) 생각하고 행동을 정한다 ------------------------------
            prompt = build_prompt(goal, self.toolbox, trace.steps, self.max_steps)
            try:
                step.raw = self.llm.chat(
                    [{"role": "system", "content": self.system},
                     {"role": "user", "content": prompt}], **kw
                )
            except Exception as exc:
                step.error = f"모델 호출 실패: {type(exc).__name__}: {exc}"
                step.observation = step.error
                trace.steps.append(step)
                연속실패 += 1
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "모델 호출이 계속 실패함"
                    break
                continue

            try:
                결정 = parse_json(step.raw)
                if not isinstance(결정, dict):
                    raise JSONParseError("최상위가 객체가 아닙니다.")
            except JSONParseError as exc:
                step.error = str(exc)
                step.observation = (
                    f"[형식 오류] 응답을 JSON으로 읽을 수 없다: {exc}\n"
                    '{"thought": ..., "action": ..., "args": {...}} 형식으로만 출력하라.'
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print(f"[{n}] 형식 오류 -> 되먹임")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "형식 오류가 계속됨"
                    break
                continue

            step.thought = str(결정.get("thought", ""))[:400]
            step.action = 결정.get("action")
            step.args = 결정.get("args") or {}
            if not isinstance(step.args, dict):
                step.args = {}

            if self.verbose:
                print(f"[{n}] 생각: {step.thought[:90]}")
                print(f"     행동: {step.action}({json.dumps(step.args, ensure_ascii=False)[:110]})")

            # --- 2) 종료 행동인가 --------------------------------------
            if step.action == "finish":
                step.ok = True
                step.observation = "(종료)"
                trace.steps.append(step)
                trace.final_answer = str(step.args.get("answer", "")).strip()
                trace.stop_reason = "모델이 finish 를 선택"
                trace.success = bool(trace.final_answer)
                if self.verbose:
                    print(f"     -> 종료\n")
                break

            # --- 3) 같은 행동을 반복하는가 ------------------------------
            열쇠 = f"{step.action}|{json.dumps(step.args, ensure_ascii=False, sort_keys=True)}"
            if self.detect_repeat and 열쇠 in 본적있는행동:
                step.ok = False
                step.observation = (
                    "[반복 감지] 방금 전과 똑같은 도구를 똑같은 인자로 다시 불렀다. "
                    "이미 그 결과는 위에 있다. 다른 도구를 쓰거나 finish 하라."
                )
                trace.steps.append(step)
                연속실패 += 1
                if self.verbose:
                    print("     -> 반복 감지\n")
                if 연속실패 >= self.max_consecutive_failures:
                    trace.stop_reason = "같은 행동을 반복함"
                    break
                continue
            본적있는행동.add(열쇠)

            # --- 4) 도구를 실행하고 관찰한다 ----------------------------
            result = self.toolbox.call(step.action, step.args)
            step.ok = result.ok
            step.observation = result.observation()
            step.error = result.error
            trace.steps.append(step)

            if self.verbose:
                print(f"     관찰: {'OK' if step.ok else '실패'} "
                      f"{step.observation.replace(chr(10), ' ')[:110]}\n")

            연속실패 = 0 if step.ok else 연속실패 + 1
            if 연속실패 >= self.max_consecutive_failures:
                trace.stop_reason = f"도구 실행이 {연속실패}회 연속 실패"
                break

        # --- 루프를 다 돌았는데 finish 가 없었다면 -----------------------
        if trace.final_answer is None and not trace.stop_reason:
            trace.stop_reason = f"스텝 상한({self.max_steps}) 도달"

        if trace.final_answer is None:
            trace.final_answer = self._force_answer(trace, **kw)

        return trace

    # ----------------------------------------------------------------
    def _force_answer(self, trace: Trace, **kw: Any) -> str:
        """finish 없이 끝났을 때, 지금까지의 관찰로 답을 만들어 본다.

        빈손으로 끝내는 것보다는 낫지만, **성공으로 치지는 않는다.**
        `trace.success` 는 False 로 남는다.
        """
        성공관찰 = [s.observation for s in trace.steps if s.ok]
        if not 성공관찰:
            return "(도구 실행에 모두 실패해 답할 수 없습니다.)"

        모음 = "\n\n".join(성공관찰)[:3000]
        try:
            return self.llm.ask(
                f"질문: {trace.goal}\n\n"
                f"아래는 도구 실행 결과다. 이것만 근거로 답하라. "
                f"근거가 부족하면 부족하다고 말하라. 숫자를 지어내지 마라.\n\n{모음}",
                system="너는 통계 분석 결과를 해석하는 도구다.",
                temperature=0.0,
                max_tokens=kw.get("max_tokens", 600),
            )
        except Exception as exc:
            return f"(답변 생성 실패: {type(exc).__name__})"


In [ ]:
%%writefile executor.py
"""전산실험2 7주차 : 코드 실행 에이전트.

도구를 미리 만들어 두는 대신, 모델이 분석 코드를 **직접 써서 실행**한다.
오류가 나면 Traceback 을 되먹여 고치게 한다.

    코드 생성 → 검사 → 실행 → 결과 관찰 → 실패면 Traceback 되먹임
                                              ↑ 3주차·6주차와 같은 구조

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 안전에 관한 정직한 경고

 여기 있는 `SafeExecutor` 는 **진짜 샌드박스가 아니다.**
 AST 검사와 제한된 네임스페이스는 "실수로 사고 치는 것"을 막는 장치이지,
 작정한 공격을 막는 장치가 아니다. 파이썬에서 언어 수준의 완전한 격리는
 사실상 불가능하다는 것이 알려져 있다.

 따라서 이 코드는 다음 전제에서만 쓴다.
   - 실행 환경이 **격리된 일회용 VM** 이다 (Colab 런타임이 그렇다)
   - 코드를 만드는 주체가 **우리가 통제하는 모델**이다
   - 그 안에 **중요한 자격증명이 없다**

 실제 서비스에서 사용자 입력으로 코드를 실행해야 한다면 컨테이너,
 gVisor, Firecracker 같은 OS 수준 격리를 쓴다. 파이썬 코드로 막지 않는다.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    import pandas as pd
    from kmu_llm import LLM
    from executor import SafeExecutor, CodeAgent

    ex = SafeExecutor({"df": pd.read_csv("sample_clinic.csv")})
    print(ex.run("result = df['bmi'].mean()").summary())

    agent = CodeAgent(LLM(), ex)
    out = agent.run("치료군과 대조군의 BMI 평균을 비교하고 p값을 구하라")
    print(out.result)
"""

from __future__ import annotations

import ast
import contextlib
import io
import signal
import traceback
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "ALLOWED_IMPORTS",
    "CodeCheckError",
    "ExecResult",
    "SafeExecutor",
    "CodeAgent",
    "CodeRun",
]

# 허용할 모듈. 화이트리스트 방식 — 목록에 없으면 막는다.
# 블랙리스트("os 를 막자")는 반드시 빠뜨리는 것이 생긴다.
ALLOWED_IMPORTS = {
    "numpy", "np",
    "pandas", "pd",
    "scipy", "scipy.stats",
    "statsmodels", "statsmodels.api", "statsmodels.formula.api",
    "statsmodels.stats.multicomp", "statsmodels.stats.proportion",
    "sklearn", "sklearn.linear_model", "sklearn.metrics",
    "matplotlib", "matplotlib.pyplot",
    "math", "statistics", "itertools", "collections", "json", "re",
}

# 이름만 나와도 막을 것들
FORBIDDEN_NAMES = {
    "__import__", "eval", "exec", "compile", "open", "input",
    "globals", "locals", "vars", "getattr", "setattr", "delattr",
    "exit", "quit", "breakpoint", "memoryview",
}

# 속성 접근으로 우회하는 전형적인 수법
FORBIDDEN_ATTRS = {
    "__globals__", "__builtins__", "__subclasses__", "__bases__",
    "__mro__", "__class__", "__code__", "__closure__", "__reduce__",
    "__getattribute__", "__dict__", "__loader__", "__spec__",
}


class CodeCheckError(ValueError):
    """실행 전 검사에서 걸렸을 때."""


# --------------------------------------------------------------------------
# 정적 검사
# --------------------------------------------------------------------------
def check_code(source: str) -> list[str]:
    """실행하기 전에 코드를 훑어 위험한 것을 찾는다.

    파싱해서 AST 를 보는 이유: 문자열 검색(`"os" in code`)은
    `"o" + "s"` 같은 것에 뚫린다. 구문 트리는 그런 속임수에 강하다.

    Returns
    -------
    list[str] : 문제 목록 (빈 목록이면 통과)
    """
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return [f"문법 오류: {exc.msg} ({exc.lineno}행)"]

    problems: list[str] = []

    for node in ast.walk(tree):
        # import 검사
        if isinstance(node, ast.Import):
            for alias in node.names:
                root = alias.name.split(".")[0]
                if alias.name not in ALLOWED_IMPORTS and root not in ALLOWED_IMPORTS:
                    problems.append(
                        f"'{alias.name}' 모듈은 허용되지 않습니다. "
                        f"허용 목록: {sorted(ALLOWED_IMPORTS)}"
                    )
        elif isinstance(node, ast.ImportFrom):
            mod = node.module or ""
            root = mod.split(".")[0]
            if mod not in ALLOWED_IMPORTS and root not in ALLOWED_IMPORTS:
                problems.append(f"'{mod}' 모듈은 허용되지 않습니다.")

        # 금지된 이름 호출
        elif isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            problems.append(f"'{node.id}' 는 사용할 수 없습니다.")

        # 던더 속성 접근 (샌드박스 탈출의 전형적 경로)
        elif isinstance(node, ast.Attribute) and node.attr in FORBIDDEN_ATTRS:
            problems.append(f"'.{node.attr}' 속성 접근은 허용되지 않습니다.")

    return problems


# --------------------------------------------------------------------------
# 실행 결과
# --------------------------------------------------------------------------
@dataclass
class ExecResult:
    """코드 한 번 실행의 결과."""

    ok: bool
    code: str
    stdout: str = ""
    result: Any = None            # 코드가 `result` 변수에 담은 값
    error: str | None = None      # 예외 타입과 메시지
    traceback: str = ""           # 모델에게 되먹일 전체 Traceback
    blocked: list[str] = field(default_factory=list)   # 검사에서 걸린 이유
    figures: int = 0              # 그려진 그림 수

    def observation(self, max_chars: int = 1500) -> str:
        """모델에게 돌려줄 관찰 문자열."""
        if self.blocked:
            return ("[실행 거부] 코드가 안전 검사를 통과하지 못했습니다:\n"
                    + "\n".join(f"- {b}" for b in self.blocked)
                    + "\n허용된 모듈만 쓰고, 파일·시스템 접근을 하지 마세요.")

        if not self.ok:
            tb = self.traceback[-max_chars:] if self.traceback else str(self.error)
            return f"[실행 실패]\n{tb}\n\n위 오류를 고쳐서 코드를 다시 작성하세요."

        parts = []
        if self.stdout.strip():
            parts.append(f"[출력]\n{self.stdout.strip()[:max_chars]}")
        if self.result is not None:
            parts.append(f"[result]\n{str(self.result)[:max_chars]}")
        if self.figures:
            parts.append(f"[그림 {self.figures}개 생성됨]")
        if not parts:
            parts.append("[실행 성공했지만 출력이 없습니다. "
                         "print() 를 쓰거나 result 변수에 값을 담으세요.]")
        return "\n\n".join(parts)

    def summary(self) -> str:
        if self.blocked:
            return f"거부됨: {self.blocked[0]}"
        if not self.ok:
            return f"실패: {self.error}"
        return f"성공 (출력 {len(self.stdout)}자, result={type(self.result).__name__})"


# --------------------------------------------------------------------------
# 실행기
# --------------------------------------------------------------------------
class SafeExecutor:
    """제한된 네임스페이스에서 코드를 실행한다.

    **진짜 샌드박스가 아니다.** 모듈 상단의 경고를 읽을 것.

    Parameters
    ----------
    context : 코드에서 쓸 수 있게 미리 넣어 줄 변수들 (보통 {"df": DataFrame})
    timeout : 초 단위 상한. POSIX(Colab 포함)에서만 동작한다.
    persist : True 면 실행 사이에 변수가 유지된다 (여러 단계 분석에 유용)
    """

    def __init__(
        self,
        context: dict[str, Any] | None = None,
        *,
        timeout: int = 30,
        persist: bool = True,
    ) -> None:
        self.timeout = timeout
        self.persist = persist
        self.context = dict(context or {})
        self.namespace: dict[str, Any] = {}
        self.history: list[ExecResult] = []
        self.reset()

    # ----------------------------------------------------------------
    def reset(self) -> None:
        """네임스페이스를 초기 상태로 되돌린다."""
        import numpy as np
        import pandas as pd
        from scipy import stats

        ns: dict[str, Any] = {
            "np": np, "numpy": np,
            "pd": pd, "pandas": pd,
            "stats": stats,
        }
        try:
            import matplotlib

            matplotlib.use("Agg")           # 화면 없이 그림만 만든다
            import matplotlib.pyplot as plt

            ns["plt"] = plt
        except Exception:
            pass

        ns.update(self.context)
        ns["result"] = None
        self.namespace = ns

    def add_context(self, **kw: Any) -> None:
        self.context.update(kw)
        self.namespace.update(kw)

    # ----------------------------------------------------------------
    def run(self, code: str) -> ExecResult:
        """코드를 검사하고 실행한다. **예외를 밖으로 던지지 않는다.**"""
        problems = check_code(code)
        if problems:
            res = ExecResult(ok=False, code=code, blocked=problems,
                             error="안전 검사 실패")
            self.history.append(res)
            return res

        if not self.persist:
            self.reset()
        self.namespace["result"] = None

        n_fig_before = self._count_figures()
        buf = io.StringIO()
        res = ExecResult(ok=False, code=code)

        try:
            with self._time_limit(self.timeout):
                with contextlib.redirect_stdout(buf):
                    exec(compile(code, "<agent_code>", "exec"), self.namespace)
            res.ok = True
            res.result = self.namespace.get("result")
        except TimeoutError as exc:
            res.error = str(exc)
            res.traceback = str(exc)
        except Exception as exc:
            res.error = f"{type(exc).__name__}: {exc}"
            res.traceback = self._clean_traceback(code)
        finally:
            res.stdout = buf.getvalue()
            res.figures = max(0, self._count_figures() - n_fig_before)

        self.history.append(res)
        return res

    # ----------------------------------------------------------------
    @staticmethod
    def _count_figures() -> int:
        try:
            import matplotlib.pyplot as plt

            return len(plt.get_fignums())
        except Exception:
            return 0

    @staticmethod
    def _clean_traceback(code: str = "") -> str:
        """모델이 고쳐야 할 부분만 남긴 Traceback 을 만든다.

        그냥 `format_exc()` 를 주면 실행기 내부 프레임과 pandas·numpy 내부
        프레임이 잔뜩 딸려 온다. 모델은 그것을 고칠 수 없고, 토큰만 먹고,
        정작 자기 코드의 몇 번째 줄이 문제인지는 묻힌다.

        그래서 **`<agent_code>` 프레임만** 남기고 마지막 예외 메시지를 붙인다.
        """
        import sys as _sys

        exc_type, exc_value, tb = _sys.exc_info()
        if exc_type is None:
            return ""

        코드줄 = code.splitlines()
        프레임 = []
        for frame, lineno in traceback.walk_tb(tb):
            if frame.f_code.co_filename != "<agent_code>":
                continue                      # 라이브러리·실행기 내부는 버린다
            본문 = 코드줄[lineno - 1].strip() if 0 < lineno <= len(코드줄) else ""
            프레임.append(f"  {lineno}행:  {본문}")

        머리 = "Traceback (내가 작성한 코드 안에서):"
        꼬리 = "".join(traceback.format_exception_only(exc_type, exc_value)).strip()

        if not 프레임:
            # 문법 오류 등으로 프레임이 안 잡히는 경우
            return 꼬리
        return "\n".join([머리, *프레임, "", 꼬리])

    @staticmethod
    @contextlib.contextmanager
    def _time_limit(seconds: int):
        """POSIX 에서만 동작하는 실행 시간 제한.

        Windows 에는 `SIGALRM` 이 없어 그냥 통과시킨다.
        **무한 루프를 만나면 셀이 멈춘다는 뜻이다.** 정직하게 알려 둔다.
        Colab 은 리눅스이므로 수업 중에는 동작한다.
        """
        if not hasattr(signal, "SIGALRM") or seconds <= 0:
            yield
            return

        def _handler(signum, frame):
            raise TimeoutError(f"실행이 {seconds}초를 넘겨 중단했습니다. "
                               f"무한 루프이거나 계산이 너무 무겁습니다.")

        old = signal.signal(signal.SIGALRM, _handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)


# --------------------------------------------------------------------------
# 코드 작성 에이전트
# --------------------------------------------------------------------------
CODE_SYSTEM = """\
너는 통계 분석 코드를 작성하는 도구다. 파이썬 코드만 출력한다.

사용 가능한 것:
- df : 분석 대상 DataFrame (이미 로드되어 있다)
- pd, np, stats (scipy.stats), plt (matplotlib.pyplot)

규칙:
- 코드만 출력한다. 설명, 인사, 마크다운 코드블록 표시를 쓰지 않는다.
- 최종 결과를 반드시 `result` 변수에 담는다. 딕셔너리를 권장한다.
- 중간 과정은 print() 로 보여준다.
- 파일을 읽거나 쓰지 않는다. 인터넷에 접속하지 않는다.
- 허용되지 않은 모듈을 import 하지 않는다.
- 숫자를 지어내지 않는다. 반드시 계산해서 얻는다.

예시:
g1 = df[df['group'] == 'treatment']['bmi'].dropna()
g2 = df[df['group'] == 'control']['bmi'].dropna()
t, p = stats.ttest_ind(g1, g2, equal_var=False)
result = {'test': 'Welch t-test', 'n1': len(g1), 'n2': len(g2),
          'statistic': round(float(t), 4), 'p_value': float(p)}
print(result)"""


@dataclass
class CodeRun:
    """코드 작성 에이전트의 실행 기록."""

    goal: str
    attempts: list[ExecResult] = field(default_factory=list)
    result: Any = None
    success: bool = False
    stop_reason: str = ""

    @property
    def n_attempts(self) -> int:
        return len(self.attempts)

    @property
    def final_code(self) -> str:
        return self.attempts[-1].code if self.attempts else ""

    def show(self) -> None:
        print(f"목표: {self.goal}")
        print("=" * 70)
        for i, a in enumerate(self.attempts, 1):
            print(f"--- 시도 {i} ---")
            print(a.code[:600])
            print(f">>> {a.summary()}")
            if not a.ok:
                print((a.traceback or "")[:400])
            print()
        print("=" * 70)
        print(f"{self.stop_reason}  |  시도 {self.n_attempts}회  |  성공 {self.success}")
        print(f"결과: {self.result}")


class CodeAgent:
    """코드를 쓰고, 실행하고, 실패하면 Traceback 을 보고 고친다.

    6주차 ReAct 루프와 같은 구조다. 관찰이 '도구 실행 결과' 에서
    '코드 실행 결과' 로 바뀌었을 뿐이다.
    """

    def __init__(
        self,
        llm: Any,
        executor: SafeExecutor,
        *,
        max_attempts: int = 4,
        verbose: bool = True,
        system: str = CODE_SYSTEM,
    ) -> None:
        self.llm = llm
        self.executor = executor
        self.max_attempts = max_attempts
        self.verbose = verbose
        self.system = system

    # ----------------------------------------------------------------
    @staticmethod
    def extract_code(text: str) -> str:
        """모델 응답에서 코드만 뽑아낸다.

        3주차 `extract_json()` 과 같은 발상이다.
        지시해도 코드블록으로 감싸는 일이 흔하다.
        """
        import re

        fence = re.search(r"```(?:python|py)?\s*(.*?)```", text, re.DOTALL)
        if fence:
            return fence.group(1).strip()
        return text.strip()

    def _data_hint(self) -> str:
        """df 의 모양을 프롬프트에 넣어 준다.

        5주차에서 확인했듯, 실제 열 이름을 보여주면 오류가 크게 준다.
        """
        df = self.executor.namespace.get("df")
        if df is None:
            return ""
        try:
            정보 = [f"- {c} ({df[c].dtype}, 결측 {int(df[c].isna().sum())})"
                    for c in df.columns]
            head = df.head(3).to_string()
            return (f"\n## 데이터 (df)\n행 {len(df)}개, 열 {len(df.columns)}개\n"
                    + "\n".join(정보) + f"\n\n앞 3행:\n{head}\n")
        except Exception:
            return ""

    # ----------------------------------------------------------------
    def run(self, goal: str, **kw: Any) -> CodeRun:
        """목표를 받아 코드를 쓰고 실행한다. 실패하면 고쳐서 다시."""
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 900)

        run = CodeRun(goal=goal)
        messages = [
            {"role": "system", "content": self.system},
            {"role": "user", "content": f"{self._data_hint()}\n## 요청\n{goal}\n\n코드만 출력하라."},
        ]

        for attempt in range(1, self.max_attempts + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                run.stop_reason = f"모델 호출 실패: {type(exc).__name__}"
                return run

            code = self.extract_code(원문)
            if self.verbose:
                print(f"--- 시도 {attempt} ---")
                print(code[:400])

            res = self.executor.run(code)
            run.attempts.append(res)

            if self.verbose:
                print(f">>> {res.summary()}\n")

            if res.ok:
                run.result = res.result
                run.success = True
                run.stop_reason = f"{attempt}회 만에 성공"
                return run

            # 실패 -> Traceback 을 되먹인다
            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": res.observation()})

        run.stop_reason = f"{self.max_attempts}회 모두 실패"
        return run


In [ ]:
%%writefile profiler.py
"""전산실험2 9주차 : 데이터 진단과 분석계획 수립.

7주차까지 우리는 **깨끗한 데이터를 전제**했다. 실제 데이터는 그렇지 않다.
`-999` 가 결측이고, `'12.5kg'` 이 수치이고, `Yes/yes/Y` 가 같은 값이다.

이 모듈은 두 가지를 한다.

    DataProfiler   데이터를 훑어 타입을 추론하고 문제를 찾아낸다 (규칙 기반)
    AnalysisPlanner LLM이 분석 계획을 세우되, **규칙이 검증한다**

핵심 설계 원칙:
    **무엇을 LLM에게 맡기고 무엇을 규칙으로 고정할 것인가.**

    규칙으로 고정  : 검정의 전제 조건 (집단 수, 대응 여부, 척도)
                    -> 틀리면 결과 자체가 무의미하므로 코드가 강제한다
    LLM에게 위임   : 변수의 의미 파악, 사용자 질문 해석, 결과 서술
                    -> 규칙으로 적을 수 없는 판단

사용 예:
    import pandas as pd
    from profiler import DataProfiler

    prof = DataProfiler(pd.read_csv("exam_greenhouse.csv"))
    print(prof.report())
    clean = prof.clean()
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "SENTINELS",
    "ColumnProfile",
    "DataProfiler",
    "TEST_RULES",
    "validate_plan",
    "AnalysisPlanner",
]

# 결측을 뜻하는 관례적 값들. 센서·설문 데이터에서 흔하다.
SENTINELS = {-999, -9999, -99, 999, 9999, -1e30}
SENTINEL_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "결측", "무응답"}

_UNIT_SUFFIX = re.compile(r"^\s*([-+]?[\d.,]+)\s*[a-zA-Z가-힣%°]+\s*$")
_DECIMAL_COMMA = re.compile(r"^\s*[-+]?\d{1,3}(,\d{1,2})\s*$")

# 같은 뜻인데 표기가 다른 값들. 설문·수기 입력에서 늘 나온다.
# 축약형(y/n)과 완전형(yes/no)이 한 열에 섞이면 대소문자 정규화만으로는
# 합쳐지지 않는다.
SYNONYM_FAMILIES: list[dict[str, str]] = [
    {"y": "yes", "yes": "yes", "true": "yes", "t": "yes", "1": "yes",
     "n": "no", "no": "no", "false": "no", "f": "no", "0": "no",
     "예": "yes", "아니오": "no", "아니요": "no"},
    {"m": "male", "male": "male", "남": "male", "남자": "male", "남성": "male",
     "f": "female", "female": "female", "여": "female", "여자": "female",
     "여성": "female"},
]


# --------------------------------------------------------------------------
# 열 하나의 진단 결과
# --------------------------------------------------------------------------
@dataclass
class ColumnProfile:
    name: str
    dtype: str
    n: int
    n_missing: int = 0
    n_unique: int = 0
    kind: str = "unknown"          # continuous / ordinal / nominal / id / binary
    issues: list[str] = field(default_factory=list)
    fixes: list[str] = field(default_factory=list)   # 적용 가능한 정리 방법
    stats: dict[str, Any] = field(default_factory=dict)
    levels: list[str] = field(default_factory=list)

    def line(self) -> str:
        문제 = f"  ⚠ {'; '.join(self.issues)}" if self.issues else ""
        return (f"- {self.name} [{self.kind}] "
                f"n={self.n - self.n_missing}/{self.n}, 고유 {self.n_unique}{문제}")


# --------------------------------------------------------------------------
# 데이터 진단
# --------------------------------------------------------------------------
class DataProfiler:
    """DataFrame 을 훑어 타입을 추론하고 문제를 찾아낸다.

    **전부 규칙 기반이다. LLM을 쓰지 않는다.**
    타입 추론과 결측 탐지는 규칙으로 충분히 정확하고,
    LLM에게 맡기면 느리고 비싸고 틀린다.
    """

    def __init__(self, df: Any, *, ordinal_max_levels: int = 10) -> None:
        self.df = df
        self.ordinal_max_levels = ordinal_max_levels
        self.profiles: dict[str, ColumnProfile] = {}
        self._scan()

    # ----------------------------------------------------------------
    def _scan(self) -> None:
        import pandas as pd

        n = len(self.df)
        for col in self.df.columns:
            s = self.df[col]
            p = ColumnProfile(name=col, dtype=str(s.dtype), n=n,
                              n_missing=int(s.isna().sum()),
                              n_unique=int(s.nunique(dropna=True)))

            문자 = s.dropna().astype(str)

            # --- 수치로 볼 수 있는가 -------------------------------
            숫자 = pd.to_numeric(s, errors="coerce")
            변환실패 = int(숫자.isna().sum() - s.isna().sum())
            수치비율 = 숫자.notna().sum() / max(n - p.n_missing, 1)

            # --- 문자열에 숨은 수치 패턴 ---------------------------
            단위붙음 = int(문자.apply(lambda v: bool(_UNIT_SUFFIX.match(v))).sum())
            쉼표소수 = int(문자.apply(lambda v: bool(_DECIMAL_COMMA.match(v))).sum())

            if 단위붙음:
                p.issues.append(f"단위가 붙은 값 {단위붙음}건 (예: '12.5kg')")
                p.fixes.append("strip_units")
            if 쉼표소수:
                p.issues.append(f"소수점에 쉼표를 쓴 값 {쉼표소수}건 (예: '3,5')")
                p.fixes.append("comma_decimal")

            수치가능 = 수치비율 > 0.7 or (단위붙음 + 쉼표소수) > 0

            # --- 결측 위장값 ---------------------------------------
            if 수치가능:
                유효 = 숫자.dropna()
                감지 = sorted({v for v in SENTINELS if (유효 == v).any()})
                if 감지:
                    개수 = int(유효.isin(감지).sum())
                    p.issues.append(f"결측 위장값 {감지} {개수}건 (실제 결측일 가능성)")
                    p.fixes.append("sentinel_to_nan")

            빈문자 = int(문자.str.strip().str.lower().isin(SENTINEL_STRINGS).sum())
            if 빈문자:
                p.issues.append(f"결측을 뜻하는 문자열 {빈문자}건")
                p.fixes.append("string_to_nan")

            # --- 타입 판정 -----------------------------------------
            if p.n_unique == n and n > 10:
                p.kind = "id"
            elif 수치가능:
                유효 = 숫자[~숫자.isin(SENTINELS)].dropna()
                if len(유효) >= 3:
                    p.stats = {
                        "mean": round(float(유효.mean()), 4),
                        "sd": round(float(유효.std(ddof=1)), 4),
                        "min": round(float(유효.min()), 4),
                        "median": round(float(유효.median()), 4),
                        "max": round(float(유효.max()), 4),
                        "skew": round(float(유효.skew()), 3),
                    }
                    self._check_outliers(p, 유효)
                    self._check_normality(p, 유효)
                # 정수이고 수준이 적으면 순서형일 수 있다
                정수형 = bool((유효 % 1 == 0).all()) if len(유효) else False
                if 정수형 and p.n_unique <= self.ordinal_max_levels:
                    p.kind = "ordinal"
                    p.issues.append(
                        f"정수 {p.n_unique}수준 — 순서형일 수 있음 "
                        f"(리커트 척도 등). 평균을 내는 것이 적절한지 확인할 것"
                    )
                else:
                    p.kind = "continuous"
                if 변환실패:
                    p.issues.append(f"수치로 못 바꾸는 값 {변환실패}건")
            else:
                정규화 = 문자.str.strip().str.lower()
                수준 = sorted(정규화.unique())
                p.levels = [str(v) for v in sorted(문자.unique())][:20]
                if len(수준) < p.n_unique:
                    p.issues.append(
                        f"대소문자·공백만 다른 값이 섞여 있음 "
                        f"(표기 {p.n_unique}종 → 실제 {len(수준)}종)"
                    )
                    p.fixes.append("normalize_case")

                # 축약형과 완전형이 섞였는가 (y / yes)
                family = self._match_synonym_family(수준)
                if family is not None:
                    합친수준 = sorted({family[v] for v in 수준})
                    if len(합친수준) < len(수준):
                        p.issues.append(
                            f"같은 뜻의 다른 표기가 섞여 있음 "
                            f"({수준} → {합친수준})"
                        )
                        p.fixes.append("merge_synonyms")
                        수준 = 합친수준
                p.kind = "binary" if len(수준) == 2 else "nominal"

            # --- 결측률 -------------------------------------------
            결측률 = p.n_missing / n if n else 0
            if 결측률 > 0.3:
                p.issues.append(f"결측률 {결측률:.0%} — 분석에서 제외를 고려")
            elif 결측률 > 0:
                p.issues.append(f"결측 {p.n_missing}건 ({결측률:.1%})")

            self.profiles[col] = p

    # ----------------------------------------------------------------
    @staticmethod
    def _match_synonym_family(levels: list[str]) -> dict[str, str] | None:
        """이 열의 값들이 알려진 동의어 집합에 전부 들어가는가.

        일부만 걸치면 `None` 을 돌려준다. 예를 들어 값이
        `['y', 'n', 'maybe']` 라면 함부로 합치지 않는다 —
        모르는 값이 섞여 있으면 손대지 않는 것이 안전하다.
        """
        for family in SYNONYM_FAMILIES:
            if all(v in family for v in levels):
                return family
        return None

    @staticmethod
    def _check_outliers(p: ColumnProfile, values: Any) -> None:
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr <= 0:
            return
        low, high = q1 - 3 * iqr, q3 + 3 * iqr      # 3배 = 극단값만
        극단 = values[(values < low) | (values > high)]
        if len(극단):
            p.issues.append(
                f"극단 이상치 {len(극단)}건 "
                f"(예: {[round(float(v), 1) for v in 극단.head(3)]}) — 입력 오류 가능"
            )
            p.stats["outlier_bounds"] = [round(float(low), 2), round(float(high), 2)]

    @staticmethod
    def _check_normality(p: ColumnProfile, values: Any) -> None:
        if len(values) < 3:
            return
        try:
            from scipy import stats

            pv = float(stats.shapiro(values[:5000]).pvalue)
            p.stats["shapiro_p"] = pv
            p.stats["normal_at_0.05"] = bool(pv >= 0.05)
        except Exception:
            pass

    # ----------------------------------------------------------------
    def report(self) -> str:
        """사람이 읽는 진단 보고서."""
        줄 = [f"데이터: {len(self.df)}행 {len(self.df.columns)}열", ""]
        for kind in ["id", "continuous", "ordinal", "binary", "nominal", "unknown"]:
            해당 = [p for p in self.profiles.values() if p.kind == kind]
            if not 해당:
                continue
            줄.append(f"[{kind}] {len(해당)}개")
            줄 += [p.line() for p in 해당]
            줄.append("")
        문제열 = [p.name for p in self.profiles.values() if p.fixes]
        if 문제열:
            줄.append(f"정리가 필요한 열: {문제열}")
        return "\n".join(줄)

    def to_prompt(self, max_cols: int = 30) -> str:
        """LLM 프롬프트에 넣을 압축 요약.

        `report()` 보다 짧다. 토큰 예산 때문이다 (4주차).
        """
        줄 = [f"행 {len(self.df)}개, 열 {len(self.df.columns)}개"]
        for p in list(self.profiles.values())[:max_cols]:
            부분 = f"- {p.name}: {p.kind}"
            if p.kind in ("binary", "nominal") and p.levels:
                부분 += f" {p.levels[:6]}"
            if p.stats.get("mean") is not None:
                부분 += f" (평균 {p.stats['mean']}, 결측 {p.n_missing})"
            if p.stats.get("normal_at_0.05") is False:
                부분 += " [정규성 기각]"
            if p.issues:
                부분 += f"  ※ {p.issues[0]}"
            줄.append(부분)
        return "\n".join(줄)

    def issues(self) -> dict[str, list[str]]:
        return {n: p.issues for n, p in self.profiles.items() if p.issues}

    # ----------------------------------------------------------------
    def clean(self, *, drop_outliers: bool = False, verbose: bool = True) -> Any:
        """진단에서 찾은 문제를 실제로 고친 DataFrame 을 돌려준다.

        **원본을 바꾸지 않는다.** 무엇을 고쳤는지 출력한다 —
        조용히 데이터를 바꾸는 것은 통계 분석에서 가장 위험한 일이다.
        """
        import numpy as np
        import pandas as pd

        out = self.df.copy()
        기록 = []

        for name, p in self.profiles.items():
            s = out[name]

            if "comma_decimal" in p.fixes:
                s = s.astype(str).str.replace(
                    r"^(\s*[-+]?\d{1,3}),(\d{1,2}\s*)$", r"\1.\2", regex=True)
                기록.append(f"{name}: 소수점 쉼표 → 마침표")

            if "strip_units" in p.fixes:
                s = s.astype(str).str.replace(r"\s*[a-zA-Z가-힣%°]+\s*$", "", regex=True)
                기록.append(f"{name}: 단위 문자 제거")

            if "string_to_nan" in p.fixes:
                s = s.replace(
                    {v: np.nan for v in SENTINEL_STRINGS}
                ).replace(r"^\s*$", np.nan, regex=True)
                기록.append(f"{name}: 결측 문자열 → NaN")

            if "normalize_case" in p.fixes:
                s = s.astype(str).str.strip().str.lower()
                기록.append(f"{name}: 대소문자·공백 정규화")

            if "merge_synonyms" in p.fixes:
                정규 = s.astype(str).str.strip().str.lower()
                family = self._match_synonym_family(sorted(정규.dropna().unique()))
                if family is not None:
                    전 = sorted(정규.unique())
                    s = 정규.map(family)
                    기록.append(f"{name}: 동의어 통합 {전} → {sorted(s.dropna().unique())}")

            if p.kind in ("continuous", "ordinal"):
                s = pd.to_numeric(s, errors="coerce")
                if "sentinel_to_nan" in p.fixes:
                    before = int(s.notna().sum())
                    s = s.mask(s.isin(SENTINELS))
                    기록.append(f"{name}: 결측 위장값 → NaN ({before - int(s.notna().sum())}건)")

                if drop_outliers and "outlier_bounds" in p.stats:
                    low, high = p.stats["outlier_bounds"]
                    before = int(s.notna().sum())
                    s = s.mask((s < low) | (s > high))
                    지운수 = before - int(s.notna().sum())
                    if 지운수:
                        기록.append(f"{name}: 극단 이상치 {지운수}건 → NaN")

            out[name] = s

        if verbose:
            if 기록:
                print("적용한 정리:")
                for r in 기록:
                    print(f"  - {r}")
            else:
                print("정리할 것이 없습니다.")
            if not drop_outliers:
                이상 = [n for n, p in self.profiles.items() if "outlier_bounds" in p.stats]
                if 이상:
                    print(f"\n[주의] 이상치가 있는 열: {이상}")
                    print("       drop_outliers=True 로 제거할 수 있으나,")
                    print("       제거 여부는 분석자가 판단할 일입니다.")
        return out


# --------------------------------------------------------------------------
# 검정 선택 규칙 — 도메인 지식을 코드로 고정한다
# --------------------------------------------------------------------------
TEST_RULES: dict[str, dict[str, Any]] = {
    "paired-t-test": {
        "설명": "같은 대상의 두 시점 비교, 차이가 정규",
        "outcome": ["continuous"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": True,
    },
    "wilcoxon-signed-rank": {
        "설명": "같은 대상의 두 시점 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": False,
    },
    "independent-t-test": {
        "설명": "서로 다른 두 집단 비교, 정규",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": True,
    },
    "mann-whitney-u": {
        "설명": "서로 다른 두 집단 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": False,
    },
    "one-way-anova": {
        "설명": "세 집단 이상 비교, 정규 + 등분산",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": True,
    },
    "kruskal-wallis": {
        "설명": "세 집단 이상 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": False,
    },
    "chi-square": {
        "설명": "두 범주형 변수의 연관성",
        "outcome": ["nominal", "binary"],
        "design": "independent",
        "n_groups": "any",
        "normality_required": False,
    },
    "pearson-correlation": {
        "설명": "두 연속형 변수의 선형 관계, 정규",
        "outcome": ["continuous"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": True,
    },
    "spearman-correlation": {
        "설명": "두 변수의 단조 관계, 정규성 불만족 또는 순서형",
        "outcome": ["continuous", "ordinal"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": False,
    },
}


def validate_plan(plan: dict[str, Any], profiler: DataProfiler) -> list[str]:
    """LLM이 세운 분석 계획을 **규칙으로 검증**한다.

    이것이 이 모듈의 핵심이다. LLM은 "이 질문에는 t검정이 맞겠다" 까지만
    하고, 그것이 **데이터의 실제 성질과 맞는지는 코드가 확인**한다.

    계획 형식::

        {
            "test": "independent-t-test",
            "outcome_column": "yield_after",
            "group_column": "fertilizer",     # 선택
            "second_column": "water_liters",  # 상관분석용, 선택
            "design": "independent",
            "reason": "...",
        }
    """
    문제: list[str] = []
    test = plan.get("test")

    if test not in TEST_RULES:
        return [f"'{test}' 는 알 수 없는 검정입니다. 가능: {sorted(TEST_RULES)}"]

    rule = TEST_RULES[test]
    prof = profiler.profiles

    # --- 결과변수 --------------------------------------------------
    outcome = plan.get("outcome_column")
    if not outcome:
        문제.append("outcome_column 이 없습니다.")
    elif outcome not in prof:
        문제.append(f"'{outcome}' 열이 데이터에 없습니다. 있는 열: {list(prof)}")
    else:
        kind = prof[outcome].kind
        if kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 결과변수가 {rule['outcome']} 이어야 하는데 "
                f"'{outcome}' 는 {kind} 입니다."
            )
        if kind == "id":
            문제.append(f"'{outcome}' 는 식별자로 보입니다. 분석 대상이 아닙니다.")

    # --- 집단변수와 집단 수 ----------------------------------------
    # 대응표본은 '집단 열'이 아니라 '두 개의 측정 열'을 쓴다.
    # n_groups=2 라는 표기에 낚여 group_column 을 요구하면 안 된다.
    group = plan.get("group_column")
    집단열_필요 = rule["design"] == "independent" and rule["n_groups"] in (2, "3+")
    if 집단열_필요:
        if not group:
            문제.append(f"{test} 에는 group_column 이 필요합니다.")
        elif group not in prof:
            문제.append(f"'{group}' 열이 데이터에 없습니다.")
        else:
            수준수 = prof[group].n_unique
            if rule["n_groups"] == 2 and 수준수 != 2:
                대안 = "one-way-anova / kruskal-wallis" if 수준수 > 2 else "(수준이 1개)"
                문제.append(
                    f"{test} 는 두 집단 비교인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ {대안} 를 고려하세요."
                )
            if rule["n_groups"] == "3+" and 수준수 < 3:
                문제.append(
                    f"{test} 는 세 집단 이상인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ independent-t-test / mann-whitney-u 를 고려하세요."
                )
            if prof[group].kind == "continuous":
                문제.append(
                    f"'{group}' 는 연속형입니다. 집단 변수로 쓰려면 범주화가 필요합니다."
                )

    # --- 대응 설계 --------------------------------------------------
    design = plan.get("design")
    if design and design != rule["design"]:
        문제.append(
            f"{test} 는 {rule['design']} 설계용인데 계획에는 {design} 이라고 되어 있습니다."
        )
    if rule["design"] == "paired" and not plan.get("second_column"):
        문제.append(
            "대응표본 검정에는 두 번째 측정 열(second_column)이 필요합니다."
        )

    # --- 상관분석 ---------------------------------------------------
    if rule["design"] == "correlation":
        second = plan.get("second_column")
        if not second:
            문제.append("상관분석에는 second_column 이 필요합니다.")
        elif second not in prof:
            문제.append(f"'{second}' 열이 데이터에 없습니다.")
        elif prof[second].kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 두 변수가 모두 {rule['outcome']} 이어야 하는데 "
                f"'{second}' 는 {prof[second].kind} 입니다."
            )

    # --- 정규성 -----------------------------------------------------
    if rule["normality_required"] and outcome in prof:
        정규 = prof[outcome].stats.get("normal_at_0.05")
        if 정규 is False:
            비모수 = {
                "paired-t-test": "wilcoxon-signed-rank",
                "independent-t-test": "mann-whitney-u",
                "one-way-anova": "kruskal-wallis",
                "pearson-correlation": "spearman-correlation",
            }.get(test, "비모수 검정")
            문제.append(
                f"'{outcome}' 는 정규성 검정에서 기각되었습니다 "
                f"(Shapiro p={prof[outcome].stats.get('shapiro_p', 0):.2e}). "
                f"→ {비모수} 를 고려하세요."
            )

    return 문제


# --------------------------------------------------------------------------
# 분석계획 수립기
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 도구다. 데이터 요약과 사용자 질문을 받아
어떤 검정을 어떤 변수로 수행할지 정한다.

너는 계산하지 않는다. 계획만 세운다.
데이터에 실제로 있는 열 이름만 쓴다. 없는 열을 지어내지 않는다.
확신이 없으면 reason 에 그렇게 적는다."""


class AnalysisPlanner:
    """LLM이 계획을 세우고, 규칙이 검증하고, 실패하면 되먹인다.

    3주차 `ask_json()` 과 같은 구조다. 달라진 것은 검증이
    **스키마 검사가 아니라 통계적 타당성 검사**라는 점이다.
    """

    def __init__(self, llm: Any, profiler: DataProfiler, *, n_retry: int = 3,
                 verbose: bool = True) -> None:
        self.llm = llm
        self.profiler = profiler
        self.n_retry = n_retry
        self.verbose = verbose

    def plan(self, question: str, **kw: Any) -> dict[str, Any]:
        """질문에 대한 분석 계획을 세운다.

        Returns
        -------
        dict : 검증을 통과한 계획.
               실패하면 {"test": None, "error": ..., "attempts": [...]}
        """
        from structured import JSONParseError, parse_json

        schema_desc = {
            "test": f"다음 중 하나: {sorted(TEST_RULES)}",
            "outcome_column": "분석할 결과변수 열 이름",
            "group_column": "집단 비교면 집단 열 이름, 아니면 null",
            "second_column": "대응표본의 두 번째 측정 열 또는 상관분석의 두 번째 변수, 아니면 null",
            "design": "paired / independent / correlation 중 하나",
            "reason": "이 검정을 고른 이유 한두 문장",
        }
        규칙설명 = "\n".join(
            f"- {k}: {v['설명']} (결과변수 {v['outcome']}, 설계 {v['design']}, 집단 {v['n_groups']})"
            for k, v in TEST_RULES.items()
        )

        base = (
            f"## 데이터 요약\n{self.profiler.to_prompt()}\n\n"
            f"## 선택 가능한 검정\n{규칙설명}\n\n"
            f"## 질문\n{question}\n\n"
            f"## 출력 형식 (JSON만)\n"
            + "\n".join(f'  "{k}": {v}' for k, v in schema_desc.items())
            + "\n\n설명이나 코드블록 없이 JSON 객체 하나만 출력하라."
        )

        messages = [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": base},
        ]
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)
        시도들 = []

        for attempt in range(1, self.n_retry + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                # 서버 문제로 루프 전체가 죽으면 안 된다 (6주차와 같은 원칙)
                시도들.append({"attempt": attempt, "error": f"{type(exc).__name__}: {exc}"})
                if self.verbose:
                    print(f"[{attempt}회] 모델 호출 실패: {type(exc).__name__}")
                if attempt == self.n_retry:
                    return {"test": None,
                            "error": f"모델 호출 실패: {type(exc).__name__}: {exc}",
                            "attempts": 시도들}
                continue

            try:
                계획 = parse_json(원문)
            except JSONParseError as exc:
                시도들.append({"attempt": attempt, "error": str(exc)})
                피드백 = f"JSON으로 읽을 수 없었습니다: {exc}\nJSON 객체만 출력하세요."
            else:
                문제 = validate_plan(계획, self.profiler)
                시도들.append({"attempt": attempt, "plan": 계획, "problems": 문제})
                if not 문제:
                    if self.verbose:
                        print(f"[계획 수립 성공 - {attempt}회] {계획.get('test')}")
                    계획["attempts"] = 시도들
                    return 계획
                if self.verbose:
                    print(f"[{attempt}회] {계획.get('test')} → 검증 실패: {문제[0]}")
                피드백 = (
                    "이 계획은 데이터의 성질과 맞지 않습니다:\n"
                    + "\n".join(f"- {p}" for p in 문제)
                    + "\n위 지적을 반영해 계획을 고쳐 JSON만 다시 출력하세요."
                )

            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": 피드백})

        return {
            "test": None,
            "error": f"{self.n_retry}회 시도 모두 검증을 통과하지 못했습니다.",
            "attempts": 시도들,
        }


In [ ]:
%%writefile rag.py
"""전산실험2 10주차 : RAG — 검색 증강 생성.

9주차에 검정 선택 규칙을 **코드에 직접 적었다.** 잘 동작하지만 한계가 있다.
규칙에 없는 상황이 오면 답할 수 없고, "왜 이 검정인가"를 물으면
"규칙에 그렇게 되어 있어서"라고밖에 말하지 못한다.

이 모듈은 **문서에서 근거를 찾아와 인용**하게 만든다.

    질문 → 검색 → 관련 문서 조각 → 프롬프트에 넣어 답변 → 인용 검증

두 가지 검색 방식을 모두 구현한다.

    TfidfRetriever      단어 겹침으로 찾는다. 설치 불필요, 항상 동작
    EmbeddingRetriever  의미로 찾는다. 서버의 /v1/embeddings 가 필요

둘을 비교하는 것 자체가 실습이다. **어휘 검색과 의미 검색은 잘하는 것이 다르다.**

사용 예:
    from rag import chunk_markdown, TfidfRetriever, answer_with_citations

    chunks = chunk_markdown(open("stat_methods.md", encoding="utf-8").read())
    r = TfidfRetriever(chunks)
    print(answer_with_citations("세 집단 비교에는 뭘 쓰나요?", r, llm))
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Chunk",
    "chunk_markdown",
    "Retriever",
    "TfidfRetriever",
    "EmbeddingRetriever",
    "build_context",
    "check_citations",
    "answer_with_citations",
    "RAG_SYSTEM",
]


# --------------------------------------------------------------------------
# 청킹
# --------------------------------------------------------------------------
@dataclass
class Chunk:
    """검색 단위. 문서를 잘라 놓은 조각 하나."""

    id: int
    title: str
    text: str
    source: str = ""

    @property
    def n_chars(self) -> int:
        return len(self.text)

    def cite(self) -> str:
        return f"[{self.id}] {self.title}"

    def block(self, max_chars: int = 1200) -> str:
        """프롬프트에 넣을 형태."""
        본문 = self.text[:max_chars]
        if len(self.text) > max_chars:
            본문 += " ...(생략)"
        return f"[{self.id}] {self.title}\n{본문}"


def chunk_markdown(
    text: str,
    *,
    heading: str = "## ",
    source: str = "",
    max_chars: int = 2000,
) -> list[Chunk]:
    """마크다운을 제목 단위로 자른다.

    **왜 제목 단위인가**
    고정 길이(예: 500자)로 자르면 문장이 중간에서 끊기고, 한 주제가
    두 조각으로 갈라진다. 문서에 이미 구조가 있으면 그것을 쓰는 편이 낫다.
    "일원분산분석" 절 전체가 한 조각이면 검색이 걸렸을 때 답이 그 안에 다 있다.

    너무 긴 절은 문단 경계에서 다시 나눈다.
    """
    조각: list[Chunk] = []
    현재제목 = "(머리말)"
    버퍼: list[str] = []

    def flush() -> None:
        본문 = "\n".join(버퍼).strip()
        if not 본문:
            return
        if len(본문) <= max_chars:
            조각.append(Chunk(len(조각) + 1, 현재제목, 본문, source))
            return
        # 긴 절은 문단 경계에서 분할
        문단들 = 본문.split("\n\n")
        part, n = [], 0
        for 문단 in 문단들:
            if n + len(문단) > max_chars and part:
                조각.append(Chunk(len(조각) + 1, f"{현재제목} (계속)" if 조각 and
                                조각[-1].title.startswith(현재제목) else 현재제목,
                                "\n\n".join(part), source))
                part, n = [], 0
            part.append(문단)
            n += len(문단)
        if part:
            조각.append(Chunk(len(조각) + 1,
                            f"{현재제목} (계속)" if 조각 and
                            조각[-1].title.startswith(현재제목) else 현재제목,
                            "\n\n".join(part), source))

    for line in text.splitlines():
        if line.startswith(heading):
            flush()
            버퍼 = []
            현재제목 = line[len(heading):].strip()
        else:
            버퍼.append(line)
    flush()

    return [c for c in 조각 if c.n_chars > 40]      # 너무 짧은 조각은 버린다


# --------------------------------------------------------------------------
# 검색기
# --------------------------------------------------------------------------
class Retriever:
    """검색기 공통 인터페이스."""

    name = "base"

    def __init__(self, chunks: list[Chunk]) -> None:
        self.chunks = chunks

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        raise NotImplementedError

    def show(self, query: str, k: int = 3) -> None:
        print(f"[{self.name}] '{query}'")
        for c, s in self.search(query, k):
            print(f"  {s:.3f}  {c.cite()}")


class TfidfRetriever(Retriever):
    """단어·글자 겹침으로 찾는다 (어휘 검색).

    한국어는 조사가 붙어 단어 경계가 흐릿하므로 **글자 n-gram** 을 쓴다.
    `char_wb` 는 단어 안에서만 n-gram 을 만들어 띄어쓰기를 존중한다.

    장점: 설치·API 불필요, 빠름, 결과를 설명할 수 있음
    단점: **같은 뜻 다른 말**을 못 찾는다 ("세 집단" vs "3개 그룹")
    """

    name = "tfidf"

    def __init__(self, chunks: list[Chunk], *, ngram_range: tuple = (2, 4)) -> None:
        super().__init__(chunks)
        from sklearn.feature_extraction.text import TfidfVectorizer

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=ngram_range, min_df=1, sublinear_tf=True
        )
        문서들 = [f"{c.title}\n{c.title}\n{c.text}" for c in chunks]  # 제목에 가중치
        self.matrix = self.vectorizer.fit_transform(문서들)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        from sklearn.metrics.pairwise import cosine_similarity

        q = self.vectorizer.transform([query])
        점수 = cosine_similarity(q, self.matrix)[0]
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


class EmbeddingRetriever(Retriever):
    """의미로 찾는다 (의미 검색).

    문장을 벡터로 바꿔 코사인 유사도로 비교한다.
    "세 집단 비교" 와 "3개 그룹의 평균 차이" 를 같은 뜻으로 인식할 수 있다.

    서버에 `/v1/embeddings` 가 없으면 만들 때 예외가 난다.
    `EmbeddingRetriever.available(llm)` 로 먼저 확인할 것.
    """

    name = "embedding"

    def __init__(self, chunks: list[Chunk], llm: Any, *,
                 model: str = "text-embedding-3-small", batch: int = 32) -> None:
        super().__init__(chunks)
        self.llm = llm
        self.model = model
        문서들 = [f"{c.title}\n{c.text}" for c in chunks]
        self.vectors = self._embed(문서들, batch)

    # ----------------------------------------------------------------
    @staticmethod
    def available(llm: Any, model: str = "text-embedding-3-small") -> bool:
        """이 서버가 임베딩을 지원하는가."""
        try:
            llm.client.embeddings.create(model=model, input=["테스트"])
            return True
        except Exception:
            return False

    def _embed(self, texts: list[str], batch: int = 32):
        import numpy as np

        벡터들 = []
        for i in range(0, len(texts), batch):
            덩어리 = texts[i : i + batch]
            응답 = self.llm.client.embeddings.create(model=self.model, input=덩어리)
            벡터들.extend([d.embedding for d in 응답.data])
        arr = np.asarray(벡터들, dtype=float)
        # 정규화해 두면 코사인 유사도가 내적이 된다
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        return arr / np.clip(norms, 1e-12, None)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        q = self._embed([query])[0]
        점수 = self.vectors @ q
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


# --------------------------------------------------------------------------
# 프롬프트 구성과 인용 검증
# --------------------------------------------------------------------------
RAG_SYSTEM = """\
너는 통계 방법론 조언자다. 아래에 주어진 참고 문서에만 근거해서 답한다.

절대 규칙:
- 참고 문서에 없는 내용은 말하지 않는다. 모르면 "참고 문서에 없다"고 답한다.
- 모든 주장 뒤에 근거 번호를 [1] 처럼 붙인다.
- 여러 문서를 쓰면 [1][3] 처럼 모두 표시한다.
- 문서에 없는 번호를 지어내지 않는다.
- 참고 문서와 다른 내용을 아는 것 같아도, 문서를 따른다."""


def build_context(hits: list[tuple[Chunk, float]], max_chars: int = 1200) -> str:
    """검색 결과를 프롬프트에 넣을 문자열로."""
    return "\n\n".join(c.block(max_chars) for c, _ in hits)


_CITATION = re.compile(r"\[(\d+)\]")


@dataclass
class CitationCheck:
    """인용이 제대로 됐는지 검사한 결과."""

    cited: list[int] = field(default_factory=list)
    valid: list[int] = field(default_factory=list)
    invalid: list[int] = field(default_factory=list)   # 존재하지 않는 번호
    unused: list[int] = field(default_factory=list)    # 줬는데 안 쓴 문서
    n_sentences: int = 0
    n_cited_sentences: int = 0

    @property
    def ok(self) -> bool:
        return not self.invalid and bool(self.valid)

    @property
    def coverage(self) -> float:
        """문장 중 몇 %가 근거를 달고 있는가."""
        return self.n_cited_sentences / self.n_sentences if self.n_sentences else 0.0

    def report(self) -> str:
        줄 = [f"인용된 번호: {self.cited}"]
        if self.invalid:
            줄.append(f"⚠ 존재하지 않는 번호: {self.invalid}  ← 지어낸 인용")
        if self.unused:
            줄.append(f"  제공했으나 쓰이지 않은 문서: {self.unused}")
        줄.append(f"  근거를 단 문장 비율: {self.coverage:.0%} "
                  f"({self.n_cited_sentences}/{self.n_sentences})")
        return "\n".join(줄)


def check_citations(answer: str, hits: list[tuple[Chunk, float]]) -> CitationCheck:
    """답변의 인용을 검증한다.

    **RAG에서 가장 중요한 검증이다.**
    모델은 근거 없이 그럴듯한 말을 하거나, 존재하지 않는 문서 번호를
    지어낼 수 있다. 인용이 있다고 해서 믿으면 안 된다.

    여기서 확인하는 것:
      1) 인용한 번호가 실제로 제공된 문서인가
      2) 제공한 문서 중 무엇을 썼는가
      3) 문장 중 몇 %가 근거를 달고 있는가

    확인하지 **못하는** 것: 인용한 내용이 그 문서에 실제로 있는가.
    그것은 사람이 보거나 별도 검증기(12주차 Critic)가 해야 한다.
    """
    제공번호 = {c.id for c, _ in hits}
    인용 = [int(m) for m in _CITATION.findall(answer)]
    고유 = sorted(set(인용))

    문장들 = [s for s in re.split(r"[.!?。]\s*|\n", answer) if len(s.strip()) > 10]
    근거단문장 = sum(1 for s in 문장들 if _CITATION.search(s))

    return CitationCheck(
        cited=고유,
        valid=[i for i in 고유 if i in 제공번호],
        invalid=[i for i in 고유 if i not in 제공번호],
        unused=sorted(제공번호 - set(고유)),
        n_sentences=len(문장들),
        n_cited_sentences=근거단문장,
    )


def answer_with_citations(
    question: str,
    retriever: Retriever,
    llm: Any,
    *,
    k: int = 3,
    max_chars: int = 1200,
    return_details: bool = False,
    **kw: Any,
) -> Any:
    """검색 → 답변 → 인용 검증까지 한 번에.

    Returns
    -------
    str, 또는 return_details=True 면
    {"answer": ..., "hits": ..., "check": CitationCheck}
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 700)

    hits = retriever.search(question, k=k)
    context = build_context(hits, max_chars)

    prompt = (
        f"## 참고 문서\n{context}\n\n"
        f"## 질문\n{question}\n\n"
        f"위 참고 문서에만 근거해서 답하라. 모든 주장에 [번호] 를 붙여라."
    )
    answer = llm.chat(
        [{"role": "system", "content": RAG_SYSTEM},
         {"role": "user", "content": prompt}], **kw
    )
    check = check_citations(answer, hits)

    if return_details:
        return {"answer": answer, "hits": hits, "check": check, "context": context}
    return answer


In [ ]:
%%writefile evaluate.py
"""전산실험2 11주차 : 에이전트 평가 — 성능을 추정 문제로 다룬다.

지금까지 매주 성공률을 재기는 했다. 그런데 n=10, n=20 수준이었고
신뢰구간이 너무 넓어 사실 아무것도 주장할 수 없었다.

이 모듈은 그것을 제대로 한다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 수업이 다른 AI 강의와 갈리는 지점

 에이전트를 k번 반복 실행해 얻은 관측은 **서로 독립이 아니다.**
 같은 문항을 여러 번 돌린 것이기 때문이다. 어떤 문항은 늘 맞고
 어떤 문항은 늘 틀린다. 문항 안의 관측은 서로 닮아 있다.

 그런데 대부분의 AI 평가 코드는 이것을 무시하고
 (맞은 횟수) / (전체 시행) 을 그냥 이항비율로 다룬다.
 → **신뢰구간이 실제보다 좁게 나온다. 없는 차이를 있다고 말하게 된다.**

 이것은 통계학에서 오래 알려진 **군집자료(clustered data)** 문제다.
 여러분은 이것을 아는 사람들이다. 제대로 하자.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

구성:
    GoldenItem / GoldenSet   정답이 정해진 평가 문항
    채점 함수들               keyword / numeric / choice / judge
    run_eval                 반복 실행해 결과를 모은다
    summarize                정확도 + 신뢰구간 (군집 보정 포함)
    compare_conditions       두 조건을 짝지어 비교 (McNemar)
    required_n               표본 크기 계산

사용 예:
    from evaluate import GoldenItem, run_eval, summarize

    items = [GoldenItem("q1", "치료 전후 차이가 있나?", expected="유의",
                        scorer="keyword")]
    df = run_eval(items, lambda q: agent.run(q).final_answer, repeats=5)
    print(summarize(df))
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from typing import Any, Callable

__all__ = [
    "GoldenItem",
    "score_keyword",
    "score_numeric",
    "score_choice",
    "make_judge",
    "run_eval",
    "summarize",
    "per_item",
    "consistency",
    "compare_conditions",
    "required_n",
    "wilson",
]


# --------------------------------------------------------------------------
# 평가 문항
# --------------------------------------------------------------------------
@dataclass
class GoldenItem:
    """정답이 정해진 평가 문항 하나.

    Parameters
    ----------
    id       문항 식별자
    question 에이전트에게 줄 질문
    expected 정답. 채점 방식에 따라 형태가 다르다
             keyword -> str 또는 list[str] (모두 포함되어야 함)
             numeric -> float
             choice  -> str (허용 목록 중 하나)
    scorer   "keyword" | "numeric" | "choice" | callable
    tolerance numeric 채점의 상대 허용오차
    tags     분석용 꼬리표 (난이도, 유형 등)
    """

    id: str
    question: str
    expected: Any
    scorer: Any = "keyword"
    tolerance: float = 0.01
    tags: dict[str, Any] = field(default_factory=dict)


# --------------------------------------------------------------------------
# 채점 함수
# --------------------------------------------------------------------------
def score_keyword(answer: str, expected: Any, **_: Any) -> bool:
    """정답 키워드가 모두 들어 있는가.

    가장 단순하고 가장 자주 쓴다. 다만 **표현이 달라지면 놓친다.**
    "유의하다" 를 기대했는데 "통계적으로 의미 있는 차이" 라고 답하면 오답 처리된다.
    이 한계를 알고 써야 한다. 대안은 `make_judge`.
    """
    if answer is None:
        return False
    본문 = str(answer).lower()
    필요 = [expected] if isinstance(expected, str) else list(expected)
    return all(str(k).lower() in 본문 for k in 필요)


_NUM = re.compile(r"[-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?")


def score_numeric(answer: str, expected: float, tolerance: float = 0.01, **_: Any) -> bool:
    """답변에 들어 있는 수치 중 하나가 정답과 충분히 가까운가.

    p값처럼 자릿수가 큰 값은 **상대오차**로 봐야 한다.
    p=1e-20 과 p=2e-20 의 절대차는 0에 가깝지만 둘은 다른 값이 아니다
    (둘 다 "매우 작다"). 반대로 0.04 와 0.06 은 절대차가 작아도
    결론이 갈린다. 여기서는 상대오차를 쓰되, 문항에서 tolerance 를 조절한다.
    """
    if answer is None:
        return False
    후보 = []
    for m in _NUM.findall(str(answer)):
        try:
            후보.append(float(m.replace(",", "")))
        except ValueError:
            pass
    if not 후보:
        return False
    if expected == 0:
        return any(abs(v) <= tolerance for v in 후보)
    return any(abs(v - expected) / abs(expected) <= tolerance for v in 후보)


def score_choice(answer: str, expected: str, **_: Any) -> bool:
    """답변이 정답 선택지를 명시했는가. 구두점·대소문자·공백 차이를 무시한다."""
    if answer is None:
        return False
    정규 = lambda s: re.sub(r"[\s\-_]", "", str(s).lower())
    return 정규(expected) in 정규(answer)


def make_judge(llm: Any, criteria: str = "", **kw: Any) -> Callable[..., bool]:
    """LLM-as-judge 채점기를 만든다.

    표현이 달라도 뜻이 맞으면 정답으로 인정하고 싶을 때 쓴다.

    **주의: 판정자도 모델이다.**
      - 판정자가 틀릴 수 있다 → 사람이 채점한 것과 일치율을 먼저 재라
      - 판정자와 피평가자가 같은 모델이면 편향이 생긴다 (자기 답을 후하게 봄)
      - 판정 자체가 확률적이다 → 판정도 여러 번 해서 다수결

    그래서 이 수업의 기본 채점은 keyword/numeric 이고,
    judge 는 **그것으로 안 되는 문항에만** 쓴다.
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 200)

    SYSTEM = (
        "너는 채점자다. 학생의 답변이 정답과 같은 뜻인지 판정한다.\n"
        "표현이 달라도 내용이 맞으면 정답으로 인정한다.\n"
        "핵심 결론이 다르면 오답이다. 애매하면 오답으로 처리한다.\n"
        '반드시 {"correct": true} 또는 {"correct": false} 형태의 JSON만 출력한다.'
        + (f"\n추가 기준: {criteria}" if criteria else "")
    )

    def judge(answer: str, expected: Any, **_: Any) -> bool:
        from structured import JSONParseError, parse_json

        if answer is None:
            return False
        prompt = (f"## 정답\n{expected}\n\n## 학생 답변\n{answer}\n\n"
                  f"같은 뜻인가? JSON만 출력하라.")
        try:
            out = llm.chat([{"role": "system", "content": SYSTEM},
                            {"role": "user", "content": prompt}], **kw)
            return bool(parse_json(out).get("correct", False))
        except (JSONParseError, Exception):
            return False

    return judge


_SCORERS = {"keyword": score_keyword, "numeric": score_numeric, "choice": score_choice}


def _resolve(scorer: Any) -> Callable[..., bool]:
    if callable(scorer):
        return scorer
    if scorer in _SCORERS:
        return _SCORERS[scorer]
    raise ValueError(f"알 수 없는 채점 방식: {scorer!r}. 가능: {list(_SCORERS)} 또는 함수")


# --------------------------------------------------------------------------
# 실행
# --------------------------------------------------------------------------
def run_eval(
    items: list[GoldenItem],
    run_fn: Callable[[str], Any],
    *,
    repeats: int = 5,
    condition: str = "base",
    verbose: bool = True,
    on_error: str = "wrong",
) -> Any:
    """각 문항을 `repeats` 회 실행하고 채점한다.

    Parameters
    ----------
    run_fn : 질문을 받아 답변(문자열 또는 객체)을 돌려주는 함수
    repeats : 문항당 반복 횟수. **1이면 안 된다.** 변동을 못 본다
    on_error : 실행이 예외로 죽었을 때 "wrong"(오답 처리) 또는 "skip"

    Returns
    -------
    DataFrame : item_id, rep, condition, correct, answer, elapsed, error
                **long format** — 한 행이 한 번의 시행
    """
    import time

    import pandas as pd

    행 = []
    for item in items:
        점수함수 = _resolve(item.scorer)
        for rep in range(1, repeats + 1):
            t0 = time.perf_counter()
            answer, err = None, None
            try:
                answer = run_fn(item.question)
            except Exception as exc:
                err = f"{type(exc).__name__}: {exc}"
                if on_error == "skip":
                    continue
            경과 = time.perf_counter() - t0

            correct = False
            if err is None:
                try:
                    correct = bool(점수함수(answer, item.expected,
                                          tolerance=item.tolerance))
                except Exception as exc:
                    err = f"채점 실패: {type(exc).__name__}: {exc}"

            행.append({
                "item_id": item.id, "rep": rep, "condition": condition,
                "correct": correct, "answer": str(answer)[:500],
                "elapsed": round(경과, 2), "error": err,
                **{f"tag_{k}": v for k, v in item.tags.items()},
            })
            if verbose:
                print("O" if correct else ("E" if err else "X"), end="", flush=True)
        if verbose:
            print(f"  {item.id}")
    return pd.DataFrame(행)


# --------------------------------------------------------------------------
# 집계
# --------------------------------------------------------------------------
def wilson(successes: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """이항비율의 Wilson 신뢰구간.

    n이 작거나 비율이 0/1에 가까울 때 정규근사(Wald)보다 훨씬 낫다.
    Wald 는 [0,1] 을 벗어나거나 폭이 0이 되어 버린다.
    """
    if n == 0:
        return (0.0, 1.0)
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def per_item(df: Any) -> Any:
    """문항별 정확도. **어떤 문항이 어려운지 보는 것이 지표 하나보다 중요하다.**"""
    import pandas as pd

    g = df.groupby("item_id", sort=False)["correct"]
    out = pd.DataFrame({"n": g.count(), "맞음": g.sum(), "정확도": g.mean()})
    out["항상맞음"] = out["정확도"] == 1.0
    out["항상틀림"] = out["정확도"] == 0.0
    return out.reset_index()


def summarize(df: Any, alpha: float = 0.05) -> dict[str, Any]:
    """정확도를 **군집 보정과 함께** 요약한다.

    ★ 이 함수가 이 모듈의 핵심이다. ★

    같은 문항을 반복 실행한 관측은 서로 독립이 아니다.
    그래서 두 가지를 함께 보고한다.

      naive_ci   : 모든 시행을 독립으로 본 Wilson 구간 (대부분의 AI 평가 코드)
      cluster_ci : 문항을 군집으로 본 구간 (문항별 정확도의 표본평균 기반)

    `design_effect` 가 1보다 크면 naive 구간이 **실제보다 좁다**는 뜻이다.
    즉 없는 차이를 있다고 말할 위험이 있다.
    """
    import numpy as np
    from scipy.stats import t as t_dist

    n_trials = len(df)
    n_correct = int(df["correct"].sum())
    p_hat = n_correct / n_trials if n_trials else 0.0

    # (1) 순진한 구간 — 모든 시행이 독립이라고 가정
    naive = wilson(n_correct, n_trials, alpha)

    # (2) 군집 보정 — 문항별 정확도를 하나의 관측으로 본다
    문항정확도 = df.groupby("item_id", sort=False)["correct"].mean().values
    m = len(문항정확도)
    p_cluster = float(문항정확도.mean()) if m else 0.0
    if m > 1:
        se = float(문항정확도.std(ddof=1) / math.sqrt(m))
        tcrit = t_dist.ppf(1 - alpha / 2, df=m - 1)
        cluster = (max(0.0, p_cluster - tcrit * se), min(1.0, p_cluster + tcrit * se))
    else:
        se, cluster = float("nan"), (0.0, 1.0)

    # (3) 급내상관 ICC — 같은 문항 안의 관측이 얼마나 닮았는가
    #     일원 임의효과 모형의 표준 추정량을 쓴다.
    #        ICC = (MSB - MSW) / (MSB + (k-1) MSW)
    #     신뢰구간 폭의 비율로 역산하면 t분포 임계값 때문에 1을 넘어 버린다.
    icc = _icc_oneway(df)

    # (4) 설계효과 — 표준 공식. 실효표본 = 전체시행 / deff
    k_bar = n_trials / m if m else 0
    deff = 1 + (k_bar - 1) * icc if not math.isnan(icc) else float("nan")

    naive_width = naive[1] - naive[0]
    cluster_width = cluster[1] - cluster[0]

    return {
        "n_items": m,
        "n_trials": n_trials,
        "reps_per_item": round(k_bar, 1),
        "accuracy": round(p_hat, 4),
        "accuracy_by_item": round(p_cluster, 4),
        "naive_ci": (round(naive[0], 4), round(naive[1], 4)),
        "cluster_ci": (round(cluster[0], 4), round(cluster[1], 4)),
        "cluster_se": round(se, 4) if not math.isnan(se) else None,
        "icc": round(icc, 3) if not math.isnan(icc) else None,
        "design_effect": round(deff, 2) if not math.isnan(deff) else None,
        "effective_n": round(n_trials / deff, 1) if deff and deff > 0 else None,
        "ci_width_ratio": round(cluster_width / naive_width, 2) if naive_width > 0 else None,
        "n_errors": int(df["error"].notna().sum()),
    }


def _icc_oneway(df: Any) -> float:
    """일원 임의효과 급내상관.

        ICC = (MSB - MSW) / (MSB + (k-1) MSW)

    문항이 군집, 반복 시행이 군집 내 관측이다.
    음수가 나오면 0으로 자른다 (군집 효과가 없다는 뜻).
    """
    그룹 = [g["correct"].astype(float).values for _, g in df.groupby("item_id", sort=False)]
    m = len(그룹)
    if m < 2:
        return float("nan")
    n_i = [len(g) for g in 그룹]
    if min(n_i) < 2:
        return float("nan")

    전체평균 = sum(g.sum() for g in 그룹) / sum(n_i)
    ss_b = sum(len(g) * (g.mean() - 전체평균) ** 2 for g in 그룹)
    ss_w = sum(((g - g.mean()) ** 2).sum() for g in 그룹)
    df_b, df_w = m - 1, sum(n_i) - m
    if df_w <= 0:
        return float("nan")
    ms_b, ms_w = ss_b / df_b, ss_w / df_w

    # 균형이 안 맞을 때의 유효 군집 크기
    N = sum(n_i)
    k0 = (N - sum(x * x for x in n_i) / N) / (m - 1)
    if k0 <= 0 or (ms_b + (k0 - 1) * ms_w) == 0:
        return float("nan")
    icc = (ms_b - ms_w) / (ms_b + (k0 - 1) * ms_w)
    return float(min(1.0, max(0.0, icc)))


def consistency(df: Any) -> dict[str, Any]:
    """같은 문항을 반복했을 때 얼마나 일관되는가.

    정확도가 같아도 성질이 전혀 다를 수 있다.

      A: 문항의 절반은 늘 맞고 절반은 늘 틀림  → 정확도 50%, 일관성 100%
      B: 모든 문항을 반반씩 맞힘              → 정확도 50%, 일관성 50%

    A는 "못 푸는 문제가 정해져 있다" 이고 B는 "매번 운이다" 이다.
    **고쳐야 할 것이 완전히 다르다.**
    """
    import numpy as np

    문항 = df.groupby("item_id", sort=False)["correct"]
    정확도 = 문항.mean()
    # 다수결과 일치하는 비율 (0.5 -> 0.5, 0 또는 1 -> 1.0)
    다수결일치 = 정확도.apply(lambda p: max(p, 1 - p))
    return {
        "항상 맞음": int((정확도 == 1.0).sum()),
        "항상 틀림": int((정확도 == 0.0).sum()),
        "들쭉날쭉": int(((정확도 > 0) & (정확도 < 1)).sum()),
        "평균 다수결 일치율": round(float(다수결일치.mean()), 3),
        "문항 간 정확도 표준편차": round(float(정확도.std(ddof=1)), 3) if len(정확도) > 1 else None,
    }


# --------------------------------------------------------------------------
# 조건 비교
# --------------------------------------------------------------------------
def compare_conditions(
    df_a: Any, df_b: Any, *, name_a: str = "A", name_b: str = "B", alpha: float = 0.05
) -> dict[str, Any]:
    """두 조건(프롬프트 A/B 등)을 비교한다.

    **같은 문항으로 두 조건을 돌렸으므로 짝지은 자료다.**
    독립 두 표본 검정을 쓰면 문항 난이도 차이가 오차에 섞여 검정력을 잃는다.
    4주차에 "대응표본을 독립표본으로 다루지 마라" 고 한 것과 같은 이야기다.

    두 가지 방법을 함께 준다.
      - 문항별 다수결을 이분값으로 만들어 **McNemar 검정**
      - 문항별 정확도 차이에 **대응표본 t / Wilcoxon**
    """
    import pandas as pd
    from scipy.stats import ttest_rel, wilcoxon

    a = df_a.groupby("item_id", sort=False)["correct"].mean()
    b = df_b.groupby("item_id", sort=False)["correct"].mean()
    공통 = a.index.intersection(b.index)
    if len(공통) < 2:
        return {"error": "공통 문항이 2개 미만입니다."}
    a, b = a.loc[공통], b.loc[공통]

    # --- McNemar (문항별 다수결을 정답/오답으로) ---
    a_bin, b_bin = (a > 0.5), (b > 0.5)
    n01 = int((~a_bin & b_bin).sum())     # A만 틀리고 B만 맞음
    n10 = int((a_bin & ~b_bin).sum())     # A만 맞고 B만 틀림
    if n01 + n10 == 0:
        mcnemar_p = 1.0
    else:
        from scipy.stats import binomtest

        mcnemar_p = float(binomtest(n10, n01 + n10, 0.5).pvalue)

    # --- 문항별 정확도 차이 ---
    diff = (b - a).values
    if len(diff) > 1 and diff.std(ddof=1) > 0:
        t_res = ttest_rel(b, a)
        t_p = float(t_res.pvalue)
        try:
            w_p = float(wilcoxon(b, a).pvalue)
        except Exception:
            w_p = float("nan")
    else:
        t_p = w_p = 1.0

    return {
        "n_items": len(공통),
        f"{name_a} 정확도": round(float(a.mean()), 4),
        f"{name_b} 정확도": round(float(b.mean()), 4),
        "차이(B-A)": round(float(b.mean() - a.mean()), 4),
        "McNemar_불일치": f"A만맞음 {n10} / B만맞음 {n01}",
        "McNemar_p": round(mcnemar_p, 4),
        "대응t_p": round(t_p, 4),
        "Wilcoxon_p": round(w_p, 4) if not math.isnan(w_p) else None,
        "유의(alpha=%.2f)" % alpha: bool(min(mcnemar_p, t_p) < alpha),
        "주의": "짝지은 자료다. 독립표본 검정을 쓰면 검정력을 잃는다.",
    }


# --------------------------------------------------------------------------
# 표본 크기
# --------------------------------------------------------------------------
def required_n(
    p_expected: float = 0.75,
    half_width: float = 0.10,
    alpha: float = 0.05,
    *,
    icc: float = 0.0,
    repeats: int = 1,
) -> dict[str, Any]:
    """원하는 정밀도를 얻으려면 문항이 몇 개 필요한가.

    `icc` 를 주면 **군집 설계를 반영**한다.
    같은 문항을 여러 번 돌려도 실효 표본은 시행 수만큼 늘지 않는다.

        설계효과 = 1 + (repeats - 1) x icc
        실효표본 = 전체시행 / 설계효과

    이것이 이번 주의 실용적 결론이다.
    **문항 10개 x 20회보다 문항 40개 x 5회가 낫다.**
    """
    from scipy.stats import norm

    z = norm.ppf(1 - alpha / 2)
    n_simple = math.ceil(z**2 * p_expected * (1 - p_expected) / half_width**2)
    deff = 1 + (repeats - 1) * icc
    n_trials = math.ceil(n_simple * deff)
    n_items = math.ceil(n_trials / repeats)

    return {
        "가정": f"정확도 약 {p_expected:.0%}, 반폭 ±{half_width:.0%}, alpha={alpha}",
        "독립가정 시 필요 시행": n_simple,
        "설계효과": round(deff, 2),
        "군집 보정 후 필요 시행": n_trials,
        "반복": repeats,
        "필요 문항 수": n_items,
    }


In [ ]:
%%writefile orchestra.py
"""전산실험2 12주차 : 다중 에이전트와 워크플로.

지금까지 만든 것은 **에이전트 하나**다. 이제 역할을 나눈다.

    Planner  질문을 단계로 쪼갠다
    Analyst  각 단계를 실제로 수행한다 (도구 / 코드)
    Critic   결과를 검토하고 반려한다
    Orchestrator  이들을 엮고, 반려되면 다시 시킨다

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 정직하게 먼저 말할 것

 **다중 에이전트가 항상 나은 것이 아니다.**

   - 느리다      : 역할이 셋이면 호출이 3배 이상
   - 비싸다      : 토큰도 그만큼
   - 오류가 쌓인다 : Planner 가 틀리면 Analyst 가 아무리 잘해도 소용없다
                    각 단계 정확도가 0.8 이어도 3단계면 0.8³ = 0.51

 그런데도 쓰는 이유는 **Critic** 때문이다.
 검토자가 붙으면 "조용히 틀린 답"이 줄어든다.
 이 수업이 계속 강조한 것과 같은 방향이다.

 11주차 하네스로 **직접 측정해서** 판단하라. 믿지 말고 재라.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from orchestra import Orchestrator
    orch = Orchestrator(llm, toolbox, profiler=prof)
    result = orch.run("비료에 따라 수확량이 다른가?")
    result.show()
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Message",
    "Plan",
    "Review",
    "OrchestraResult",
    "Planner",
    "Analyst",
    "Critic",
    "Orchestrator",
]


# --------------------------------------------------------------------------
# 에이전트 사이에 오가는 것
# --------------------------------------------------------------------------
@dataclass
class Message:
    """누가 누구에게 무엇을 말했는가. 전체 흐름을 추적하는 단위."""

    sender: str
    kind: str          # plan / result / review / final / error
    content: Any
    round: int = 0

    def line(self, width: int = 100) -> str:
        본문 = self.content if isinstance(self.content, str) else json.dumps(
            self.content, ensure_ascii=False)
        return f"[R{self.round}] {self.sender:8s} {self.kind:7s} {본문[:width]}"


@dataclass
class Plan:
    """Planner 가 세운 단계 목록."""

    steps: list[dict[str, Any]] = field(default_factory=list)
    reason: str = ""
    ok: bool = True
    error: str | None = None

    def __len__(self) -> int:
        return len(self.steps)

    def brief(self) -> str:
        return "\n".join(
            f"{i}. {s.get('goal', '?')}" for i, s in enumerate(self.steps, 1)
        )


@dataclass
class Review:
    """Critic 의 판정."""

    approved: bool
    problems: list[str] = field(default_factory=list)
    suggestion: str = ""
    raw: str = ""

    def brief(self) -> str:
        if self.approved:
            return "승인"
        return "반려: " + "; ".join(self.problems[:3])


@dataclass
class OrchestraResult:
    """전체 실행 기록."""

    question: str
    plan: Plan | None = None
    answer: str | None = None
    reviews: list[Review] = field(default_factory=list)
    messages: list[Message] = field(default_factory=list)
    rounds: int = 0
    approved: bool = False
    n_llm_calls: int = 0
    elapsed: float = 0.0

    def show(self, width: int = 110) -> None:
        print(f"질문: {self.question}")
        print("=" * 78)
        for m in self.messages:
            print(m.line(width))
        print("=" * 78)
        print(f"라운드 {self.rounds} | 승인 {self.approved} | "
              f"LLM 호출 {self.n_llm_calls}회 | {self.elapsed:.1f}초")
        print(f"최종 답변: {str(self.answer)[:400]}")

    def to_dict(self) -> dict[str, Any]:
        return {
            "question": self.question,
            "answer": self.answer,
            "approved": self.approved,
            "rounds": self.rounds,
            "n_llm_calls": self.n_llm_calls,
            "elapsed": round(self.elapsed, 2),
            "plan": self.plan.steps if self.plan else [],
            "reviews": [{"approved": r.approved, "problems": r.problems}
                        for r in self.reviews],
        }


# --------------------------------------------------------------------------
# 역할 1 — Planner
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 역할이다. 직접 분석하지 않는다.

사용자의 질문을 **실행 가능한 단계**로 쪼갠다.
단계는 1~4개로 하고, 각 단계는 도구 하나로 처리할 수 있을 만큼 작아야 한다.

규칙:
- 데이터에 실제로 있는 열 이름만 쓴다.
- 검정을 고르기 전에 데이터를 확인하는 단계를 먼저 넣는다.
- 질문이 단순하면 단계를 억지로 늘리지 않는다. 1단계면 1단계다.

출력은 JSON 하나뿐이다:
{"steps": [{"goal": "무엇을 할지 한 문장"}, ...], "reason": "왜 이렇게 나눴는지"}"""


class Planner:
    """질문을 단계로 쪼갠다."""

    name = "Planner"

    def __init__(self, llm: Any, profiler: Any = None, *, max_steps: int = 4) -> None:
        self.llm = llm
        self.profiler = profiler
        self.max_steps = max_steps
        self.calls = 0

    def plan(self, question: str, feedback: str = "", **kw: Any) -> Plan:
        from structured import JSONParseError, parse_json

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 600)

        데이터 = f"## 데이터\n{self.profiler.to_prompt()}\n\n" if self.profiler else ""
        되먹임 = f"\n## 이전 계획에 대한 지적\n{feedback}\n계획을 고쳐서 다시 세워라.\n" if feedback else ""

        try:
            self.calls += 1
            원문 = self.llm.chat(
                [{"role": "system", "content": PLANNER_SYSTEM},
                 {"role": "user", "content": f"{데이터}## 질문\n{question}\n{되먹임}\nJSON만 출력하라."}],
                **kw)
            data = parse_json(원문)
            steps = data.get("steps") or []
            if not isinstance(steps, list) or not steps:
                return Plan(ok=False, error="steps 가 비었습니다.")
            steps = [s if isinstance(s, dict) else {"goal": str(s)}
                     for s in steps][: self.max_steps]
            return Plan(steps=steps, reason=str(data.get("reason", "")))
        except JSONParseError as exc:
            return Plan(ok=False, error=f"계획 파싱 실패: {exc}")
        except Exception as exc:
            return Plan(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------
# 역할 2 — Analyst
# --------------------------------------------------------------------------
class Analyst:
    """단계를 실제로 수행한다.

    안에서 6주차 `ReActAgent` 를 그대로 쓴다.
    **이미 만든 것을 감싸는 것**이지 새로 만드는 것이 아니다.
    """

    name = "Analyst"

    def __init__(self, llm: Any, toolbox: Any, *, max_steps: int = 6) -> None:
        from react import ReActAgent

        self.agent = ReActAgent(llm, toolbox, max_steps=max_steps, verbose=False)
        self.llm = llm
        self.calls = 0

    def execute(self, goal: str, context: str = "", **kw: Any) -> dict[str, Any]:
        질문 = f"{context}\n\n{goal}" if context else goal
        trace = self.agent.run(질문, **kw)
        self.calls += len(trace.steps) + 1
        return {
            "goal": goal,
            "answer": trace.final_answer,
            "success": trace.success,
            "tools": trace.tools_used,
            "n_failed": trace.n_failed,
            "observations": [s.observation for s in trace.steps if s.ok][-3:],
        }


# --------------------------------------------------------------------------
# 역할 3 — Critic
# --------------------------------------------------------------------------
CRITIC_SYSTEM = """\
너는 통계 분석 결과를 검토하는 역할이다. 분석을 다시 하지 않는다. 검토만 한다.

다음을 확인한다:
1. 질문에 실제로 답했는가
2. 근거가 되는 수치(p값, 표본 수, 통계량)가 제시되었는가
3. 검정 선택이 데이터의 성질과 맞는가
4. 유의하지 않은 결과를 "차이가 없다"고 단정하지 않았는가
5. 상관을 인과로 서술하지 않았는가
6. 숫자를 지어낸 흔적이 없는가 (도구 결과에 없는 수치)

엄격하게 보되, **트집을 잡지는 마라.**
답이 맞고 근거가 있으면 승인한다.

출력은 JSON 하나뿐이다:
{"approved": true/false, "problems": ["문제1", ...], "suggestion": "무엇을 고쳐야 하는지"}"""


class Critic:
    """분석 결과를 검토하고 반려한다.

    **다중 에이전트를 쓰는 진짜 이유가 이것이다.**
    검토자가 붙으면 "조용히 틀린 답"이 줄어든다.
    """

    name = "Critic"

    def __init__(self, llm: Any, *, strict: bool = True, system: str = CRITIC_SYSTEM) -> None:
        self.llm = llm
        self.strict = strict
        self.system = system
        self.calls = 0

    def review(self, question: str, answer: str, evidence: str = "", **kw: Any) -> Review:
        from structured import JSONParseError, parse_json

        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)

        prompt = (f"## 원래 질문\n{question}\n\n"
                  f"## 분석가의 답변\n{answer}\n\n"
                  f"## 도구 실행 결과 (근거)\n{evidence[:2500]}\n\n"
                  f"검토하고 JSON만 출력하라.")
        try:
            self.calls += 1
            원문 = self.llm.chat(
                [{"role": "system", "content": self.system},
                 {"role": "user", "content": prompt}], **kw)
            data = parse_json(원문)
            problems = data.get("problems") or []
            if not isinstance(problems, list):
                problems = [str(problems)]
            return Review(
                approved=bool(data.get("approved", False)),
                problems=[str(p) for p in problems],
                suggestion=str(data.get("suggestion", "")),
                raw=원문,
            )
        except JSONParseError as exc:
            # 검토자가 형식을 어기면 **승인으로 처리하지 않는다.**
            # 검증을 못 했는데 통과시키는 것이 가장 나쁘다.
            return Review(approved=False,
                          problems=[f"검토자 응답을 읽을 수 없음: {exc}"],
                          raw=str(exc))
        except Exception as exc:
            return Review(approved=False,
                          problems=[f"검토 실패: {type(exc).__name__}: {exc}"])


# --------------------------------------------------------------------------
# 오케스트레이터
# --------------------------------------------------------------------------
class Orchestrator:
    """Planner → Analyst → Critic 을 엮는다. 반려되면 다시 시킨다.

    Parameters
    ----------
    max_rounds : Critic 이 반려했을 때 다시 시도할 횟수 상한
    use_critic : False 면 Critic 을 끄고 단일 에이전트에 가깝게 동작
    replan_on_reject : True 면 반려 시 계획부터 다시. False 면 분석만 다시
    """

    def __init__(
        self,
        llm: Any,
        toolbox: Any,
        *,
        profiler: Any = None,
        max_rounds: int = 2,
        use_critic: bool = True,
        replan_on_reject: bool = False,
        verbose: bool = True,
    ) -> None:
        self.planner = Planner(llm, profiler)
        self.analyst = Analyst(llm, toolbox)
        self.critic = Critic(llm)
        self.llm = llm
        self.max_rounds = max_rounds
        self.use_critic = use_critic
        self.replan_on_reject = replan_on_reject
        self.verbose = verbose

    # ----------------------------------------------------------------
    def run(self, question: str, **kw: Any) -> OrchestraResult:
        """질문을 받아 끝까지 돌린다. **예외를 밖으로 던지지 않는다.**"""
        import time

        t0 = time.perf_counter()
        self.planner.calls = self.analyst.calls = self.critic.calls = 0
        res = OrchestraResult(question=question)
        feedback = ""

        for rnd in range(1, self.max_rounds + 1):
            res.rounds = rnd

            # --- 1) 계획 -------------------------------------------
            if res.plan is None or self.replan_on_reject:
                plan = self.planner.plan(question, feedback, **kw)
                res.plan = plan
                if not plan.ok:
                    res.messages.append(Message("Planner", "error", plan.error, rnd))
                    if self.verbose:
                        print(f"[R{rnd}] Planner 실패: {plan.error}")
                    break
                res.messages.append(Message("Planner", "plan", plan.brief(), rnd))
                if self.verbose:
                    print(f"[R{rnd}] Planner: {len(plan)}단계")
                    for i, s in enumerate(plan.steps, 1):
                        print(f"        {i}. {s.get('goal', '?')[:70]}")

            # --- 2) 실행 -------------------------------------------
            결과들, 근거들 = [], []
            맥락 = feedback and f"이전 시도에서 다음 지적을 받았다: {feedback}\n" or ""
            for i, step in enumerate(res.plan.steps, 1):
                out = self.analyst.execute(step.get("goal", ""), 맥락, **kw)
                결과들.append(out)
                근거들.extend(out["observations"])
                res.messages.append(
                    Message("Analyst", "result",
                            f"{i}. {out['goal'][:40]} -> {str(out['answer'])[:60]}", rnd))
                if self.verbose:
                    print(f"        Analyst {i}: {'OK' if out['success'] else '실패'} "
                          f"{str(out['answer'])[:60]}")

            답변 = self._synthesize(question, 결과들, **kw)
            res.answer = 답변
            res.messages.append(Message("Analyst", "final", 답변, rnd))

            # --- 3) 검토 -------------------------------------------
            if not self.use_critic:
                res.approved = True
                break

            review = self.critic.review(question, 답변, "\n\n".join(근거들), **kw)
            res.reviews.append(review)
            res.messages.append(Message("Critic", "review", review.brief(), rnd))
            if self.verbose:
                print(f"        Critic: {review.brief()[:90]}")

            if review.approved:
                res.approved = True
                break

            feedback = (review.suggestion or "; ".join(review.problems))[:600]
            if rnd == self.max_rounds and self.verbose:
                print(f"        -> {self.max_rounds}라운드 모두 반려됨")

        # 끝까지 반려되면 그 사실을 답변에 남긴다
        if self.use_critic and not res.approved and res.answer:
            지적 = "; ".join(res.reviews[-1].problems[:3]) if res.reviews else ""
            res.answer = (f"{res.answer}\n\n[검토 미통과] 검토자가 다음을 지적했습니다: {지적}\n"
                          f"이 결과를 그대로 사용하지 마십시오.")

        res.n_llm_calls = self.planner.calls + self.analyst.calls + self.critic.calls
        res.elapsed = time.perf_counter() - t0
        return res

    # ----------------------------------------------------------------
    def _synthesize(self, question: str, 결과들: list[dict], **kw: Any) -> str:
        """단계별 결과를 하나의 답변으로 합친다."""
        if len(결과들) == 1:
            return str(결과들[0]["answer"] or "")

        모음 = "\n\n".join(
            f"[단계 {i}] {r['goal']}\n결과: {r['answer']}"
            for i, r in enumerate(결과들, 1))
        try:
            return self.llm.ask(
                f"질문: {question}\n\n아래는 단계별 분석 결과다. "
                f"이것만 근거로 질문에 답하라. 숫자를 지어내지 마라.\n\n{모음}",
                system="너는 분석 결과를 종합해 답하는 역할이다.",
                temperature=0.0, max_tokens=kw.get("max_tokens", 600))
        except Exception:
            return 모음


In [ ]:
%%writefile report.py
"""전산실험2 13주차 : 리포트 생성과 해석 검사.

분석 결과를 사람이 읽을 문서로 만든다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 모듈의 핵심은 예쁜 출력이 아니라 **해석 검사**다.

 통계 결과를 서술로 옮길 때 사람도 자주 틀린다.
   - 상관을 인과로 쓴다             ("급수량이 수확량을 증가시킨다")
   - 유의하지 않은 것을 "차이 없다"고 단정한다
   - p값을 "효과가 클 확률"로 오해한다
   - 효과크기 없이 p값만 보고한다

 LLM은 이것을 **더** 자주 틀린다. 학습 데이터에 그런 문장이 잔뜩 있기 때문이다.

 그래서 생성된 서술을 **규칙으로 검사**한다.
 9주차와 같은 발상이다: 기계적으로 판정 가능하고 틀리면 치명적인 것은 규칙으로.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from report import Report, check_interpretation

    rep = Report("혈압 분석")
    rep.add_section("데이터", prof.report())
    rep.add_result("대응표본 t검정", {"p_value": 0.001, "n": 171})
    rep.add_interpretation("치료 전후로 혈압이 유의하게 감소했습니다 (p<0.001).")
    print(rep.to_markdown())
"""

from __future__ import annotations

import html
import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Issue",
    "check_interpretation",
    "phrase_significance",
    "Report",
    "ReportSection",
]


# --------------------------------------------------------------------------
# 해석 검사
# --------------------------------------------------------------------------
@dataclass
class Issue:
    """서술에서 발견된 문제."""

    level: str          # "error" (반드시 고칠 것) / "warn" (검토 필요)
    kind: str
    excerpt: str
    message: str
    suggestion: str = ""

    def line(self) -> str:
        표 = "✕" if self.level == "error" else "△"
        본문 = f"{표} [{self.kind}] {self.message}\n    \"{self.excerpt}\""
        if self.suggestion:
            본문 += f"\n    → {self.suggestion}"
        return 본문


# 상관·연관 분석에서 쓰면 안 되는 인과 표현
_CAUSAL = [
    (r"영향(을|를)?\s*(준다|미친다|미쳤|줍니다|주었)", "영향을 준다"),
    (r"(증가|감소|향상|악화)\s*시(킨다|킵니다|켰|켜)", "증가시킨다"),
    (r"때문(이다|입니다|에)", "때문이다"),
    (r"(으)?로\s*인(해|하여|한)", "으로 인해"),
    (r"원인(이다|입니다|으로)", "원인이다"),
    (r"효과(가|를)\s*(있|가져|나타)", "효과가 있다"),
    (r"\bcause[sd]?\b", "cause"),
    (r"\bleads? to\b", "leads to"),
]

# 유의하지 않은 결과를 단정하는 표현
_NULL_CLAIM = [
    (r"차이(가|는)?\s*없(다|습니다|음|었)", "차이가 없다"),
    (r"관련(이|성이)?\s*없(다|습니다|음)", "관련이 없다"),
    (r"연관(이|성이)?\s*없(다|습니다|음)", "연관이 없다"),
    (r"상관(이|관계가)?\s*없(다|습니다|음)", "상관이 없다"),
    (r"동일(하다|합니다|함)", "동일하다"),
    (r"같(다|습니다)\s*$", "같다"),
    (r"\bno (difference|effect|association|relationship)\b", "no difference"),
]

# p값 오해
_P_MISUSE = [
    (r"귀무가설[^.。\n]{0,20}(참|맞)[^.。\n]{0,6}확률", "p값을 귀무가설이 참일 확률로 서술"),
    (r"대립가설[^.。\n]{0,20}(참|맞)[^.。\n]{0,6}확률", "p값을 대립가설이 참일 확률로 서술"),
    (r"우연[^.。\n]{0,12}확률[^.。\n]{0,10}p\s*값", "p값을 우연일 확률로 서술"),
    (r"p\s*값[^.。\n]{0,12}우연[^.。\n]{0,8}확률", "p값을 우연일 확률로 서술"),
    (r"p\s*값(이|은)?\s*작(으므로|아서|기)[^.。\n]{0,14}효과(가|는)\s*(크|강)", "p값을 효과크기로 오해"),
    (r"\bprobability that the null\b", "p as probability of null"),
]

# '차이가 없다는 증거가 아니다' 처럼 **올바르게** 부정하는 문맥.
# 이것을 오탐하면 정답을 틀렸다고 하는 셈이라 반드시 걸러야 한다.
#
# 주의: '아닙니다' 는 '아니' + 'ㅂ니다' 가 축약되어 '아니' 라는 글자가 남지 않는다.
#       한국어 활용형을 다룰 때 늘 걸리는 지점이라 어간 변이형을 모두 적는다.
_NEGATED_CLAIM = re.compile(
    "|".join([
        # (가) "~없다는 증거가 아니다" — 명사를 거쳐 부정
        r"^\s*(는|은|다는|이라는|라는)?\s*"
        r"(증거|뜻|의미|말|것|주장|근거)[가는이은을]?\s*"
        r"(아니|아닙|아님|아냐|아녜|될\s*수\s*없|할\s*수\s*없)",
        # (나) "~없다고 할 수 없다" — 바로 부정
        r"^\s*(고|다고|라고|이라고)?\s*"
        r"(할|볼|말할|단정할|결론지을)\s*수(\s*는)?\s*없",
    ])
)


def check_interpretation(
    text: str,
    *,
    significant: bool | None = None,
    test_kind: str = "",
    has_effect_size: bool | None = None,
) -> list[Issue]:
    """생성된 해석 문장을 규칙으로 검사한다.

    Parameters
    ----------
    significant : 검정 결과가 유의했는가. None 이면 관련 검사를 건너뛴다
    test_kind   : "correlation" 이면 인과 표현을 error 로 본다
    has_effect_size : 효과크기가 보고되었는가

    Returns
    -------
    list[Issue] : 빈 목록이면 통과

    **완전하지 않다.** 규칙으로 잡히는 전형적 오류만 잡는다.
    "이 검사를 통과했으니 해석이 옳다"고 말하면 안 된다.
    """
    issues: list[Issue] = []
    if not text:
        return [Issue("error", "빈 해석", "", "해석 문장이 없습니다.")]

    상관분석 = "correl" in test_kind.lower() or "상관" in test_kind

    # --- 1) 인과 표현 --------------------------------------------------
    for pat, 이름 in _CAUSAL:
        for m in re.finditer(pat, text):
            발췌 = _around(text, m.start(), m.end())
            issues.append(Issue(
                level="error" if 상관분석 else "warn",
                kind="인과 표현",
                excerpt=발췌,
                message=f"'{이름}' 는 인과를 함의합니다."
                        + (" 상관분석에서는 쓸 수 없습니다." if 상관분석 else ""),
                suggestion="'~와 연관이 관찰되었다', '~와 함께 높게 나타났다' 로 바꾸세요.",
            ))

    # --- 2) 유의하지 않은데 '차이 없다' ---------------------------------
    if significant is False:
        for pat, 이름 in _NULL_CLAIM:
            for m in re.finditer(pat, text):
                # '차이가 없다는 증거가 아니다' 는 올바른 서술이다. 오탐하면 안 된다.
                if _NEGATED_CLAIM.match(text[m.end() : m.end() + 30]):
                    continue
                issues.append(Issue(
                    level="error",
                    kind="귀무가설 채택",
                    excerpt=_around(text, m.start(), m.end()),
                    message=f"'{이름}' 는 유의하지 않다는 것을 '차이가 없다'로 단정한 것입니다.",
                    suggestion="'차이가 있다고 보기 어렵다' 또는 "
                               "'유의한 차이가 관찰되지 않았다' 로 바꾸세요.",
                ))

    # --- 3) 유의한데 표현이 과함 ---------------------------------------
    if significant is True:
        for pat, 이름 in [(r"확실(히|하게|합니다)", "확실히"),
                         (r"증명(했|됐|되었|한다)", "증명했다"),
                         (r"입증(했|됐|되었)", "입증했다")]:
            for m in re.finditer(pat, text):
                issues.append(Issue(
                    level="warn", kind="과잉 확신",
                    excerpt=_around(text, m.start(), m.end()),
                    message=f"'{이름}' 는 통계적 유의성이 보장하지 않는 확신입니다.",
                    suggestion="'유의한 차이가 관찰되었다' 정도로 낮추세요.",
                ))

    # --- 4) p값 오해 ---------------------------------------------------
    for pat, 이름 in _P_MISUSE:
        for m in re.finditer(pat, text):
            issues.append(Issue(
                level="error", kind="p값 오해",
                excerpt=_around(text, m.start(), m.end()),
                message=f"{이름}. p값은 '귀무가설이 참일 때 이 정도 결과가 나올 확률' 입니다.",
                suggestion="p값의 정의에 맞게 서술하세요.",
            ))

    # --- 5) 효과크기 누락 ----------------------------------------------
    if has_effect_size is False and significant is True:
        issues.append(Issue(
            level="warn", kind="효과크기 누락", excerpt="",
            message="유의성만 보고하고 효과크기가 없습니다. "
                    "표본이 크면 미미한 차이도 유의해집니다.",
            suggestion="Cohen's d, eta squared, Cramer's V 등을 함께 보고하세요.",
        ))

    # --- 6) 수치 근거 없음 ---------------------------------------------
    if significant is not None and not re.search(r"p\s*[<=>]|p\s*값|p-value", text, re.I):
        issues.append(Issue(
            level="warn", kind="근거 수치 누락", excerpt="",
            message="p값이 서술에 없습니다.",
            suggestion="검정통계량, p값, 표본 수를 함께 적으세요.",
        ))

    return issues


def _around(text: str, start: int, end: int, pad: int = 22) -> str:
    a, b = max(0, start - pad), min(len(text), end + pad)
    return ("..." if a else "") + text[a:b].replace("\n", " ") + ("..." if b < len(text) else "")


def _josa(word: str, pair: str = "이/가") -> str:
    """받침 유무에 맞는 조사를 붙인다.

    '연관가' 같은 문장은 리포트의 신뢰를 떨어뜨린다.
    자동 생성 문서일수록 이런 사소한 것이 티가 난다.
    """
    받침있음, 받침없음 = pair.split("/")
    if not word:
        return 받침없음
    ch = word[-1]
    if not ("가" <= ch <= "힣"):
        return 받침없음          # 영문·숫자로 끝나면 안전한 쪽으로
    return 받침있음 if (ord(ch) - 0xAC00) % 28 else 받침없음


def phrase_significance(p: float, alpha: float = 0.05, what: str = "차이") -> str:
    """유의성을 **올바른 표현**으로 만들어 준다.

    LLM에게 서술을 맡기기 전에 이 문구를 주면 오류가 크게 준다.
    특히 유의하지 않을 때의 표현이 중요하다.
    """
    이가 = _josa(what, "이/가")
    을를 = _josa(what, "을/를")

    if p < alpha:
        강도 = "매우 " if p < 0.001 else ""
        p표기 = "< 0.001" if p < 0.001 else f"= {p:.4f}"
        return (f"유의수준 {alpha} 에서 {강도}유의한 {what}{이가} 관찰되었습니다 (p {p표기}).")
    return (f"유의수준 {alpha} 에서 유의한 {what}{이가} 관찰되지 않았습니다 (p = {p:.4f}). "
            f"이는 {what}{이가} 없다는 증거가 아니라, 이 표본으로는 {what}{을를} "
            f"확인하지 못했다는 뜻입니다.")


# --------------------------------------------------------------------------
# 리포트
# --------------------------------------------------------------------------
@dataclass
class ReportSection:
    title: str
    body: str = ""
    kind: str = "text"           # text / table / result / code / figure / warning
    data: Any = None


class Report:
    """분석 리포트를 조립한다.

    설계 원칙:
      1. **어떻게 계산했는지 함께 남긴다** (재현성 — 7주차의 문제)
      2. 해석은 **검사를 거친 뒤** 넣는다
      3. 경고와 한계를 **숨기지 않는다**
    """

    def __init__(self, title: str, *, subtitle: str = "") -> None:
        self.title = title
        self.subtitle = subtitle
        self.sections: list[ReportSection] = []
        self.issues: list[Issue] = []
        self.meta: dict[str, Any] = {}

    # ----------------------------------------------------------------
    def add_section(self, title: str, body: str, kind: str = "text") -> "Report":
        self.sections.append(ReportSection(title, body, kind))
        return self

    def add_table(self, title: str, df: Any, *, max_rows: int = 30) -> "Report":
        try:
            본문 = df.head(max_rows).to_markdown(index=False)
        except Exception:
            본문 = str(df)
        self.sections.append(ReportSection(title, 본문, "table", df))
        return self

    def add_result(self, title: str, result: dict[str, Any]) -> "Report":
        """검정 결과 딕셔너리를 표로 넣는다 (5주차 도구의 출력 형식)."""
        줄 = ["| 항목 | 값 |", "|---|---|"]
        for k, v in result.items():
            if isinstance(v, dict):
                v = ", ".join(f"{kk}={vv}" for kk, vv in v.items())
            if v is None:
                continue
            줄.append(f"| {k} | {v} |")
        self.sections.append(ReportSection(title, "\n".join(줄), "result", result))
        return self

    def add_code(self, title: str, code: str, *, lang: str = "python") -> "Report":
        """**어떤 코드로 계산했는지** 남긴다. 재현성의 핵심."""
        self.sections.append(
            ReportSection(title, f"```{lang}\n{code.strip()}\n```", "code", code))
        return self

    def add_interpretation(
        self,
        text: str,
        *,
        significant: bool | None = None,
        test_kind: str = "",
        has_effect_size: bool | None = None,
        title: str = "해석",
    ) -> "Report":
        """해석을 넣되 **먼저 검사한다.**"""
        문제 = check_interpretation(
            text, significant=significant, test_kind=test_kind,
            has_effect_size=has_effect_size)
        self.issues.extend(문제)
        self.sections.append(ReportSection(title, text, "text"))
        if 문제:
            경고 = "\n".join(i.line() for i in 문제)
            self.sections.append(
                ReportSection("해석 검사 결과", 경고, "warning", 문제))
        return self

    def add_warning(self, text: str, title: str = "주의") -> "Report":
        self.sections.append(ReportSection(title, text, "warning"))
        return self

    def set_meta(self, **kw: Any) -> "Report":
        self.meta.update(kw)
        return self

    # ----------------------------------------------------------------
    @property
    def n_errors(self) -> int:
        return sum(1 for i in self.issues if i.level == "error")

    @property
    def n_warnings(self) -> int:
        return sum(1 for i in self.issues if i.level == "warn")

    def environment(self) -> dict[str, str]:
        """재현성을 위한 실행 환경 기록."""
        import platform
        import sys

        정보 = {"python": sys.version.split()[0], "platform": platform.system()}
        for mod in ("numpy", "pandas", "scipy", "statsmodels"):
            try:
                정보[mod] = __import__(mod).__version__
            except Exception:
                pass
        정보.update({k: str(v) for k, v in self.meta.items()})
        return 정보

    # ----------------------------------------------------------------
    def to_markdown(self, *, include_env: bool = True) -> str:
        줄 = [f"# {self.title}"]
        if self.subtitle:
            줄.append(f"\n*{self.subtitle}*")
        줄.append("")

        if self.n_errors:
            줄 += [f"> ⚠ **해석 검사에서 오류 {self.n_errors}건이 발견되었습니다.**",
                   "> 이 리포트를 그대로 사용하기 전에 확인하십시오.", ""]

        for s in self.sections:
            줄.append(f"## {s.title}")
            줄.append("")
            if s.kind == "warning":
                줄.append("> " + s.body.replace("\n", "\n> "))
            else:
                줄.append(s.body)
            줄.append("")

        if include_env:
            줄 += ["## 실행 환경", "",
                   "| 항목 | 값 |", "|---|---|"]
            줄 += [f"| {k} | {v} |" for k, v in self.environment().items()]
            줄 += ["", "> 이 표는 결과를 재현하는 데 필요합니다. 지우지 마십시오.", ""]

        return "\n".join(줄)

    def to_html(self) -> str:
        """자체 완결 HTML. Colab 에서 바로 표시하거나 파일로 저장한다."""
        본문 = []
        for s in self.sections:
            cls = "warning" if s.kind == "warning" else s.kind
            본문.append(f'<section class="{cls}"><h2>{html.escape(s.title)}</h2>')
            if s.kind in ("table", "result"):
                본문.append(_md_table_to_html(s.body))
            elif s.kind == "code":
                코드 = s.body.strip().removeprefix("```python").removesuffix("```")
                본문.append(f"<pre><code>{html.escape(코드.strip())}</code></pre>")
            else:
                본문.append("".join(
                    f"<p>{html.escape(p)}</p>" for p in s.body.split("\n") if p.strip()))
            본문.append("</section>")

        배너 = ""
        if self.n_errors:
            배너 = (f'<div class="banner">해석 검사에서 오류 {self.n_errors}건이 '
                   f"발견되었습니다. 그대로 사용하기 전에 확인하십시오.</div>")

        환경 = "".join(f"<tr><td>{html.escape(k)}</td><td>{html.escape(str(v))}</td></tr>"
                     for k, v in self.environment().items())

        return f"""<!doctype html><html lang="ko"><head><meta charset="utf-8">
<title>{html.escape(self.title)}</title><style>
 body{{font-family:-apple-system,'Malgun Gothic',sans-serif;max-width:860px;
       margin:2rem auto;padding:0 1rem;line-height:1.65;color:#1a1a1a}}
 h1{{border-bottom:2px solid #333;padding-bottom:.4rem}}
 h2{{margin-top:2rem;color:#2c3e50;font-size:1.2rem}}
 table{{border-collapse:collapse;width:100%;margin:.6rem 0;font-size:.92rem}}
 th,td{{border:1px solid #ddd;padding:.45rem .6rem;text-align:left}}
 th{{background:#f4f6f8}}
 pre{{background:#f6f8fa;padding:.9rem;overflow-x:auto;border-radius:6px;
      font-size:.85rem;border:1px solid #e1e4e8}}
 section.warning{{background:#fff8e1;border-left:4px solid #f0a500;
                  padding:.6rem 1rem;margin:1rem 0}}
 .banner{{background:#fdecea;border-left:4px solid #d93025;padding:.8rem 1rem;
          margin:1rem 0;font-weight:600}}
 .subtitle{{color:#666;font-style:italic}}
</style></head><body>
<h1>{html.escape(self.title)}</h1>
{f'<p class="subtitle">{html.escape(self.subtitle)}</p>' if self.subtitle else ''}
{배너}
{''.join(본문)}
<section><h2>실행 환경</h2><table><tr><th>항목</th><th>값</th></tr>{환경}</table>
<p><small>이 표는 결과를 재현하는 데 필요합니다.</small></p></section>
</body></html>"""

    def save(self, path: str) -> str:
        내용 = self.to_html() if str(path).endswith((".html", ".htm")) else self.to_markdown()
        with open(path, "w", encoding="utf-8") as f:
            f.write(내용)
        return path

    def show(self) -> None:
        """Colab 에서 HTML 로 표시."""
        try:
            from IPython.display import HTML, display

            display(HTML(self.to_html()))
        except Exception:
            print(self.to_markdown())


def _md_table_to_html(md_table: str) -> str:
    """간단한 마크다운 표를 HTML 로. (구분선 행은 버린다)"""
    줄들 = [l for l in md_table.strip().split("\n") if l.strip().startswith("|")]
    if not 줄들:
        return f"<pre>{html.escape(md_table)}</pre>"
    행들 = []
    for i, l in enumerate(줄들):
        칸 = [c.strip() for c in l.strip().strip("|").split("|")]
        if all(set(c) <= set("-: ") for c in 칸):
            continue
        태그 = "th" if i == 0 else "td"
        행들.append("<tr>" + "".join(f"<{태그}>{html.escape(c)}</{태그}>" for c in 칸) + "</tr>")
    return "<table>" + "".join(행들) + "</table>"


In [ ]:
%%writefile statagent.py
"""전산실험2 13주차 : StatAgent — 지금까지의 부품을 전부 합친다.

    CSV 업로드
      → DataProfiler (9주)   진단·정리
      → AnalysisPlanner(9주) 계획 + 규칙 검증
      → StatToolbox  (5주)   도구로 실행 (되면)
      → CodeAgent    (7주)   도구에 없으면 코드로
      → Retriever    (10주)  방법론 근거 검색
      → Critic       (12주)  검토
      → Report       (13주)  해석 검사 + 리포트

설계 원칙 (이 수업에서 계속 말한 것):
  1. **규칙으로 되는 것은 규칙으로.** LLM 은 규칙으로 적을 수 없는 판단에만
  2. **계산은 도구가.** 모델은 무엇을 부를지만 정한다
  3. **조용히 틀리느니 명확하게 실패.** 못 하면 못 한다고 말한다
  4. **어떻게 계산했는지 남긴다.** 재현성

사용 예:
    from statagent import StatAgent
    agent = StatAgent(llm, "sample_clinic.csv")
    result = agent.ask("치료 전후로 혈압이 유의하게 변했나요?")
    result.report.show()
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Any

__all__ = ["AnalysisResult", "StatAgent", "build_ui"]


@dataclass
class AnalysisResult:
    """질문 하나에 대한 분석 전체 기록."""

    question: str
    answer: str = ""
    plan: dict[str, Any] = field(default_factory=dict)
    test_result: dict[str, Any] | None = None
    evidence: list[str] = field(default_factory=list)     # RAG 인용
    route: str = ""                                       # tool / code / failed
    approved: bool | None = None
    issues: list[Any] = field(default_factory=list)       # 해석 검사 결과
    report: Any = None
    elapsed: float = 0.0
    n_llm_calls: int = 0
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.error is None and bool(self.answer)

    def summary(self) -> str:
        상태 = "성공" if self.ok else "실패"
        검토 = {True: "승인", False: "반려", None: "미검토"}[self.approved]
        오류 = sum(1 for i in self.issues if getattr(i, "level", "") == "error")
        return (f"[{상태}] 경로={self.route} 검토={검토} 해석오류={오류} "
                f"{self.elapsed:.1f}초 호출={self.n_llm_calls}")


class StatAgent:
    """CSV 하나에 묶인 통계 분석 에이전트.

    Parameters
    ----------
    llm       : LLM 인스턴스
    data      : CSV 경로 또는 DataFrame
    kb_path   : 방법론 지식베이스 (없으면 RAG 를 끈다)
    use_critic: 12주차 Critic 사용 여부
    auto_clean: 진단 후 자동 정리 여부 (이상치는 여전히 건드리지 않는다)
    """

    def __init__(
        self,
        llm: Any,
        data: Any,
        *,
        kb_path: str | None = None,
        use_critic: bool = True,
        auto_clean: bool = True,
        verbose: bool = True,
    ) -> None:
        import pandas as pd

        from profiler import AnalysisPlanner, DataProfiler
        from tools import StatToolbox

        self.llm = llm
        self.verbose = verbose
        self.use_critic = use_critic

        # --- 1) 적재 ---------------------------------------------------
        self.raw = pd.read_csv(data) if isinstance(data, str) else data.copy()
        self.source = data if isinstance(data, str) else "(DataFrame)"

        # --- 2) 진단 (규칙 기반, LLM 없음) -----------------------------
        self.profiler0 = DataProfiler(self.raw)
        self.clean_log: list[str] = []
        if auto_clean and any(p.fixes for p in self.profiler0.profiles.values()):
            import io
            import contextlib

            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                self.df = self.profiler0.clean(verbose=True)
            self.clean_log = [l.strip(" -") for l in buf.getvalue().splitlines()
                              if l.strip().startswith("-")]
        else:
            self.df = self.raw.copy()

        self.profiler = DataProfiler(self.df)
        self.toolbox = StatToolbox(self.df)
        self.planner = AnalysisPlanner(llm, self.profiler, verbose=False)

        # --- 3) 선택 부품 ---------------------------------------------
        self.retriever = self._load_kb(kb_path)
        self.critic = self._load_critic() if use_critic else None

        if verbose:
            print(f"StatAgent 준비 완료")
            print(f"  데이터   : {self.df.shape[0]}행 {self.df.shape[1]}열 ({self.source})")
            print(f"  정리     : {len(self.clean_log)}건 적용")
            for l in self.clean_log:
                print(f"             - {l}")
            print(f"  도구     : {len(self.toolbox)}개")
            print(f"  지식베이스: {'있음' if self.retriever else '없음'}")
            print(f"  검토자   : {'있음' if self.critic else '없음'}")
            문제 = self.profiler.issues()
            if 문제:
                print(f"  ⚠ 남은 주의사항 {len(문제)}개 열 — profile_report() 로 확인")

    # ----------------------------------------------------------------
    def _load_kb(self, kb_path: str | None):
        if not kb_path:
            return None
        try:
            from rag import TfidfRetriever, chunk_markdown

            텍스트 = open(kb_path, encoding="utf-8").read()
            return TfidfRetriever(chunk_markdown(텍스트, source=kb_path))
        except Exception as exc:
            if self.verbose:
                print(f"  (지식베이스 로드 실패: {type(exc).__name__})")
            return None

    def _load_critic(self):
        try:
            from orchestra import Critic

            return Critic(self.llm)
        except Exception:
            return None

    # ----------------------------------------------------------------
    def profile_report(self) -> str:
        return self.profiler.report()

    # ----------------------------------------------------------------
    def ask(self, question: str, **kw: Any) -> AnalysisResult:
        """질문 하나를 처리한다. **예외를 밖으로 던지지 않는다.**"""
        t0 = time.perf_counter()
        res = AnalysisResult(question=question)
        호출 = 0

        try:
            # ---- 1) 계획 (LLM) + 규칙 검증 ---------------------------
            plan = self.planner.plan(question, **kw)
            호출 += len(plan.get("attempts", []))
            res.plan = {k: v for k, v in plan.items() if k != "attempts"}

            if not plan.get("test"):
                res.route = "failed"
                res.error = plan.get("error", "분석 계획을 세우지 못했습니다.")
                res.answer = (f"이 질문에 맞는 분석 계획을 세우지 못했습니다.\n"
                              f"사유: {res.error}\n"
                              f"질문을 더 구체적으로 적어 주시거나, "
                              f"어떤 열을 비교하고 싶은지 알려 주세요.")
                res.report = self._build_report(res)
                return res

            if self.verbose:
                print(f"  계획: {plan['test']} "
                      f"({plan.get('outcome_column')} ~ "
                      f"{plan.get('group_column') or plan.get('second_column')})")

            # ---- 2) 실행: 도구 우선, 안 되면 코드 ---------------------
            결과, 경로 = self._execute(plan, question, **kw)
            res.test_result, res.route = 결과, 경로
            if 경로 == "code":
                호출 += 3          # CodeAgent 시도 수 근사

            if 결과 is None:
                res.error = "분석 실행에 실패했습니다."
                res.answer = (f"계획은 세웠으나({plan['test']}) 실행에 실패했습니다.\n"
                              f"데이터를 확인해 주세요.")
                res.report = self._build_report(res)
                return res

            # ---- 3) 방법론 근거 검색 (선택) --------------------------
            if self.retriever:
                hits = self.retriever.search(f"{plan['test']} 전제 조건 해석", k=2)
                res.evidence = [f"[{c.id}] {c.title}" for c, s in hits if s > 0.05]

            # ---- 4) 해석 서술 (LLM) ----------------------------------
            해석 = self._interpret(question, plan, 결과, **kw)
            호출 += 1
            res.answer = 해석

            # ---- 5) 검토 (선택) --------------------------------------
            if self.critic:
                근거 = str(결과)[:2000]
                review = self.critic.review(question, 해석, 근거, **kw)
                호출 += 1
                res.approved = review.approved
                if not review.approved:
                    res.answer += ("\n\n[검토 미통과] " + "; ".join(review.problems[:3])
                                   + "\n이 결과를 그대로 사용하기 전에 확인하십시오.")

            res.report = self._build_report(res)
            res.issues = res.report.issues if res.report else []

            # 해석 검사에서 오류가 나왔으면 **사용자에게 알린다.**
            # 리포트 안에만 적어 두면 아무도 안 읽는다.
            해석오류 = [i for i in res.issues if getattr(i, "level", "") == "error"]
            if 해석오류:
                res.answer += ("\n\n[해석 검사 실패] 생성된 해석에 다음 문제가 있습니다:\n"
                               + "\n".join(f"- {i.message}" for i in 해석오류[:3])
                               + "\n이 서술을 그대로 인용하지 마십시오.")

        except Exception as exc:
            res.error = f"{type(exc).__name__}: {exc}"
            안내 = ""
            if isinstance(exc, (ConnectionError, TimeoutError)) or "Connection" in type(exc).__name__:
                안내 = ("\n서버에 연결하지 못했습니다. 잠시 후 다시 시도하거나 "
                       "LLM(provider='gemini') 으로 전환해 보세요.")
            res.answer = f"분석 중 오류가 발생했습니다: {res.error}{안내}"
            res.route = res.route or "failed"
            try:
                res.report = self._build_report(res)
            except Exception:
                pass

        res.elapsed = time.perf_counter() - t0
        res.n_llm_calls = 호출
        return res

    # ----------------------------------------------------------------
    _TOOL_MAP = {
        "independent-t-test": ("compare_two_groups", "t"),
        "mann-whitney-u": ("compare_two_groups", "mannwhitney"),
        "paired-t-test": ("compare_paired", "t"),
        "wilcoxon-signed-rank": ("compare_paired", "wilcoxon"),
        "one-way-anova": ("compare_many_groups", "anova"),
        "kruskal-wallis": ("compare_many_groups", "kruskal"),
        "chi-square": ("test_association", None),
        "pearson-correlation": ("correlate", "pearson"),
        "spearman-correlation": ("correlate", "spearman"),
    }

    def _execute(self, plan: dict, question: str, **kw: Any):
        """도구로 되면 도구로, 안 되면 코드로.

        **자주 쓰는 표준 분석은 도구로 고정한다** (7주차 재현성 문제).
        도구에 없는 것만 코드 생성으로 넘긴다.
        """
        test = plan["test"]
        매핑 = self._TOOL_MAP.get(test)

        if 매핑:
            도구, method = 매핑
            args = self._tool_args(도구, plan, method)
            r = self.toolbox.call(도구, args)
            if r.ok:
                return r.value, "tool"
            if self.verbose:
                print(f"  도구 실패 -> 코드로 전환: {str(r.error)[:80]}")

        return self._execute_by_code(plan, question, **kw), "code"

    @staticmethod
    def _tool_args(도구: str, plan: dict, method: str | None) -> dict:
        out: dict[str, Any] = {}
        o, g, s = (plan.get("outcome_column"), plan.get("group_column"),
                   plan.get("second_column"))
        if 도구 == "compare_paired":
            out = {"before_column": s, "after_column": o}
        elif 도구 == "correlate":
            out = {"column_a": o, "column_b": s}
        elif 도구 == "test_association":
            out = {"column_a": o, "column_b": g}
        else:
            out = {"value_column": o, "group_column": g}
        if method:
            out["method"] = method
        return {k: v for k, v in out.items() if v is not None}

    def _execute_by_code(self, plan: dict, question: str, **kw: Any):
        try:
            from executor import CodeAgent, SafeExecutor

            ex = SafeExecutor({"df": self.df})
            지시 = (f"{question}\n\n"
                    f"검정 방법: {plan['test']}\n"
                    f"결과변수: {plan.get('outcome_column')}\n"
                    f"집단변수: {plan.get('group_column')}\n"
                    f"두 번째 변수: {plan.get('second_column')}\n"
                    f"검정 이름, 표본 수, 통계량, p값, 효과크기를 result 에 담아라.")
            run = CodeAgent(self.llm, ex, max_attempts=3, verbose=False).run(지시)
            if run.success and run.result is not None:
                self._last_code = run.final_code
                return run.result
        except Exception:
            pass
        return None

    # ----------------------------------------------------------------
    def _interpret(self, question: str, plan: dict, result: Any, **kw: Any) -> str:
        """결과를 문장으로 옮긴다.

        **올바른 유의성 표현을 미리 만들어 프롬프트에 넣는다.**
        LLM에게 맡기면 "차이가 없다"고 단정하는 일이 잦다 (13주차 해석 검사).
        """
        from report import phrase_significance

        p = None
        if isinstance(result, dict):
            p = result.get("p_value")

        지침 = ""
        if isinstance(p, (int, float)):
            지침 = ("\n다음 문장을 그대로 포함시켜라 (표현을 바꾸지 마라):\n"
                    f'"{phrase_significance(float(p))}"\n')

        근거 = ""
        if self.retriever:
            hits = self.retriever.search(f"{plan['test']} 해석 전제 조건", k=2)
            if hits and hits[0][1] > 0.05:
                근거 = "\n## 참고 (방법론 문서)\n" + "\n\n".join(
                    c.block(500) for c, _ in hits) + "\n"

        prompt = (
            f"## 질문\n{question}\n\n"
            f"## 수행한 분석\n{plan['test']}\n\n"
            f"## 도구가 계산한 결과 (이 숫자만 쓸 것)\n{result}\n"
            f"{근거}{지침}\n"
            f"위 결과를 3~5문장으로 해석하라.\n"
            f"규칙:\n"
            f"- 도구 결과에 없는 숫자를 지어내지 마라.\n"
            f"- 상관을 인과로 서술하지 마라.\n"
            f"- 유의하지 않은 결과를 '차이가 없다'고 단정하지 마라.\n"
            f"- 효과크기와 표본 수를 함께 언급하라."
        )
        try:
            return self.llm.ask(
                prompt, system="너는 통계 분석 결과를 해석하는 도구다.",
                temperature=0.0, max_tokens=kw.get("max_tokens", 600))
        except Exception as exc:
            return f"(해석 생성 실패: {type(exc).__name__}) 원 결과: {result}"

    # ----------------------------------------------------------------
    def _build_report(self, res: AnalysisResult):
        from report import Report

        rep = Report(f"분석 리포트: {res.question[:50]}",
                     subtitle=f"데이터: {self.source}")
        rep.set_meta(rows=len(self.df), cols=len(self.df.columns),
                     route=res.route, source=self.source)

        if self.clean_log:
            rep.add_section("데이터 정리", "\n".join(f"- {l}" for l in self.clean_log))

        문제 = self.profiler.issues()
        if 문제:
            rep.add_warning(
                "\n".join(f"{k}: {'; '.join(v)}" for k, v in 문제.items()),
                "데이터 주의사항")

        if res.plan.get("test"):
            rep.add_result("분석 계획", {
                "검정": res.plan.get("test"),
                "결과변수": res.plan.get("outcome_column"),
                "집단/두번째 변수": res.plan.get("group_column")
                                  or res.plan.get("second_column"),
                "선택 이유": res.plan.get("reason"),
                "실행 경로": res.route,
            })

        if res.test_result:
            rep.add_result("검정 결과", res.test_result
                           if isinstance(res.test_result, dict)
                           else {"결과": res.test_result})

        if res.answer:
            유의 = None
            효과 = None
            if isinstance(res.test_result, dict):
                유의 = res.test_result.get("significant")
                효과 = res.test_result.get("effect_size") is not None
            rep.add_interpretation(
                res.answer, significant=유의,
                # plan 에 test 키가 있어도 값이 None 일 수 있다. or "" 로 막는다.
                test_kind=res.plan.get("test") or "", has_effect_size=효과)

        if res.evidence:
            rep.add_section("방법론 근거", "\n".join(f"- {e}" for e in res.evidence))

        # 재현성: 어떻게 계산했는지 남긴다
        코드 = getattr(self, "_last_code", None)
        if res.route == "code" and 코드:
            rep.add_code("실행한 코드", 코드)
        elif res.route == "tool" and res.plan.get("test"):
            매핑 = self._TOOL_MAP.get(res.plan["test"])
            if 매핑:
                도구, method = 매핑
                args = self._tool_args(도구, res.plan, method)
                rep.add_code("실행한 도구 호출",
                             f"box.call({도구!r}, {args!r})")

        if res.approved is not None:
            rep.add_section("검토 결과",
                            "검토자 승인" if res.approved
                            else "검토자 반려 — 위 해석을 확인하십시오.")
        return rep

    # ----------------------------------------------------------------
    def ask_many(self, questions: list[str], **kw: Any) -> list[AnalysisResult]:
        out = []
        for q in questions:
            if self.verbose:
                print(f"\n── {q}")
            r = self.ask(q, **kw)
            out.append(r)
            if self.verbose:
                print(f"  {r.summary()}")
        return out


# --------------------------------------------------------------------------
# Gradio UI
# --------------------------------------------------------------------------
def build_ui(llm: Any, *, kb_path: str | None = None, use_critic: bool = True):
    """CSV 를 올리고 질문하는 웹 UI 를 만든다.

    Colab 에서:
        demo = build_ui(llm, kb_path="stat_methods.md")
        demo.launch(share=True, debug=False)
    """
    import gradio as gr

    상태: dict[str, Any] = {"agent": None}

    def 업로드(파일):
        if 파일 is None:
            return "파일을 올려 주세요.", ""
        try:
            상태["agent"] = StatAgent(llm, 파일.name, kb_path=kb_path,
                                     use_critic=use_critic, verbose=False)
            a = 상태["agent"]
            요약 = [f"**{a.df.shape[0]}행 {a.df.shape[1]}열** 을 읽었습니다.", ""]
            if a.clean_log:
                요약 += ["**자동 정리**"] + [f"- {l}" for l in a.clean_log] + [""]
            문제 = a.profiler.issues()
            if 문제:
                요약 += ["**주의사항**"]
                요약 += [f"- `{k}`: {v[0]}" for k, v in list(문제.items())[:6]]
            return "\n".join(요약), a.profile_report()
        except Exception as exc:
            return f"파일을 읽지 못했습니다: {type(exc).__name__}: {exc}", ""

    def 질문(q):
        if 상태["agent"] is None:
            return "먼저 CSV 파일을 올려 주세요.", ""
        if not q or not q.strip():
            return "질문을 입력해 주세요.", ""
        r = 상태["agent"].ask(q.strip())
        상세 = r.report.to_markdown(include_env=False) if r.report else ""
        머리 = f"*{r.summary()}*\n\n"
        return 머리 + r.answer, 상세

    with gr.Blocks(title="StatAgent") as demo:
        gr.Markdown("# StatAgent\nCSV 를 올리고 자연어로 질문하세요.\n\n"
                    "> 자동 분석 결과입니다. **논문에 쓰기 전에 반드시 검토하십시오.**")
        with gr.Row():
            with gr.Column(scale=1):
                파일 = gr.File(label="CSV 파일", file_types=[".csv"])
                업로드결과 = gr.Markdown()
                with gr.Accordion("데이터 진단 상세", open=False):
                    진단 = gr.Textbox(lines=18, show_label=False)
            with gr.Column(scale=2):
                질문상자 = gr.Textbox(label="질문",
                                  placeholder="예) 치료 전후로 혈압이 유의하게 변했나요?")
                보내기 = gr.Button("분석", variant="primary")
                답변 = gr.Markdown(label="답변")
                with gr.Accordion("리포트 전체", open=False):
                    상세 = gr.Markdown()

        파일.change(업로드, 파일, [업로드결과, 진단])
        보내기.click(질문, 질문상자, [답변, 상세])
        질문상자.submit(질문, 질문상자, [답변, 상세])

    return demo


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile sample_clinic.csv
patient_id,age,sex,group,bp_before,bp_after,bmi,smoker,satisfaction,income,region
P0001,40.0,M,control,153.9,156.7,25.1,Y,3,12756.0,Busan
P0002,39.0,F,control,146.9,154.0,24.9,N,3,21248.0,Daegu
P0003,49.0,M,treatment,149.4,121.3,31.5,N,4,24254.0,Daegu
P0004,69.0,F,treatment,161.9,155.9,25.7,N,3,56575.0,Daegu
P0005,64.0,M,treatment,168.8,155.2,23.3,N,3,72272.0,Seoul
P0006,48.0,F,treatment,159.8,142.8,25.8,N,4,38254.0,Daegu
P0007,48.0,M,treatment,167.4,156.9,26.9,N,3,7703.0,Daegu
P0008,67.0,M,treatment,480.0,133.9,22.2,N,2,22419.0,Gwangju
P0009,41.0,M,treatment,134.8,111.0,29.2,N,4,22793.0,Seoul
P0010,63.0,F,control,91.7,82.4,24.5,N,2,44750.0,Daegu
P0011,56.0,F,control,143.2,140.8,27.8,N,5,37430.0,Seoul
P0012,53.0,M,control,133.5,129.2,23.7,Y,3,33852.0,Seoul
P0013,61.0,M,treatment,121.7,109.7,29.1,N,4,41307.0,Daegu
P0014,37.0,F,treatment,160.1,143.6,24.4,N,1,53924.0,Daegu
P0015,48.0,F,control,155.3,161.4,30.8,Y,1,92024.0,Gwangju
P0016,30.0,M,control,141.8,137.4,25.4,Y,3,20452.0,Daegu
P0017,70.0,F,control,112.4,111.2,25.2,N,4,23648.0,Seoul
P0018,57.0,M,control,137.3,135.3,20.6,Y,2,10089.0,Seoul
P0019,46.0,F,treatment,123.7,125.5,24.5,N,2,22951.0,Busan
P0020,73.0,F,treatment,162.2,,23.7,Y,3,39878.0,Busan
P0021,57.0,M,control,144.5,134.2,29.9,N,4,75988.0,Busan
P0022,63.0,F,treatment,164.4,157.8,26.0,N,4,65927.0,Busan
P0023,33.0,M,treatment,115.2,109.3,21.5,N,3,40416.0,Busan
P0024,48.0,M,treatment,125.3,108.5,30.6,N,3,19877.0,Gwangju
P0025,44.0,F,control,156.0,161.7,22.5,N,4,44856.0,Seoul
P0026,68.0,M,treatment,156.3,155.5,23.2,Y,1,26890.0,Seoul
P0027,63.0,M,treatment,135.4,111.0,,N,3,7287.0,Daegu
P0028,59.0,F,treatment,132.9,117.3,,N,2,16354.0,Daegu
P0029,43.0,M,control,152.7,156.8,23.0,N,4,26488.0,Daegu
P0030,69.0,F,control,144.4,133.7,24.8,N,2,10399.0,Gwangju
P0031,35.0,F,control,138.2,132.3,27.7,Y,4,10883.0,Busan
P0032,43.0,F,treatment,135.8,117.1,31.1,N,1,36864.0,Busan
P0033,50.0,F,control,156.0,143.6,28.7,N,1,24430.0,Seoul
P0034,50.0,F,treatment,158.9,146.9,23.8,N,3,22788.0,Daegu
P0035,65.0,F,control,162.9,156.7,29.6,N,1,10734.0,Daegu
P0036,54.0,F,treatment,162.2,143.5,27.4,N,2,136149.0,Busan
P0037,69.0,F,control,133.8,127.7,28.6,N,2,50707.0,Seoul
P0038,37.0,M,control,142.0,149.3,23.9,Y,3,18634.0,Daegu
P0039,52.0,M,treatment,165.0,148.7,26.0,N,1,54997.0,Gwangju
P0040,68.0,F,treatment,132.9,114.7,23.6,N,2,17947.0,Daegu
P0041,41.0,F,control,152.3,146.6,23.3,N,1,42218.0,Seoul
P0042,48.0,F,treatment,152.7,153.0,28.3,N,3,11540.0,Daegu
P0043,58세,F,control,138.4,142.3,27.9,N,1,33251.0,Seoul
P0044,68.0,F,treatment,149.0,122.4,32.2,N,1,57593.0,Daegu
P0045,63.0,F,treatment,160.6,152.0,24.2,N,5,36447.0,Busan
P0046,35.0,M,control,172.7,177.5,28.0,Y,4,12268.0,Gwangju
P0047,45.0,F,treatment,141.6,121.4,21.0,Y,5,100463.0,Daegu
P0048,64.0,F,treatment,142.5,117.9,24.1,N,5,21024.0,Daegu
P0049,62.0,M,treatment,151.6,136.5,29.1,N,4,16482.0,Seoul
P0050,88.0,M,control,126.8,112.0,32.0,Y,2,46835.0,Seoul
P0051,62.0,F,control,147.8,146.1,30.9,Y,5,19540.0,Seoul
P0052,51.0,F,treatment,127.1,105.1,28.3,Y,5,60886.0,Daegu
P0053,61.0,F,control,156.6,,31.1,Y,3,22335.0,Seoul
P0054,63.0,M,control,142.9,137.9,23.4,N,4,21115.0,Gwangju
P0055,44.0,F,control,163.1,165.8,26.2,N,1,17212.0,Daegu
P0056,50.0,F,control,134.5,131.7,30.4,N,2,48016.0,Daegu
P0057,46.0,F,treatment,148.8,134.7,26.7,N,2,5933.0,Seoul
P0058,43.0,M,treatment,149.4,127.6,25.2,N,3,55878.0,Busan
P0059,69.0,F,control,145.2,136.6,24.0,N,5,22919.0,Seoul
P0060,39.0,F,control,159.1,153.1,30.4,Y,3,38223.0,Daegu
P0061,52.0,M,treatment,135.1,135.7,27.0,N,3,28871.0,Gwangju
P0062,53.0,M,control,150.1,147.0,16.5,N,5,15375.0,Daegu
P0063,37.0,M,control,144.8,136.7,23.1,N,1,17474.0,Gwangju
P0064,55.0,M,treatment,137.7,126.9,29.0,Y,4,15744.0,Busan
P0065,57.0,F,treatment,166.2,153.8,20.1,N,4,25969.0,Gwangju
P0066,55.0,F,treatment,145.1,137.3,24.0,Y,3,41480.0,Busan
P0067,51.0,F,control,140.6,136.7,25.2,N,4,9821.0,Busan
P0068,68.0,F,treatment,161.3,,24.2,N,2,21397.0,Gwangju
P0069,58.0,M,control,143.2,139.3,16.0,N,1,41523.0,Busan
P0070,86.0,F,control,145.2,146.3,27.0,Y,5,47069.0,Seoul
P0071,77.0,F,control,136.6,150.1,30.5,Y,5,11059.0,Seoul
P0072,69.0,M,control,157.1,156.2,24.5,Y,4,27664.0,Gwangju
P0073,47.0,M,control,156.2,155.2,20.7,Y,5,43633.0,Daegu
P0074,43.0,F,control,133.9,132.4,24.6,Y,4,28681.0,Daegu
P0075,62.0,F,treatment,132.1,125.4,32.7,N,2,24491.0,Busan
P0076,73.0,F,control,152.9,152.8,25.9,N,1,17686.0,Daegu
P0077,55.0,M,control,155.6,143.1,21.2,Y,4,7742.0,Daegu
P0078,51.0,M,treatment,140.6,127.6,26.8,N,3,34283.0,Seoul
P0079,52.0,F,treatment,153.8,141.9,25.0,Y,5,8775.0,Daegu
P0080,51.0,M,control,163.9,145.5,,Y,2,37663.0,Daegu
P0081,52.0,M,treatment,137.3,120.4,22.6,Y,2,22161.0,Seoul
P0082,41.0,F,control,159.0,145.6,20.8,N,4,54669.0,Daegu
P0083,53.0,M,control,135.9,137.9,30.1,N,1,30137.0,Seoul
P0084,49.0,M,control,152.6,150.7,32.6,Y,4,43018.0,Seoul
P0085,48.0,F,control,145.7,149.6,28.8,N,3,42605.0,Daegu
P0086,46.0,F,control,152.8,152.3,30.5,N,4,20892.0,Busan
P0087,69.0,M,treatment,168.0,146.2,28.4,Y,4,39125.0,Daegu
P0088,68.0,F,control,134.1,135.3,24.6,N,5,25873.0,Seoul
P0089,42.0,M,control,126.5,120.9,23.6,N,1,27999.0,Daegu
P0090,50.0,F,treatment,153.2,144.1,28.9,N,5,38156.0,Gwangju
P0091,53.0,M,control,167.5,157.8,24.4,Y,5,18802.0,Daegu
P0092, 47 ,F,control,143.4,139.1,30.8,N,2,49663.0,Seoul
P0093,55.0,M,control,127.7,123.9,20.6,Y,5,71117.0,Daegu
P0094,64.0,F,control,115.5,113.1,27.1,N,4,18509.0,Gwangju
P0095,69.0,M,control,156.0,159.8,26.6,N,3,44618.0,Daegu
P0096,50.0,F,treatment,165.2,153.3,25.9,Y,2,21331.0,Daegu
P0097,46.0,F,treatment,126.7,,25.2,N,4,26323.0,Busan
P0098,57.0,F,control,133.5,133.3,26.9,N,5,36500.0,Daegu
P0099,51.0,F,control,147.6,148.3,21.9,N,1,21461.0,Seoul
P0100,53.0,F,treatment,143.4,132.1,24.7,N,4,30403.0,Gwangju
P0101,71.0,F,treatment,157.0,153.9,24.9,N,2,32798.0,Busan
P0102,48.0,F,control,161.7,157.3,23.2,Y,3,47229.0,Busan
P0103,63.0,F,treatment,167.4,163.1,28.6,N,2,9597.0,Busan
P0104,56.0,F,treatment,136.4,117.3,27.9,N,4,27215.0,Daegu
P0105,62.0,F,control,144.3,141.1,26.9,N,5,40688.0,Daegu
P0106,41.0,F,treatment,162.2,151.1,30.6,N,4,25559.0,Seoul
P0107,54.0,F,treatment,144.2,128.9,26.7,N,4,37429.0,Daegu
P0108,28.0,F,control,148.0,143.9,23.8,N,5,40623.0,Daegu
P0109,59.0,F,control,147.8,147.4,23.4,N,1,25702.0,Daegu
P0110,54.0,F,control,168.9,175.6,26.7,Y,2,78180.0,Seoul
P0111,49.0,F,control,175.1,170.3,29.2,N,5,57014.0,Seoul
P0112,71.0,M,control,139.5,143.8,19.4,Y,4,13156.0,Gwangju
P0113,56.0,M,treatment,170.8,148.1,30.2,N,4,12769.0,Gwangju
P0114,66.0,F,control,130.1,130.0,24.4,N,1,41639.0,Gwangju
P0115,59.0,F,control,144.4,144.5,27.8,N,1,31583.0,Daegu
P0116,49.0,M,control,151.1,160.5,32.5,Y,2,95215.0,Gwangju
P0117,57.0,M,treatment,145.5,130.9,26.1,Y,1,25382.0,Daegu
P0118,52.0,F,control,137.3,,,N,4,23605.0,Daegu
P0119,31.0,F,control,132.7,141.8,24.8,N,2,6766.0,Seoul
P0120,64.0,F,treatment,119.6,101.6,29.3,N,5,22762.0,Daegu
P0121,72.0,M,control,165.0,156.9,23.2,N,1,12619.0,Gwangju
P0122,75.0,M,control,173.9,162.4,22.9,Y,4,11479.0,Seoul
P0123,49.0,M,control,142.6,135.5,24.1,Y,3,74950.0,Busan
P0124,66.0,F,treatment,167.8,154.9,19.7,N,3,49309.0,Daegu
P0125,37.0,F,control,168.1,163.2,27.1,N,2,37980.0,Busan
P0126,68.0,F,control,167.5,170.8,24.0,N,5,14773.0,Busan
P0127,55.0,F,treatment,119.6,,23.5,N,3,38653.0,Gwangju
P0128,70.0,F,control,153.7,147.5,25.6,N,5,15347.0,Daegu
P0129,63.0,F,treatment,175.8,154.4,22.3,N,1,21481.0,Gwangju
P0130,45.0,F,control,145.8,146.8,25.7,N,3,65265.0,Busan
P0131,51.0,M,treatment,157.3,150.8,25.0,N,5,11772.0,Daegu
P0132,49.0,F,treatment,146.1,136.6,30.7,N,4,35350.0,Daegu
P0133,61.0,M,control,161.7,157.6,23.8,Y,4,39063.0,Daegu
P0134,68.0,M,treatment,124.9,9.0,27.2,N,5,31211.0,Daegu
P0135,35.0,F,treatment,121.8,121.1,26.0,Y,2,26634.0,Daegu
P0136,46.0,M,treatment,146.1,128.2,29.0,N,2,18184.0,Seoul
P0137,59.0,F,treatment,164.9,170.6,22.3,N,2,14512.0,Seoul
P0138,47.0,M,treatment,146.2,145.5,22.6,N,3,38313.0,Gwangju
P0139,52.0,M,treatment,128.1,114.8,26.9,N,5,153003.0,Daegu
P0140,53.0,M,control,159.3,150.5,23.7,N,3,31681.0,Seoul
P0141,56.0,M,control,152.8,150.4,26.1,Y,1,11898.0,Busan
P0142,60.0,M,control,134.9,139.9,28.5,Y,3,14893.0,Gwangju
P0143,53.0,F,treatment,154.9,133.3,28.1,Y,2,26853.0,Seoul
P0144,59.0,M,treatment,173.3,161.8,22.0,N,1,26175.0,Daegu
P0145,40.0,M,treatment,128.9,,26.3,N,4,18401.0,Daegu
P0146,49.0,M,control,121.1,109.8,23.3,N,3,29908.0,Seoul
P0147,48.0,M,treatment,132.0,117.1,22.8,N,2,21847.0,Daegu
P0148,65.0,M,treatment,137.5,,27.4,Y,3,40174.0,Gwangju
P0149,51.0,F,treatment,154.0,152.7,21.5,N,2,20761.0,Daegu
P0150,61.0,F,control,156.5,147.6,19.7,Y,4,30679.0,Busan
P0151,29.0,M,control,134.7,129.5,29.5,Y,5,10778.0,Daegu
P0152,38.0,M,control,130.7,124.8,26.4,N,1,39255.0,Daegu
P0153,36.0,F,treatment,121.2,111.4,21.9,N,2,70491.0,Seoul
P0154,61.0,M,control,169.9,167.4,20.4,N,3,31538.0,Daegu
P0155,45.0,F,treatment,177.4,170.8,26.4,N,1,35181.0,Seoul
P0156,59.0,M,treatment,146.7,127.6,16.0,N,4,19414.0,Seoul
P0157,44.0,M,control,161.3,,,N,5,50047.0,Daegu
P0158,56.0,M,control,151.4,151.0,27.7,N,4,77464.0,Daegu
P0159,49.0,F,treatment,148.0,132.3,22.1,Y,3,28447.0,Seoul
P0160,38.0,F,control,126.6,125.9,26.6,N,1,25910.0,Seoul
P0161,63.0,F,treatment,137.1,116.3,32.5,N,1,59235.0,Seoul
P0162,57.0,F,treatment,141.5,126.4,29.3,N,5,25987.0,Gwangju
P0163,58.0,M,treatment,167.1,156.4,30.1,N,1,17583.0,Daegu
P0164,59.0,F,treatment,153.1,140.0,16.0,N,1,24298.0,Daegu
P0165,48.0,M,treatment,137.0,121.3,25.6,N,5,17519.0,Busan
P0166,55.0,F,control,168.1,161.8,24.3,N,1,43619.0,Busan
P0167,45.0,F,treatment,135.0,118.8,25.8,Y,2,22110.0,Daegu
P0168,60.0,M,treatment,147.4,135.6,29.3,N,4,30704.0,Daegu
P0169,37.0,F,control,135.1,128.0,32.3,N,4,16009.0,Seoul
P0170,76.0,M,control,172.8,175.9,26.5,N,3,23731.0,Daegu
P0171,60.0,M,treatment,153.8,136.3,23.7,N,4,18904.0,Daegu
P0172,58.0,F,control,130.9,125.2,22.3,Y,1,14560.0,Seoul
P0173,43.0,M,treatment,142.7,128.0,20.9,N,1,19676.0,Gwangju
P0174,74.0,F,treatment,138.1,130.7,24.6,N,1,22869.0,Seoul
P0175,61.0,F,control,145.6,129.1,22.7,Y,2,42134.0,Seoul
P0176,65.0,M,treatment,126.7,118.6,20.2,N,1,129301.0,Busan
P0177,62.0,M,treatment,150.3,136.0,27.8,N,3,55536.0,Daegu
P0178,62.0,F,treatment,146.7,134.0,20.3,N,2,24833.0,Daegu
P0179,70.0,F,control,153.7,141.8,30.0,N,1,20660.0,Seoul
P0180,54.0,F,treatment,143.6,130.4,29.4,Y,2,22577.0,Busan


In [ ]:
%%writefile exam_greenhouse.csv
pot_id,fertilizer,greenhouse,yield_before,yield_after,water_liters,temp_c,leaf_count,pest_damage
G0001,C,south,63.63,63.91,2.33,25.8,30.0,No
G0002,C,north,44.55,38.06,2.23,22.1,22.0,no
G0003,A,north,28.47,28.83,2.85,20.5,30.0,no
G0004,B,south,60.01,58.01,0.78,-999,34.0,No
G0005,B,south,68.94,85.16,1.57,25.4,34.0,No
G0006,A,south,41.56,48.18,"2,58",25.1,32.0,no
G0007,A,south,48.84,48.73,2.5,25.0,47.0,No
G0008,A,south,56.79,56.91,1.73,21.4,31.0,No
G0009,A,south,39.05,42.84,2.04,22.7,33.0,No
G0010,A,south,58.79,60.51,2.12,-999,35.0,Yes
G0011,B,north,61.89,71.84,2.01,20.3,27.0,No
G0012,A,north,53.67,61.11,2.89,23.7,32.0,Yes
G0013,A,south,52.87,57.02,3.55,24.6,29.0,Y
G0014,C,north,49.44,43.73,1.72,22.0,49.0,no
G0015,C,south,44.32,51.82,2.98,25.0,36.0,no
G0016,C,north,43.19,41.9,2.32,23.0,26.0,no
G0017,A,south,56.04,56.93,1.97,27.0,28.0,no
G0018,A,south,58.25,68.38,2.25,22.2,22.0,no
G0019,B,south,66.43,81.83,2.32,27.6,33.0,No
G0020,B,north,512.0,67.2,2.56,29.7,27.0,No
G0021,A,south,44.29,42.75,3.57,27.7,32.0,No
G0022,A,north,34.38,34.04,2.06,20.3,24.0,no
G0023,A,south,55.24,62.74,2.59,20.5,33.0,No
G0024,C,north,53.45,60.79,3.19,22.4,26.0,No
G0025,A,south,55.04,61.16,2.89,21.4,26.0,no
G0026,A,south,66.51,66.72,2.26,28.2,36.0,no
G0027,C,north,43.58,50.97,2.79,26.9,26.0,No
G0028,B,south,59.41,67.56,2.62,16.8,30.0,no
G0029,B,north,63.05,69.99,2.97,22.7,36.0,No
G0030,B,north,53.61,60.34,3.61,20.5,37.0,no
G0031,A,north,49.31,50.56,1.38,27.2,34.0,No
G0032,B,south,51.58,61.88,2.52,-999,27.0,No
G0033,B,south,52.52,59.53,2.43,23.6,,No
G0034,B,south,64.49,68.34,2.16,24.9,30.0,no
G0035,C,north,44.66,51.11,2.95,19.3,35.0,no
G0036,C,south,54.42,63.11,3.35,26.2,31.0,No
G0037,C,south,46.66,53.42,1.82,-999,30.0,No
G0038,A,north,47.05,47.01,2.56,22.2,24.0,no
G0039,B,south,69.03,73.57kg,2.6,22.5,19.0,no
G0040,A,south,52.02,56.99,2.2,24.5,34.0,no
G0041,B,north,73.02,83.96,2.35,-999,,No
G0042,B,north,57.19,67.53,2.21,22.3,38.0,No
G0043,C,south,48.89,51.07,3.31,28.4,22.0,no
G0044,C,south,45.51,43.91,2.03,25.5,35.0,No
G0045,C,south,73.1,72.83,2.67,22.4,36.0,no
G0046,A,south,49.77,53.55,2.16,21.1,26.0,no
G0047,A,south,50.02,54.03,1.98,20.2,25.0,no
G0048,B,north,52.5,60.99,2.18,25.8,22.0,No
G0049,C,south,47.19,48.26,2.22,27.9,32.0,No
G0050,C,south,52.92,57.25,"2,28",25.5,39.0,yes
G0051,A,south,52.14,50.21,3.8,24.8,27.0,No
G0052,C,north,58.18,53.26,3.48,21.7,24.0,No
G0053,B,north,60.12,72.63,1.81,21.3,38.0,No
G0054,C,south,48.38,45.24,2.59,-999,17.0,No
G0055,A,south,68.79,71.0,2.87,25.7,27.0,no
G0056,C,south,55.3,54.23,1.96,22.3,40.0,No
G0057,B,north,54.27,59.11,2.5,20.2,30.0,no
G0058,A,north,51.87,51.74,2.79,23.9,39.0,no
G0059,B,south,57.39,72.06,2.1,24.4,26.0,no
G0060,A,north,55.27,59.75,2.22,27.7,31.0,No
G0061,C,south,40.18,46.98,1.55,22.6,34.0,Y
G0062,A,north,42.66,49.74,1.95,25.9,34.0,yes
G0063,C,south,43.26,41.39,2.6,24.1,27.0,No
G0064,A,south,48.12,51.89,1.6,24.6,34.0,No
G0065,B,north,52.48,52.5,1.57,23.5,31.0,No
G0066,A,north,42.64,42.22,2.19,21.4,28.0,No
G0067,B,south,59.28,63.65,"1,91",22.8,37.0,no
G0068,A,north,49.49,51.99,2.58,23.4,40.0,No
G0069,B,south,60.73,65.78,2.81,19.2,31.0,No
G0070,C,north,58.42,63.93,2.28,27.5,30.0,No
G0071,A,south,56.53,55.65,2.68,19.5,40.0,No
G0072,A,north,42.11,44.37,2.54,24.3,31.0,No
G0073,A,north,52.74,51.42,1.51,24.4,23.0,yes
G0074,C,north,50.85,54.03,2.31,25.1,42.0,No
G0075,C,south,47.49,54.63,1.91,27.1,31.0,no
G0076,A,north,43.56,47.87,3.95,22.5,35.0,Yes
G0077,C,south,52.88,49.8,3.15,27.3,37.0,Yes
G0078,B,south,67.53,72.61,2.61,23.4,31.0,yes
G0079,B,north,56.41,60.02,2.77,22.3,31.0,no
G0080,B,north,62.92,58.97,3.46,26.3,28.0,No
G0081,B,south,63.73,66.0,2.28,27.9,36.0,No
G0082,C,south,45.18,46.22,1.4,25.0,28.0,No
G0083,B,south,59.25,67.01,2.2,24.6,30.0,No
G0084,C,south,59.92,67.33,1.85,18.8,30.0,Y
G0085,A,north,52.66,53.71,1.62,25.0,,no
G0086,C,south,47.7,42.38,2.9,28.3,26.0,no
G0087,A,south,60.8,66.25,2.82,27.7,29.0,No
G0088,B,north,68.3,74.46,2.34,27.4,24.0,No
G0089,A,north,41.89,47.62,2.21,24.2,30.0,Yes
G0090,C,north,52.73,55.13,2.61,28.4,29.0,no
G0091,A,north,52.56,56.19,2.46,25.1,35.0,Yes
G0092,B,north,57.11,58.33,1.96,21.5,40.0,yes
G0093,A,north,44.24,47.81,2.67,18.9,28.0,no
G0094,A,south,55.45,57.43,2.66,26.2,27.0,no
G0095,B,north,58.35,64.22,2.01,18.6,44.0,yes
G0096,B,south,59.22,72.91,1.69,23.3,32.0,No
G0097,B,south,56.92,59.72,2.48,25.0,31.0,No
G0098,B,south,71.57,79.2,"3,06",25.6,28.0,No
G0099,C,south,65.59,63.09,2.72,26.9,36.0,yes
G0100,C,south,55.92,53.6,1.83,25.1,37.0,No
G0101,B,north,56.99,55.43,0.61,23.4,28.0,No
G0102,C,south,55.58,53.03,1.95,26.1,21.0,no
G0103,B,south,57.01,69.18,"2,53",23.2,35.0,yes
G0104,B,north,53.63,60.25,1.88,28.0,27.0,No
G0105,C,south,49.91,51.07,2.38,20.6,26.0,No
G0106,C,south,42.83,43.04,1.89,26.9,35.0,yes
G0107,A,north,52.26,47.42,2.62,23.5,29.0,no
G0108,C,north,47.88,49.17,1.72,23.9,36.0,No
G0109,A,south,53.71,55.9,2.51,25.0,31.0,Y
G0110,C,north,46.1,48.32,1.77,25.6,26.0,No
G0111,B,south,57.65,64.51,2.09,23.8,35.0,No
G0112,C,north,54.48,53.27,2.74,25.1,28.0,No
G0113,A,north,54.8,53.37,2.84,23.8,25.0,no
G0114,B,south,61.46,64.15,2.2,24.7,23.0,No
G0115,C,north,37.61,42.91,2.76,24.9,28.0,No
G0116,A,north,47.21,52.37,2.33,30.1,34.0,no
G0117,B,north,56.41,61.96,"3,56",26.9,33.0,Yes
G0118,B,south,64.33,60.61,2.11,22.5,22.0,No
G0119,B,north,55.62,65.12,1.32,26.7,33.0,No
G0120,A,south,56.54,56.58,1.44,21.1,,no
G0121,A,south,55.05,54.88,3.2,21.2,22.0,no
G0122,A,north,49.31,56.34,2.88,26.8,27.0,no
G0123,C,south,61.84,66.2,2.34,25.2,41.0,No
G0124,B,north,48.53,56.65kg,2.04,25.2,25.0,No
G0125,A,south,37.91,39.22,2.66,29.2,30.0,no
G0126,B,north,59.34,65.72,2.84,23.7,33.0,Yes
G0127,B,north,68.35,74.41,2.0,23.3,36.0,No
G0128,A,north,57.22,48.59,1.38,23.7,24.0,no
G0129,A,south,51.69,49.64,3.01,21.3,32.0,No
G0130,C,south,60.8,62.09,1.73,31.1,29.0,No
G0131,B,south,57.66,60.29,2.77,26.8,32.0,no
G0132,A,north,46.24,52.36,2.73,22.2,33.0,yes
G0133,B,north,57.4,55.62,2.07,27.8,27.0,no
G0134,C,south,36.04,45.16,2.53,22.4,32.0,No
G0135,B,south,51.75,58.96,2.59,21.2,30.0,No
G0136,C,south,59.84,58.73,3.11,23.7,30.0,no
G0137,A,south,66.09,60.19,0.98,22.7,33.0,No
G0138,B,south,59.66,63.69,3.35,21.6,41.0,No
G0139,B,south,60.55,64.43,2.05,28.1,22.0,yes
G0140,B,north,68.88,62.12,2.17,17.6,30.0,Yes
G0141,B,south,72.84,78.61,2.87,26.1,36.0,no
G0142,C,north,49.3,52.61,2.61,23.7,40.0,No
G0143,C,north,50.82,55.4,2.04,24.9,28.0,no
G0144,A,south,59.18,61.6,2.45,26.9,32.0,No
G0145,A,south,53.25,64.61,1.31,28.1,900.0,yes
G0146,A,south,52.95,53.67,3.14,21.1,32.0,No
G0147,A,south,50.76,50.9,2.59,27.2,26.0,No
G0148,A,south,64.77,65.06,2.83,23.0,34.0,no
G0149,A,north,45.39,50.96kg,2.27,20.3,34.0,yes
G0150,C,south,46.53,48.82,2.4,23.2,,yes
G0151,A,south,54.04,58.03,1.67,26.2,33.0,No
G0152,B,north,52.18,60.16kg,2.4,21.5,39.0,no
G0153,A,north,40.83,38.31,2.83,19.5,41.0,no
G0154,B,south,64.8,72.95,1.64,24.7,40.0,Yes
G0155,B,north,35.3,54.17,2.3,25.9,34.0,No
G0156,C,north,49.58,46.72,2.66,21.5,25.0,No
G0157,C,south,48.37,51.85,2.61,18.9,25.0,Yes
G0158,A,north,45.83,43.47,2.56,23.4,34.0,Yes
G0159,C,south,41.96,47.06,2.46,27.3,25.0,Yes
G0160,B,north,59.19,63.6,2.72,24.4,34.0,No
G0161,B,south,55.91,67.17,1.38,23.0,30.0,No
G0162,A,north,56.61,60.92,2.78,23.6,29.0,no
G0163,B,north,50.27,57.73,"2,48",24.1,29.0,no
G0164,C,south,59.15,61.58,2.17,24.4,42.0,yes
G0165,B,north,50.53,51.35,2.85,26.0,32.0,No
G0166,B,north,56.1,65.7,2.72,22.1,29.0,no
G0167,A,north,51.51,45.5,1.63,21.5,29.0,no
G0168,A,north,39.29,39.65kg,2.26,-999,26.0,yes
G0169,A,south,45.81,49.61,1.42,17.4,,Yes
G0170,A,south,51.33,60.67,2.61,23.6,25.0,No
G0171,B,north,58.03,65.13,1.56,26.8,33.0,no
G0172,A,south,47.5,41.3,3.7,27.3,28.0,Yes
G0173,C,south,62.38,55.42,1.75,27.3,24.0,no
G0174,A,north,49.02,51.32,2.11,21.8,34.0,No
G0175,A,south,52.3,61.08,1.97,25.7,37.0,No
G0176,B,south,57.87,67.41,"2,91",25.1,22.0,Y
G0177,C,south,68.87,70.77,2.03,28.3,,No
G0178,A,south,57.68,64.22,2.49,29.2,27.0,Yes
G0179,A,north,39.51,44.83,2.64,23.0,27.0,No
G0180,A,north,38.79,37.13,2.65,28.7,39.0,No
G0181,A,north,70.14,65.83kg,2.45,20.8,33.0,No
G0182,C,north,57.1,59.82,1.83,22.4,26.0,No
G0183,B,north,60.78,66.46,2.55,25.1,36.0,Yes
G0184,B,south,67.57,74.94,2.88,-999,37.0,No
G0185,B,north,51.63,51.22,2.99,26.1,37.0,Yes
G0186,C,north,51.58,51.87,2.31,22.7,39.0,No
G0187,A,north,43.53,43.3,2.58,28.3,19.0,No
G0188,B,north,59.97,58.43,3.95,22.9,30.0,No
G0189,A,north,54.2,55.23,2.46,26.9,29.0,No
G0190,C,south,44.75,47.68,2.29,22.7,31.0,No
G0191,B,south,50.15,59.47,2.15,21.5,34.0,no
G0192,A,south,51.78,64.11,2.41,24.5,36.0,no
G0193,A,south,52.26,50.29,2.39,27.6,24.0,No
G0194,B,south,50.26,58.03,2.07,25.5,32.0,No
G0195,B,north,48.43,50.39,2.28,-999,34.0,No
G0196,C,south,47.54,53.9,2.97,26.1,41.0,No
G0197,B,south,58.36,71.86,2.7,-999,32.0,No
G0198,B,north,65.57,71.83,3.17,23.4,18.0,no
G0199,A,south,53.28,54.1,2.63,16.6,23.0,yes
G0200,A,south,45.83,51.69,1.74,24.5,30.0,No
G0201,A,south,50.09,50.18,1.9,23.1,41.0,No
G0202,B,north,63.8,70.25,2.58,-999,31.0,no
G0203,C,south,63.12,57.35,2.77,21.9,33.0,yes
G0204,B,south,57.3,61.12,3.19,24.8,30.0,No
G0205,C,north,38.13,42.29,0.9,18.6,34.0,no
G0206,C,south,51.98,53.82,3.19,25.4,27.0,No
G0207,A,south,52.94,57.02,2.41,25.7,29.0,No
G0208,B,north,56.92,65.96,1.89,25.9,41.0,No
G0209,C,south,65.93,68.41,3.13,20.5,24.0,No
G0210,B,south,57.97,64.46,2.71,26.8,26.0,No


In [ ]:
%%writefile stat_methods.md
# 통계 방법론 참고 문서

전산실험2 10주차 RAG 실습용 지식베이스.
에이전트가 검정을 고를 때 **근거로 인용할 문서**입니다.

각 절이 하나의 청크가 됩니다. 절 제목을 검색 단서로 삼도록 작성했습니다.

---

## 독립표본 t검정 (independent samples t-test)

서로 다른 두 집단의 평균을 비교할 때 쓴다. 예를 들어 치료군과 대조군의
혈압 평균이 다른지 확인하는 경우다.

**전제 조건**
1. 두 집단이 서로 독립이다. 같은 대상을 두 번 측정한 것이 아니다.
2. 결과변수가 연속형이다.
3. 각 집단에서 결과변수가 정규분포를 따른다.
4. 두 집단의 분산이 같다 (등분산). 이 가정이 깨지면 Welch 보정을 쓴다.

**등분산 가정에 대하여**
등분산 검정(Levene, Bartlett)을 먼저 하고 t검정을 고르는 2단계 절차는
1종 오류율을 부풀린다는 비판이 있다. 현대적 권고는 **처음부터 Welch t검정을
쓰는 것**이다. Welch는 등분산일 때도 성능 손실이 거의 없다.
scipy에서는 `ttest_ind(a, b, equal_var=False)`이다.

**정규성이 깨지면**
표본이 각 집단 30 이상이면 중심극한정리로 t검정이 비교적 견고하다.
표본이 작거나 치우침이 심하면 Mann-Whitney U 검정을 쓴다.

**효과크기**
Cohen's d를 보고한다. 0.2 작음, 0.5 중간, 0.8 큼이 관례적 기준이지만
분야마다 다르므로 맥락에서 해석해야 한다. p값만 보고하지 말 것.

---

## 대응표본 t검정 (paired samples t-test)

같은 대상을 두 번 측정한 값을 비교할 때 쓴다. 치료 전후 혈압,
같은 학생의 사전·사후 점수 같은 경우다.

**독립표본과 혼동하지 말 것**
가장 흔한 실수다. 같은 사람에게서 나온 두 값은 독립이 아니다.
독립표본 t검정을 쓰면 개인차가 오차에 섞여 들어가 검정력을 잃는다.
설계가 대응인지 독립인지는 **데이터가 아니라 실험 설계**가 결정한다.

**전제 조건**
1. 짝지어진 관측이다.
2. 두 값의 **차이**가 정규분포를 따른다. 각 값이 정규일 필요는 없다.
3. 차이가 연속형이다.

**정규성이 깨지면**
Wilcoxon 부호순위 검정을 쓴다. 이상치 하나가 차이의 정규성을 깨뜨리는
경우가 흔하므로, 정규성 검정 전에 이상치를 먼저 확인해야 한다.

**효과크기**
Cohen's dz = 차이의 평균 / 차이의 표준편차.
또는 차이의 중앙값과 신뢰구간을 보고한다.

---

## Mann-Whitney U 검정

독립표본 t검정의 비모수 대안이다. 정규성 가정이 깨졌거나 결과변수가
순서형일 때 쓴다.

**무엇을 검정하는가**
흔한 오해가 있다. Mann-Whitney는 "중앙값이 같다"를 검정하는 것이 **아니다**.
두 분포에서 무작위로 하나씩 뽑았을 때 한쪽이 클 확률이 1/2인지를 본다.
두 분포의 모양이 같다고 가정할 수 있을 때만 중앙값 비교로 해석할 수 있다.

**전제 조건**
1. 두 집단이 독립이다.
2. 결과변수가 최소 순서형이다.
3. (중앙값 해석을 원하면) 두 분포의 모양이 비슷하다.

**효과크기**
순위이연상관(rank-biserial correlation)을 보고한다.

---

## Wilcoxon 부호순위 검정 (signed-rank test)

대응표본 t검정의 비모수 대안이다. 차이의 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 짝지어진 관측이다.
2. 차이의 분포가 0을 중심으로 대칭이다.

두 번째 조건이 깨지면 부호검정(sign test)을 고려한다.
부호검정은 대칭성을 요구하지 않지만 검정력이 낮다.

**차이가 0인 짝**
차이가 정확히 0인 관측은 순위를 매길 수 없다. scipy는 기본적으로
제외한다(`zero_method='wilcox'`). 그런 짝이 많으면 결과 해석에 주의한다.

---

## 일원분산분석 (one-way ANOVA)

세 집단 이상의 평균을 비교할 때 쓴다.

**두 집단에 쓰지 말 것**
두 집단이면 t검정을 쓴다. 수학적으로 F = t²이라 결과는 같지만,
효과의 방향을 알 수 없고 신뢰구간을 얻기 어렵다.

**왜 t검정을 여러 번 하면 안 되는가**
세 집단에 t검정을 세 번 하면 적어도 하나에서 1종 오류가 날 확률이
0.05가 아니라 약 0.14로 오른다. 이것을 다중비교 문제라고 한다.
ANOVA는 "적어도 하나의 집단이 다르다"를 한 번에 검정한다.

**전제 조건**
1. 집단이 서로 독립이다.
2. 각 집단에서 결과변수가 정규분포를 따른다.
3. 집단 간 분산이 같다 (등분산). Levene 검정으로 확인한다.

**등분산이 깨지면**
Welch ANOVA를 쓴다. `scipy.stats`에는 없고 `pingouin.welch_anova`나
직접 구현이 필요하다. 또는 Kruskal-Wallis를 쓴다.

**유의하면 그 다음**
ANOVA는 "어딘가 다르다"만 알려준다. 어느 집단끼리 다른지는
사후검정(post-hoc)으로 확인한다. Tukey HSD가 표준적이다.
사후검정 없이 "B가 A보다 높다"고 말하면 안 된다.

**효과크기**
에타제곱(eta squared) 또는 오메가제곱을 보고한다.

---

## Kruskal-Wallis 검정

일원분산분석의 비모수 대안이다. 세 집단 이상, 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 집단이 독립이다.
2. 결과변수가 최소 순서형이다.

**유의하면 그 다음**
Dunn 검정으로 사후 비교를 한다. 다중비교 보정(Bonferroni, Holm)을
반드시 적용한다.

**효과크기**
엡실론제곱(epsilon squared)을 보고한다.

---

## 카이제곱 독립성 검정 (chi-square test of independence)

두 범주형 변수가 서로 연관되어 있는지 검정한다.
흡연 여부와 질병 발생의 관계 같은 경우다.

**전제 조건**
1. 관측이 서로 독립이다. 같은 사람이 두 번 세어지면 안 된다.
2. **모든 칸의 기대빈도가 5 이상**이다. 이것이 가장 자주 깨지는 조건이다.

**기대빈도가 5 미만이면**
2×2 표에서는 Fisher 정확검정을 쓴다. 더 큰 표에서는 범주를 합치거나
몬테카를로 방법으로 p값을 구한다. `scipy.stats.chi2_contingency`가
반환하는 `expected` 배열을 반드시 확인할 것.

**연속성 보정**
2×2 표에서 Yates 연속성 보정을 적용할지에 대해 논쟁이 있다.
scipy는 기본적으로 적용한다(`correction=True`). 보수적이라는 비판이 있으므로
보고할 때 어느 쪽을 썼는지 명시한다.

**대응 설계라면**
같은 대상을 두 번 측정한 범주형 자료(치료 전후 개선 여부 등)에는
McNemar 검정을 쓴다. 카이제곱을 쓰면 독립 가정이 깨진다.

**효과크기**
Cramer's V를 보고한다. 표 크기에 따라 해석 기준이 다르다.

---

## 피어슨 상관 (Pearson correlation)

두 연속형 변수의 **선형** 관계 강도를 잰다.

**전제 조건**
1. 두 변수가 모두 연속형이다.
2. 관계가 선형이다. 산점도를 반드시 먼저 그려 볼 것.
3. 이변량 정규성. 특히 이상치에 매우 민감하다.

**흔한 오해**
상관계수 0은 "관계가 없다"가 아니라 "**선형** 관계가 없다"는 뜻이다.
U자 관계는 상관이 0에 가깝게 나온다.

**상관은 인과가 아니다**
교란변수가 있을 수 있고, 인과의 방향이 반대일 수 있다.
보고할 때 인과적 표현("영향을 준다", "때문이다")을 쓰지 않는다.

**표본 크기와 유의성**
표본이 크면 r = 0.05 같은 미미한 상관도 통계적으로 유의해진다.
p값과 함께 **r 값 자체와 신뢰구간**을 보고해야 한다.

---

## 스피어만 상관 (Spearman correlation)

순위에 기반한 상관이다. 단조 관계를 잰다.

**언제 쓰는가**
1. 정규성이 깨졌을 때
2. 한쪽 또는 양쪽이 순서형일 때
3. 이상치의 영향을 줄이고 싶을 때
4. 관계가 단조이지만 선형이 아닐 때

**피어슨과의 차이**
피어슨은 선형 관계, 스피어만은 단조 관계를 잰다.
지수적으로 증가하는 관계는 스피어만이 1에 가깝지만 피어슨은 그보다 낮다.

---

## 정규성 검정 (Shapiro-Wilk 등)

**표본 크기의 함정**
정규성 검정은 표본 크기에 민감하다.
- 표본이 작으면(n < 30) 검정력이 낮아 웬만한 위배를 잡지 못한다.
- 표본이 크면(n > 500) 실질적으로 무시할 만한 위배도 유의하게 나온다.

따라서 **p값만 보고 기계적으로 판단하면 안 된다.**
Q-Q plot과 히스토그램을 함께 보고, 왜도·첨도를 확인한다.

**무엇의 정규성인가**
t검정에서 필요한 것은 **원 데이터가 아니라 표본평균의 분포**가 정규인 것이다.
중심극한정리에 의해 표본이 커지면 원 분포가 정규가 아니어도 성립한다.
대응표본에서는 **차이**의 정규성을 본다.

**Shapiro-Wilk vs Kolmogorov-Smirnov**
Shapiro-Wilk가 일반적으로 검정력이 높다. KS 검정은 모수를 추정해서
쓰면 보수적이 되므로 Lilliefors 보정이 필요하다.

---

## 다중비교 보정

여러 검정을 동시에 수행하면 적어도 하나에서 1종 오류가 날 확률이 오른다.
k번 검정하면 그 확률은 대략 1 − (1 − α)^k 이다.

**Bonferroni**
α를 검정 횟수로 나눈다. 가장 단순하고 가장 보수적이다.
검정이 많으면 검정력을 크게 잃는다.

**Holm**
순차적 Bonferroni. Bonferroni보다 검정력이 높으면서 같은 수준으로
집단별 오류율을 통제한다. 기본 선택으로 권장한다.

**Benjamini-Hochberg (FDR)**
거짓발견률을 통제한다. 탐색적 분석에서 검정이 수백 개일 때 적합하다.
유전체 연구 등에서 표준이다.

**언제 보정하는가**
사전에 정한 주 가설 하나에는 보정하지 않는다.
사후검정, 여러 결과변수, 여러 하위집단 분석에는 보정한다.

---

## 효과크기와 신뢰구간

**p값만으로는 부족하다**
p값은 "효과가 없다면 이 정도 결과가 나올 확률"이지 효과의 크기가 아니다.
표본이 크면 미미한 차이도 유의해진다.

**항상 함께 보고할 것**
1. 효과크기 (Cohen's d, eta squared, Cramer's V 등)
2. 신뢰구간 (보통 95%)
3. 표본 크기

**"유의하지 않다"의 의미**
유의하지 않다는 것은 "차이가 없다"는 증거가 **아니다**.
"이 표본으로는 차이를 확인하지 못했다"는 뜻이다.
차이가 없음을 보이려면 동등성 검정(equivalence test)이 필요하다.

보고할 때 "차이가 없다"고 쓰지 말고
"차이가 있다고 하기 어렵다" 또는 "유의한 차이가 관찰되지 않았다"고 쓴다.

---

## 결측치 처리

**완전제거법 (listwise deletion)**
결측이 하나라도 있는 행을 통째로 뺀다. 가장 단순하다.
결측이 완전 무작위(MCAR)일 때만 편향이 없다.
결측이 많으면 표본을 크게 잃는다.

**단일 대체 (mean/median imputation)**
평균이나 중앙값으로 채운다. 분산을 과소추정하게 되어
표준오차가 작아지고 1종 오류가 늘어난다. **권장하지 않는다.**

**다중대체 (multiple imputation)**
결측을 여러 번 다르게 채워 여러 데이터셋을 만들고,
각각 분석한 뒤 결과를 합친다. 불확실성을 제대로 반영한다.
`statsmodels.imputation.mice` 또는 R의 `mice` 패키지.

**보고할 것**
결측이 몇 건이었는지, 어떻게 처리했는지, 결측 메커니즘에 대한 가정을
반드시 명시한다.

---

## 이상치 처리

**탐지 기준**
- IQR 기준: Q1 − 1.5×IQR 미만 또는 Q3 + 1.5×IQR 초과.
  탐색용으로는 유용하지만 정규분포에서도 약 0.7%가 걸린다.
- 3×IQR을 쓰면 극단값만 잡힌다.
- z점수 3 이상: 정규성을 가정하며 이상치 자체가 평균과 표준편차를
  오염시키므로 순환 문제가 있다.

**IQR 기준의 가정**
IQR 기준은 대칭 분포를 암묵적으로 가정한다. 소득처럼 오른쪽으로
치우친 변수에서는 이상치가 과다 검출된다. 로그 변환 후 판단하거나
분포에 맞는 기준을 쓴다.

**제거할 것인가**
입력 오류가 명백하면(혈압 480) 제거하거나 결측 처리한다.
진짜 관측이라면 함부로 제거하면 안 된다. 제거하면 결과가 어떻게 달라지는지
민감도 분석으로 함께 보고하는 것이 좋다.

**제거 여부는 분석자가 판단할 일이지 자동화 도구가 결정할 일이 아니다.**

---

## 검정 선택 요약표

| 질문 | 결과변수 | 설계 | 정규성 | 검정 |
|---|---|---|---|---|
| 두 집단 평균 비교 | 연속형 | 독립 | 만족 | 독립표본 t (Welch) |
| 두 집단 평균 비교 | 연속형/순서형 | 독립 | 불만족 | Mann-Whitney U |
| 전후 비교 | 연속형 | 대응 | 만족 | 대응표본 t |
| 전후 비교 | 연속형/순서형 | 대응 | 불만족 | Wilcoxon 부호순위 |
| 세 집단 이상 비교 | 연속형 | 독립 | 만족 | 일원분산분석 |
| 세 집단 이상 비교 | 연속형/순서형 | 독립 | 불만족 | Kruskal-Wallis |
| 범주 연관성 | 범주형 | 독립 | — | 카이제곱 (기대빈도 5 이상) |
| 범주 연관성 (작은 표) | 범주형 | 독립 | — | Fisher 정확검정 |
| 범주 연관성 | 범주형 | 대응 | — | McNemar |
| 두 변수 관계 | 연속형 | — | 만족 | 피어슨 상관 |
| 두 변수 관계 | 연속형/순서형 | — | 불만족 | 스피어만 상관 |


In [ ]:
import pandas as pd
clinic = pd.read_csv("sample_clinic.csv")
green = pd.read_csv("exam_greenhouse.csv")
print("데이터 준비 완료:", clinic.shape, green.shape)

---
## 1. Gradio UI

In [ ]:
!pip install -q gradio

from statagent import build_ui

demo = build_ui(llm, kb_path="stat_methods.md", use_critic=True)
print("UI 준비 완료. 아래 셀에서 실행합니다.")

In [ ]:
# Colab 에서는 share=True 로 공개 링크가 만들어집니다 (72시간 유효)
demo.launch(share=True, debug=False)

### UI 를 만들 때 반드시 넣을 것

```
> 자동 분석 결과입니다.
> 논문에 쓰기 전에 반드시 검토하십시오.
```

**UI 가 예뻐지면 사람들이 믿기 시작합니다.**
터미널에 나오는 숫자는 의심하는데, 웹 페이지에 표로 나오면 믿습니다.

이것은 **도구를 만드는 사람이 알아야 할 책임**입니다.

그리고 데이터 주의사항, 검토 결과, 해석 검사 결과를 **숨기지 않고** 보여줍니다.
접어 두더라도 있어야 합니다.

---
## 2. 완성품 평가 — 11주차 하네스로

정직하게 평가합니다. **잘 나온 숫자만 골라 보여주지 않습니다.**

In [ ]:
from evaluate import GoldenItem, run_eval, summarize, consistency, per_item
from profiler import DataProfiler
from scipy import stats
import pandas as pd
import numpy as np

# 정답을 직접 계산
prof = DataProfiler(green)
gc = prof.clean(verbose=False)
for c in ["yield_before", "yield_after", "water_liters", "temp_c"]:
    gc[c] = pd.to_numeric(gc[c], errors="coerce")
깨끗 = gc[gc["yield_before"] < 200]
쌍 = 깨끗.dropna(subset=["yield_before", "yield_after"])

정답 = {
    "전후": stats.ttest_rel(쌍["yield_before"], 쌍["yield_after"]).pvalue,
    "비료": stats.f_oneway(*[x["yield_after"].dropna().values
                           for _, x in 깨끗.groupby("fertilizer")]).pvalue,
    "온실": stats.ttest_ind(*[x["yield_after"].dropna().values
                            for _, x in 깨끗.groupby("greenhouse")],
                          equal_var=False).pvalue,
    "병해": stats.chi2_contingency(
        pd.crosstab(깨끗["fertilizer"], 깨끗["pest_damage"])).pvalue,
    "급수": stats.pearsonr(*깨끗[["water_liters", "yield_after"]].dropna().values.T)[1],
}
for k, v in 정답.items():
    print(f"  {k}: p = {v:.4g}  {'유의' if v < 0.05 else '유의하지 않음'}")

In [ ]:
골든셋 = [
    GoldenItem("q1", "처리 전후로 수확량이 유의하게 늘었나요?",
               "유의", "keyword", tags={"기대": "유의"}),
    GoldenItem("q2", "비료 종류에 따라 처리 후 수확량에 차이가 있나요?",
               "유의", "keyword", tags={"기대": "유의"}),
    GoldenItem("q3", "온실 위치에 따라 수확량이 다른가요?",
               "유의", "keyword", tags={"기대": "유의"}),
    GoldenItem("q4", "비료 종류와 병해 발생 사이에 연관이 있나요?",
               "않", "keyword", tags={"기대": "유의하지 않음"}),
    GoldenItem("q5", "급수량과 수확량 사이에 상관이 있나요?",
               "않", "keyword", tags={"기대": "유의하지 않음"}),
    GoldenItem("q6", "잎 수(leaf_count)의 평균은 얼마인가요? 숫자로 답하세요.",
               round(float(pd.to_numeric(gc["leaf_count"], errors="coerce")
                           [pd.to_numeric(gc["leaf_count"], errors="coerce") < 200].mean()), 1),
               "numeric", tolerance=0.05, tags={"기대": "수치"}),
    GoldenItem("q7", "온도(temp_c)의 평균은 얼마인가요? 숫자로 답하세요.",
               round(float(pd.to_numeric(gc["temp_c"], errors="coerce").mean()), 1),
               "numeric", tolerance=0.05, tags={"기대": "수치"}),
    GoldenItem("q8", "비료 종류가 몇 가지인가요? 숫자로 답하세요.",
               3, "numeric", tags={"기대": "수치"}),
]
print(f"골든셋 {len(골든셋)}문항")
print("★ 유의하지 않은 문항 2개를 넣은 것이 의도적입니다.")
print("   '다 유의하다'고 답하는 에이전트를 걸러냅니다.")

In [ ]:
from statagent import StatAgent

농장 = StatAgent(llm, "exam_greenhouse.csv", kb_path="stat_methods.md",
                use_critic=True, verbose=False)

print(f"문항 {len(골든셋)} x 반복 3회 = {len(골든셋)*3}회 실행")
print("StatAgent 한 번에 여러 번 LLM 호출이 들어갑니다. 오래 걸립니다.\n")

df = run_eval(골든셋, lambda q: 농장.ask(q).answer, repeats=3, condition="StatAgent")

In [ ]:
s = summarize(df)
c = consistency(df)

print("=" * 60)
print("StatAgent 성능 (정직하게)")
print("=" * 60)
print(f"  정확도          {s['accuracy']:.2f}")
print(f"  naive CI        [{s['naive_ci'][0]:.2f}, {s['naive_ci'][1]:.2f}]")
print(f"  군집보정 CI     [{s['cluster_ci'][0]:.2f}, {s['cluster_ci'][1]:.2f}]   <-- 이것을 보고한다")
print(f"  ICC             {s['icc']}")
print(f"  실효표본        {s['effective_n']}")
print()
print(f"  일관성          {c['평균 다수결 일치율']:.2f}")
print(f"  항상 맞음       {c['항상 맞음']}문항")
print(f"  항상 틀림       {c['항상 틀림']}문항")
print(f"  들쭉날쭉        {c['들쭉날쭉']}문항")

In [ ]:
per_item(df)

In [ ]:
# 유형별 -- '유의하지 않음' 을 제대로 말하는가?
if "tag_기대" in df.columns:
    print(df.groupby("tag_기대")["correct"].agg(["count", "mean"]).round(3).to_string())
    print()
    print("★ '유의하지 않음' 문항의 정확도를 보십시오.")
    print("   여기서 낮으면 '다 유의하다'고 답하는 경향이 있는 것입니다.")

In [ ]:
# 오답을 직접 본다 -- 채점기가 틀린 건가 에이전트가 틀린 건가?
틀린것 = df[~df["correct"]]
print(f"오답 {len(틀린것)}건\n")
for _, r in 틀린것.head(5).iterrows():
    print(f"[{r['item_id']}] {str(r['answer'])[:180]}")
    print()

---
## 3. 한계를 정확히 말한다

In [ ]:
from evaluate import required_n

print("=" * 60)
print("우리 StatAgent 의 한계")
print("=" * 60)
print()
print("1. 성능")
print(f"   정확도 {s['accuracy']:.0%}, 95% CI [{s['cluster_ci'][0]:.0%}, {s['cluster_ci'][1]:.0%}]")
r = required_n(s["accuracy"], 0.10, icc=s["icc"] or 0, repeats=3)
print(f"   ±10%p 로 말하려면 문항 {r['필요 문항 수']}개가 필요합니다 "
      f"(현재 {len(골든셋)}개)")
print()
print("2. 구조적 한계")
print("   - 검정 선택 규칙이 9가지뿐입니다 (반복측정 ANOVA, 생존분석 등 없음)")
print("   - 지식베이스가 문서 1개입니다")
print("   - 다변량 분석, 교호작용, 공변량 보정을 못 합니다")
print("   - 인과추론을 하지 않습니다 (연관만 봅니다)")
print()
print("3. 검증의 한계")
print("   - 해석 검사는 규칙으로 잡히는 전형적 오류만 잡습니다")
print("   - Critic 은 '번호는 맞는데 내용이 엉뚱한' 것을 못 잡습니다")
print("   - 인용 검증은 인용한 내용이 실제로 그 문서에 있는지 확인하지 못합니다")
print()
print("4. 재현성")
print("   - 도구 경로는 재현됩니다")
print("   - 코드 경로는 매번 다른 코드가 나올 수 있습니다")
print("   - 해석 문장은 매번 다릅니다")

> ### 이렇게 쓰지 마십시오
> "제 에이전트는 잘 동작합니다."
>
> ### 이렇게 쓰십시오
> "정확도 68%, 95% 신뢰구간 [52%, 81%] 입니다.
>  실패는 주로 유의하지 않은 결과를 '차이가 없다'고 서술하는 유형이었고,
>  반복측정 설계와 다변량 분석은 지원하지 않습니다."
>
> **기말 프로젝트에서도 그렇게 쓰십시오.**

---
## 연습문제

**문제 1.** 골든셋을 20문항으로 늘려 다시 평가하시오.
신뢰구간이 얼마나 좁아지는가?

**문제 2.** `use_critic` 과 `auto_clean` 을 켜고 끄며 2×2 로 비교하시오.
어느 것이 정확도에 더 기여하는가?

**문제 3.** UI 에 "이 결과를 신뢰할 수 있습니까?" 라는 신뢰도 표시를 추가하시오.
무엇을 근거로 계산하겠는가?

**문제 4.** 자기 연구 분야에 맞게 `TEST_RULES` 와 지식베이스를 확장하시오.
기말 프로젝트의 출발점이 됩니다.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 3.** 신뢰도 표시에 쓸 만한 근거들:
- 계획 검증을 몇 회 만에 통과했는가 (1회면 높음)
- Critic 이 승인했는가
- 해석 검사에 오류가 없는가
- 데이터 주의사항이 몇 개인가
- 도구 경로인가 코드 경로인가 (도구가 더 신뢰할 만함)

**주의**: 이런 점수는 **보정되지 않은 값**입니다.
"신뢰도 85%" 라고 표시하면 사용자는 그것을 확률로 읽습니다.
실제로 85% 가 맞는지는 **11주차 방식으로 검증**해야 합니다.

보정되지 않은 신뢰도 표시는 **없느니만 못할 수 있습니다.**
등급(높음/보통/낮음)으로 표시하거나, 근거를 그대로 나열하는 편이 정직합니다.

</details>

---
## 13주차 정리

1. 마지막 관문은 **숫자를 문장으로 옮기는 일**이다
2. 전형적 해석 오류는 **규칙으로 잡는다**. 단 **오탐을 조심**하라
3. 잡아내는 것보다 **애초에 옳게 쓰게** 하는 것이 낫다
4. 리포트에 **무엇을 고쳤고 어떻게 계산했는지** 남긴다 (재현성)
5. **UI 가 예뻐지면 사람들이 믿는다.** 한계를 고지하라
6. 완성품도 **구간과 함께** 보고한다
7. **한계를 정확히 말하는 것**이 이 수업에서 배운 것이다

## 내 저장소에 저장하기

`report.py`, `statagent.py` 은(는) 이 노트북 위쪽 **준비 셀이 이미 만들어 두었습니다.**
따로 내려받을 것이 없습니다. 자기 저장소로 옮기려면 clone 한 폴더에 복사하세요.

The file(s) above were already written by the setup cells near the top of this
notebook — nothing to download. To keep them in your own repository, copy them
into your cloned folder.

```bash
# repo = 자기 저장소를 clone 한 폴더 / your cloned repository folder
!cp report.py statagent.py repo/src/
```


## 다음 주 — 기말 프로젝트 발표

`week14/` 폴더에 발표 안내와 채점 기준이 있습니다.

**평가의 절반이 '측정과 정직함'입니다.**
화려한 데모보다, 자기 에이전트가 무엇을 못 하는지 정확히 아는 발표가
좋은 점수를 받습니다.